In [ ]:
import pandas as pd

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

print("=" * 70)
print("INSPECTING API DATASET WITHOUT LOADING IT")
print("=" * 70)

# Read only first 5 rows
sample = pd.read_csv(API_PATH, nrows=5)

print("Columns:", len(sample.columns))
print("Shape of sample:", sample.shape)

print("\nFirst 20 columns:")
print(sample.columns[:20].tolist())

print("\nLast 10 columns:")
print(sample.columns[-10:].tolist())

print("\nData types:")
print(sample.dtypes.value_counts())

print("\nFirst rows:")
print(sample.iloc[:, :10])

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

# ============================================================
# API FREQUENCY ANALYSIS - MEMORY SAFE
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

FREQ_THRESHOLD = 20
CHUNK_SIZE = 1000

print("=" * 70)
print("API FREQUENCY ANALYSIS - MEMORY SAFE")
print("=" * 70)

# ------------------------------------------------------------
# Read header only
# ------------------------------------------------------------

header = pd.read_csv(API_PATH, nrows=0)

ID_COL = "SHA256"
TARGET = "Type"

API_FEATURES = [
    c for c in header.columns
    if c not in [ID_COL, TARGET]
]

print("Total API features:", len(API_FEATURES))
print("Samples expected: 29,498")
print("Frequency threshold:", FREQ_THRESHOLD)

# ------------------------------------------------------------
# Initialize frequency counter
# ------------------------------------------------------------

api_counts = pd.Series(
    0,
    index=API_FEATURES,
    dtype=np.int64
)

# ------------------------------------------------------------
# Process file in chunks
# ------------------------------------------------------------

print("\nProcessing dataset in chunks...")

processed = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=API_FEATURES,
    chunksize=CHUNK_SIZE
):

    # Count samples where API is present
    api_counts += (chunk > 0).sum(axis=0)

    processed += len(chunk)

    if processed % 5000 == 0 or processed >= 29498:
        print(
            f"Processed: {processed:,} / 29,498 samples"
        )

    del chunk
    gc.collect()

# ------------------------------------------------------------
# Frequency results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FREQUENCY RESULTS")
print("=" * 70)

print("Total APIs:", len(api_counts))

print(
    "APIs appearing >= 1 sample:",
    (api_counts >= 1).sum()
)

print(
    "APIs appearing >= 5 samples:",
    (api_counts >= 5).sum()
)

print(
    "APIs appearing >= 10 samples:",
    (api_counts >= 10).sum()
)

print(
    "APIs appearing >= 20 samples:",
    (api_counts >= 20).sum()
)

print(
    "APIs appearing >= 50 samples:",
    (api_counts >= 50).sum()
)

print(
    "APIs appearing >= 100 samples:",
    (api_counts >= 100).sum()
)

print(
    "APIs appearing >= 500 samples:",
    (api_counts >= 500).sum()
)

print(
    "APIs appearing >= 1000 samples:",
    (api_counts >= 1000).sum()
)

# ------------------------------------------------------------
# Frequency statistics
# ------------------------------------------------------------

print("\nFrequency statistics:")

print(api_counts.describe())

# ------------------------------------------------------------
# Top APIs
# ------------------------------------------------------------

frequency_df = pd.DataFrame({
    "API": api_counts.index,
    "Samples_With_API": api_counts.values
})

frequency_df["Percentage"] = (
    frequency_df["Samples_With_API"] /
    processed * 100
)

frequency_df = frequency_df.sort_values(
    "Samples_With_API",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("TOP 30 MOST COMMON APIs")
print("=" * 70)

print(
    frequency_df.head(30).to_string(index=False)
)

# ------------------------------------------------------------
# Rare APIs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAREST APIs")
print("=" * 70)

print(
    frequency_df[
        frequency_df["Samples_With_API"] == 1
    ].head(30).to_string(index=False)
)

# ------------------------------------------------------------
# Save frequency analysis
# ------------------------------------------------------------

FREQ_OUTPUT = "/kaggle/working/API_Frequency_Analysis.csv"

frequency_df.to_csv(
    FREQ_OUTPUT,
    index=False
)

# ------------------------------------------------------------
# Save frequency-qualified APIs
# ------------------------------------------------------------

freq_features = frequency_df[
    frequency_df["Samples_With_API"] >= FREQ_THRESHOLD
]["API"].tolist()

FREQ_FEATURE_OUTPUT = "/kaggle/working/API_Frequency_Selected.csv"

pd.DataFrame({
    "API": freq_features,
    "Samples_With_API": [
        api_counts[x] for x in freq_features
    ]
}).to_csv(
    FREQ_FEATURE_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("FREQUENCY FILTER COMPLETE")
print("=" * 70)

print("Frequency threshold:", FREQ_THRESHOLD)
print("Starting API features:", len(API_FEATURES))
print("Remaining API features:", len(freq_features))

print(
    "Percentage retained:",
    round(
        len(freq_features) /
        len(API_FEATURES) * 100,
        2
    ),
    "%"
)

print("\nSaved:")
print(FREQ_OUTPUT)
print(FREQ_FEATURE_OUTPUT)

# Cleanup
del api_counts
del frequency_df
gc.collect()

In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# API MUTUAL INFORMATION FEATURE SELECTION
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
FREQ_FEATURE_PATH = "/kaggle/working/API_Frequency_Selected.csv"

MI_THRESHOLD = 0.01
CHUNK_SIZE = 1000

print("=" * 70)
print("API MUTUAL INFORMATION FEATURE SELECTION")
print("=" * 70)

# ============================================================
# LOAD SELECTED API FEATURES
# ============================================================

freq_df = pd.read_csv(FREQ_FEATURE_PATH)

freq_features = freq_df["API"].tolist()

print("\nFrequency-qualified APIs:", len(freq_features))
print("MI threshold:", MI_THRESHOLD)

# ============================================================
# LOAD TARGET + SELECTED FEATURES ONLY
# ============================================================

print("\nLoading selected API features...")

usecols = ["Type"] + freq_features

chunks = []

processed = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=usecols,
    chunksize=CHUNK_SIZE
):

    # Convert API features to compact binary representation
    X_chunk = (
        chunk[freq_features]
        .to_numpy(dtype=np.uint8)
    )

    chunks.append(X_chunk)

    processed += len(chunk)

    if processed % 5000 == 0 or processed >= 29498:
        print(
            f"Processed: {processed:,} / 29,498 samples"
        )

    del chunk
    gc.collect()

# Combine compact chunks
X = np.vstack(chunks)

del chunks
gc.collect()

# Load target separately
target_df = pd.read_csv(
    API_PATH,
    usecols=["Type"]
)

y = target_df["Type"].to_numpy()

del target_df
gc.collect()

print("\n" + "=" * 70)
print("FEATURE MATRIX READY")
print("=" * 70)

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("Approx X memory:",
      round(X.nbytes / (1024**2), 2), "MB")

print("y shape:", y.shape)

# ============================================================
# MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING MUTUAL INFORMATION")
print("=" * 70)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("\nThis may take some time...")

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

# ============================================================
# MI RESULTS
# ============================================================

mi_df = pd.DataFrame({
    "API": freq_features,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("MUTUAL INFORMATION STATISTICS")
print("=" * 70)

print(mi_df["MI_Score"].describe())

# ============================================================
# TOP 30
# ============================================================

print("\n" + "=" * 70)
print("TOP 30 APIs BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(30).to_string(index=False)
)

# ============================================================
# MI THRESHOLD ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("MI THRESHOLD ANALYSIS")
print("=" * 70)

for threshold in [
    0.001,
    0.005,
    0.010,
    0.020,
    0.030,
    0.050,
    0.100
]:

    count = (
        mi_df["MI_Score"] >= threshold
    ).sum()

    print(
        f"MI >= {threshold:<6}: "
        f"{count:4d} APIs"
    )

# ============================================================
# MI FILTER
# ============================================================

mi_selected = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

print("\n" + "=" * 70)
print("MI FILTER RESULT")
print("=" * 70)

print(
    "Before MI filtering:",
    len(mi_df)
)

print(
    "After MI filtering:",
    len(mi_selected)
)

print(
    "Percentage retained:",
    round(
        len(mi_selected) /
        len(mi_df) * 100,
        2
    ),
    "%"
)

# ============================================================
# SAVE
# ============================================================

MI_OUTPUT = "/kaggle/working/API_Candidate_Features_MI.csv"

mi_df.to_csv(
    MI_OUTPUT,
    index=False
)

MI_SELECTED_OUTPUT = "/kaggle/working/API_MI_Selected.csv"

mi_selected.to_csv(
    MI_SELECTED_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("All MI scores:")
print(MI_OUTPUT)

print("\nMI-qualified APIs:")
print(MI_SELECTED_OUTPUT)

# ============================================================
# CLEANUP
# ============================================================

del X
del y
del mi_scores
del mi_df
del mi_selected

gc.collect()

print("\n" + "=" * 70)
print("API MI ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
import os

print("=" * 70)
print("API TOP-MI FEATURE REDUCTION")
print("=" * 70)

MI_PATH = "/kaggle/working/API_MI_Selected.csv"

# ============================================================
# LOAD MI RESULTS ONLY
# ============================================================

mi_df = pd.read_csv(MI_PATH)

print("MI-qualified APIs:", len(mi_df))

# Sort by MI
mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# SELECT TOP 200
# ============================================================

TOP_N = 200

top_api_df = mi_df.head(TOP_N).copy()

print("\nTop API selection")
print("-" * 70)

print("Before:", len(mi_df))
print("After :", len(top_api_df))

print("\nTop 30 APIs:")
print(
    top_api_df.head(30).to_string(index=False)
)

# ============================================================
# SAVE
# ============================================================

TOP_API_PATH = "/kaggle/working/API_Top200_MI.csv"

top_api_df.to_csv(
    TOP_API_PATH,
    index=False
)

print("\nSaved:")
print(TOP_API_PATH)

print("\n" + "=" * 70)
print("TOP-MI API REDUCTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import os

print("=" * 70)
print("VERIFYING SELECTED FEATURE FILES")
print("=" * 70)

files = [
    "/kaggle/working/API_Top200_MI.csv",
    "/kaggle/working/API_MI_Selected.csv",
    "/kaggle/working/DLL_Candidate_Features.csv",
    "/kaggle/working/DLL_Candidate_Features_MI.csv",
]

for path in files:
    print(
        f"{os.path.basename(path):40s}",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

print("\n" + "=" * 70)

api = pd.read_csv(
    "/kaggle/working/API_Top200_MI.csv"
)

dll = pd.read_csv(
    "/kaggle/working/DLL_Candidate_Features.csv"
)

print("Top API features :", len(api))
print("Final DLL features:", len(dll))

print("\nAPI columns:")
print(api.columns.tolist())

print("\nDLL columns:")
print(dll.columns.tolist())

In [ ]:
# ============================================================
# MEMORY-SAFE COMPACT API + DLL DATASET BUILDER - FIXED
# ============================================================

import os
import gc
import pandas as pd
import numpy as np

print("=" * 70)
print("BUILDING COMPACT API + DLL DATASET - MEMORY SAFE")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

API_FEATURES_PATH = "/kaggle/working/API_Top200_MI.csv"
DLL_FEATURES_PATH = "/kaggle/working/DLL_Candidate_Features.csv"

API_TEMP = "/kaggle/working/API_Compact_Temp.csv"
FINAL_PATH = "/kaggle/working/API_DLL_Compact_227.csv"

# ============================================================
# LOAD FEATURE LISTS
# ============================================================

api_selected = pd.read_csv(API_FEATURES_PATH)
dll_selected = pd.read_csv(DLL_FEATURES_PATH)

api_features = api_selected["API"].tolist()
dll_features = dll_selected["DLL"].tolist()

print(f"Selected API features : {len(api_features)}")
print(f"Selected DLL features : {len(dll_features)}")
print(f"Total behavioral features : {len(api_features) + len(dll_features)}")

# ============================================================
# VERIFY API COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING API COLUMNS")
print("=" * 70)

api_header = pd.read_csv(API_PATH, nrows=0)

required_api = ["SHA256", "Type"] + api_features

missing_api = [
    col for col in required_api
    if col not in api_header.columns
]

if missing_api:
    print("Missing API columns:")
    print(missing_api)
    raise ValueError("Some selected API columns are missing.")

print("All selected API columns found.")

# ============================================================
# VERIFY DLL COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING DLL COLUMNS")
print("=" * 70)

dll_header = pd.read_csv(DLL_PATH, nrows=0)

required_dll = ["SHA256", "Type"] + dll_features

missing_dll = [
    col for col in required_dll
    if col not in dll_header.columns
]

if missing_dll:
    print("Missing DLL columns:")
    print(missing_dll)
    raise ValueError("Some selected DLL columns are missing.")

print("All selected DLL columns found.")

# ============================================================
# STEP 1: EXTRACT SELECTED API FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: EXTRACTING SELECTED API FEATURES")
print("=" * 70)

if os.path.exists(API_TEMP):
    os.remove(API_TEMP)

api_usecols = ["SHA256", "Type"] + api_features

# IMPORTANT:
# SHA256 = string
# Type = uint8
# API features = uint8

api_dtype = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in api_features:
    api_dtype[feature] = "uint8"

first_chunk = True
total_rows = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=api_usecols,
    chunksize=500,
    dtype=api_dtype,
    low_memory=False
):

    total_rows += len(chunk)

    chunk.to_csv(
        API_TEMP,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    del chunk
    gc.collect()

    if total_rows % 2500 == 0:
        print(f"API processed: {total_rows:,} rows")

print(f"\nAPI extraction complete: {total_rows:,} rows")

# ============================================================
# LOAD COMPACT API DATA
# ============================================================

print("\nLoading compact API dataset...")

api_dtype_load = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in api_features:
    api_dtype_load[feature] = "uint8"

api_df = pd.read_csv(
    API_TEMP,
    dtype=api_dtype_load
)

print("Compact API shape:", api_df.shape)

# ============================================================
# STEP 2: EXTRACT DLL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: EXTRACTING SELECTED DLL FEATURES")
print("=" * 70)

dll_usecols = ["SHA256", "Type"] + dll_features

dll_dtype = {
    "SHA256": "string",
    "Type": "uint8"
}

for feature in dll_features:
    dll_dtype[feature] = "uint8"

dll_df = pd.read_csv(
    DLL_PATH,
    usecols=dll_usecols,
    dtype=dll_dtype,
    low_memory=False
)

print("DLL dataset shape:", dll_df.shape)

# ============================================================
# CHECK ROW ALIGNMENT
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DATA ALIGNMENT")
print("=" * 70)

if len(api_df) != len(dll_df):
    raise ValueError(
        f"Row mismatch! API={len(api_df)}, DLL={len(dll_df)}"
    )

print("Row counts match:", len(api_df))

# ============================================================
# CHECK SHA256 ALIGNMENT
# ============================================================

if api_df["SHA256"].equals(dll_df["SHA256"]):

    print("SHA256 order already aligned.")

else:

    print("SHA256 order differs.")
    print("Aligning DLL dataset to API order...")

    dll_df = dll_df.set_index("SHA256")

    api_hashes = api_df["SHA256"]

    dll_df = dll_df.loc[api_hashes].reset_index()

    print("DLL rows successfully aligned.")

# ============================================================
# VERIFY LABEL ALIGNMENT
# ============================================================

if not api_df["Type"].equals(dll_df["Type"]):

    raise ValueError(
        "Type labels do not match between API and DLL datasets."
    )

print("Target labels match.")

# ============================================================
# STEP 3: COMBINE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: COMBINING API + DLL FEATURES")
print("=" * 70)

final_df = pd.concat(
    [
        api_df[["SHA256", "Type"] + api_features],
        dll_df[dll_features]
    ],
    axis=1
)

print("Final shape:", final_df.shape)

# ============================================================
# VERIFY FINAL DATASET
# ============================================================

expected_columns = (
    2
    + len(api_features)
    + len(dll_features)
)

print("\nExpected columns:", expected_columns)
print("Actual columns  :", final_df.shape[1])

if final_df.shape[1] != expected_columns:
    raise ValueError("Final column count mismatch.")

# ============================================================
# VERIFY TARGET
# ============================================================

print("\nTarget distribution:")

print(
    final_df["Type"]
    .value_counts()
    .sort_index()
)

# ============================================================
# VERIFY FEATURE VALUES
# ============================================================

print("\nChecking behavioral feature values...")

feature_columns = api_features + dll_features

unique_values = set(
    final_df[feature_columns]
    .stack()
    .unique()
)

print("Unique behavioral values:", sorted(unique_values))

if not unique_values.issubset({0, 1}):
    raise ValueError(
        "Unexpected values detected. "
        "Behavioral features should be binary 0/1."
    )

print("All behavioral features are binary.")

# ============================================================
# MEMORY
# ============================================================

memory_mb = (
    final_df.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"\nFinal dataframe memory: {memory_mb:.2f} MB")

# ============================================================
# SAVE
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINAL DATASET")
print("=" * 70)

final_df.to_csv(
    FINAL_PATH,
    index=False
)

print("\nSaved successfully:")
print(FINAL_PATH)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("COMPACT DATASET CREATED SUCCESSFULLY")
print("=" * 70)

print(f"Samples              : {len(final_df):,}")
print(f"API features         : {len(api_features)}")
print(f"DLL features         : {len(dll_features)}")
print(f"Behavioral features  : {len(feature_columns)}")
print(f"Total columns        : {len(final_df.columns)}")

print("\nFinal shape:")
print(final_df.shape)

print("\nTarget distribution:")
print(final_df["Type"].value_counts().sort_index())

print("\nOutput:")
print(FINAL_PATH)

In [ ]:
# ============================================================
# DIAGNOSING API / DLL ROW MISMATCH
# ============================================================

import pandas as pd

API_TEMP = "/kaggle/working/API_Compact_Temp.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

print("=" * 70)
print("DIAGNOSING API / DLL ROW MISMATCH")
print("=" * 70)

# ------------------------------------------------------------
# LOAD ONLY SHA256 + TYPE
# ------------------------------------------------------------

print("\nLoading API identifiers...")

api_ids = pd.read_csv(
    API_TEMP,
    usecols=["SHA256", "Type"],
    dtype={
        "SHA256": "string",
        "Type": "uint8"
    }
)

print("API rows:", len(api_ids))

print("\nLoading DLL identifiers...")

dll_ids = pd.read_csv(
    DLL_PATH,
    usecols=["SHA256", "Type"],
    dtype={
        "SHA256": "string",
        "Type": "uint8"
    }
)

print("DLL rows:", len(dll_ids))

# ============================================================
# DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DUPLICATES")
print("=" * 70)

api_duplicates = api_ids["SHA256"].duplicated().sum()
dll_duplicates = dll_ids["SHA256"].duplicated().sum()

print("API duplicate SHA256:", api_duplicates)
print("DLL duplicate SHA256:", dll_duplicates)

# ============================================================
# SET COMPARISON
# ============================================================

api_hashes = set(api_ids["SHA256"])
dll_hashes = set(dll_ids["SHA256"])

api_only = api_hashes - dll_hashes
dll_only = dll_hashes - api_hashes

print("\n" + "=" * 70)
print("SHA256 SET COMPARISON")
print("=" * 70)

print("Unique API hashes:", len(api_hashes))
print("Unique DLL hashes:", len(dll_hashes))

print("API-only hashes:", len(api_only))
print("DLL-only hashes:", len(dll_only))

# ============================================================
# SHOW API-ONLY SAMPLES
# ============================================================

if len(api_only) > 0:

    print("\n" + "=" * 70)
    print("API-ONLY SAMPLES")
    print("=" * 70)

    api_only_df = api_ids[
        api_ids["SHA256"].isin(api_only)
    ].copy()

    print(api_only_df.to_string(index=False))

# ============================================================
# SHOW DLL-ONLY SAMPLES
# ============================================================

if len(dll_only) > 0:

    print("\n" + "=" * 70)
    print("DLL-ONLY SAMPLES")
    print("=" * 70)

    dll_only_df = dll_ids[
        dll_ids["SHA256"].isin(dll_only)
    ].copy()

    print(dll_only_df.to_string(index=False))

# ============================================================
# LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTIONS")
print("=" * 70)

print("\nAPI:")
print(api_ids["Type"].value_counts().sort_index())

print("\nDLL:")
print(dll_ids["Type"].value_counts().sort_index())

# ============================================================
# FINAL DIAGNOSIS
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSIS")
print("=" * 70)

if len(api_only) == 7 and len(dll_only) == 0:
    print("Exactly 7 API-only samples found.")
    print("DLL dataset is a subset of API dataset.")

elif len(api_only) == 0 and len(dll_only) == 0:
    print("SHA256 sets match despite row-count difference.")
    print("Likely duplicate SHA256 rows exist in one dataset.")

else:
    print("API and DLL datasets contain different samples.")
    print("We need to align using SHA256 intersection.")

print("\nDiagnosis complete.")

In [ ]:
# ============================================================
# BUILD FINAL ALIGNED API + DLL DATASET
# SHA256-BASED ALIGNMENT - MEMORY SAFE
# ============================================================

import pandas as pd
import os
import gc

print("=" * 70)
print("BUILDING FINAL ALIGNED API + DLL DATASET")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

API_FEATURE_PATH = "/kaggle/working/API_Top200_MI.csv"
DLL_FEATURE_PATH = "/kaggle/working/DLL_Candidate_Features.csv"

OUTPUT_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

# ============================================================
# LOAD SELECTED FEATURE LISTS
# ============================================================

api_features = pd.read_csv(API_FEATURE_PATH)["API"].tolist()
dll_features = pd.read_csv(DLL_FEATURE_PATH)["DLL"].tolist()

print(f"\nSelected API features : {len(api_features)}")
print(f"Selected DLL features : {len(dll_features)}")
print(f"Total behavioral features : {len(api_features) + len(dll_features)}")

# ============================================================
# STEP 1: LOAD DLL DATASET
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: LOADING DLL DATASET")
print("=" * 70)

dll_usecols = ["SHA256", "Type"] + dll_features

dll_df = pd.read_csv(
    DLL_PATH,
    usecols=dll_usecols
)

print(f"DLL rows loaded: {len(dll_df):,}")

# ============================================================
# CHECK DUPLICATES
# ============================================================

print("\nChecking DLL duplicates...")

dll_duplicates = dll_df["SHA256"].duplicated().sum()

print(f"DLL duplicate SHA256 rows: {dll_duplicates}")

# Keep first occurrence
dll_df = dll_df.drop_duplicates(
    subset="SHA256",
    keep="first"
).reset_index(drop=True)

print(f"DLL unique samples: {len(dll_df):,}")

# ============================================================
# STEP 2: CREATE DLL LOOKUP
# ============================================================

print("\nCreating DLL SHA256 lookup...")

dll_hashes = set(dll_df["SHA256"])

print(f"DLL unique hashes: {len(dll_hashes):,}")

# ============================================================
# STEP 3: STREAM API DATASET
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: STREAMING API DATASET")
print("=" * 70)

api_usecols = ["SHA256", "Type"] + api_features

matched_chunks = []

total_rows = 0
matched_rows = 0
api_seen = set()

for chunk in pd.read_csv(
    API_PATH,
    usecols=api_usecols,
    dtype={
        "SHA256": "string",
        "Type": "int8"
    },
    chunksize=2500
):

    total_rows += len(chunk)

    # Remove duplicate SHA256 within API dataset
    chunk = chunk[
        ~chunk["SHA256"].isin(api_seen)
    ].copy()

    # Add hashes to seen set
    api_seen.update(chunk["SHA256"].tolist())

    # Keep only samples present in DLL dataset
    chunk = chunk[
        chunk["SHA256"].isin(dll_hashes)
    ].copy()

    if len(chunk) > 0:
        matched_rows += len(chunk)
        matched_chunks.append(chunk)

    if total_rows % 10000 < 2500:
        print(
            f"Processed: {total_rows:,} | "
            f"Matched: {matched_rows:,}"
        )

print("\nAPI streaming complete.")

# ============================================================
# COMBINE API MATCHES
# ============================================================

api_df = pd.concat(
    matched_chunks,
    ignore_index=True
)

del matched_chunks
gc.collect()

print(f"\nAPI matched unique samples: {len(api_df):,}")

# ============================================================
# STEP 4: MERGE API + DLL USING SHA256
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: MERGING API + DLL DATA")
print("=" * 70)

# Rename Type columns
api_df = api_df.rename(
    columns={"Type": "Type_API"}
)

dll_df = dll_df.rename(
    columns={"Type": "Type_DLL"}
)

# Inner join using SHA256
final_df = api_df.merge(
    dll_df,
    on="SHA256",
    how="inner",
    suffixes=("", "_DLL")
)

print(f"Final merged shape: {final_df.shape}")

# ============================================================
# STEP 5: VERIFY LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: VERIFYING LABEL CONSISTENCY")
print("=" * 70)

label_mismatch = (
    final_df["Type_API"] != final_df["Type_DLL"]
).sum()

print(f"Label mismatches: {label_mismatch}")

if label_mismatch > 0:

    print("\nWARNING: Label mismatch detected!")

    mismatch_df = final_df[
        final_df["Type_API"] != final_df["Type_DLL"]
    ][
        ["SHA256", "Type_API", "Type_DLL"]
    ]

    print(mismatch_df.head(20))

else:
    print("All API and DLL labels match.")

# ============================================================
# STEP 6: CREATE FINAL TARGET
# ============================================================

final_df["Type"] = final_df["Type_API"]

# Remove duplicate target columns
final_df = final_df.drop(
    columns=["Type_API", "Type_DLL"]
)

# ============================================================
# STEP 7: REMOVE ANY REMAINING DUPLICATES
# ============================================================

before = len(final_df)

final_df = final_df.drop_duplicates(
    subset="SHA256",
    keep="first"
).reset_index(drop=True)

after = len(final_df)

print(f"\nDuplicate samples removed: {before - after}")
print(f"Final unique samples: {after:,}")

# ============================================================
# STEP 8: VERIFY FEATURE COUNT
# ============================================================

expected_features = (
    1 +                 # SHA256
    len(api_features) +
    len(dll_features) +
    1                   # Type
)

print("\n" + "=" * 70)
print("FINAL DATASET VERIFICATION")
print("=" * 70)

print(f"Expected columns : {expected_features}")
print(f"Actual columns   : {len(final_df.columns)}")

print(f"\nSHA256 column present: {'SHA256' in final_df.columns}")
print(f"Type column present  : {'Type' in final_df.columns}")

print(
    f"API features present : "
    f"{sum(x in final_df.columns for x in api_features)} / {len(api_features)}"
)

print(
    f"DLL features present : "
    f"{sum(x in final_df.columns for x in dll_features)} / {len(dll_features)}"
)

# ============================================================
# TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("FINAL TARGET DISTRIBUTION")
print("=" * 70)

print(final_df["Type"].value_counts().sort_index())

# ============================================================
# SAVE
# ============================================================

print("\nSaving final dataset...")

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved to:")
print(OUTPUT_PATH)

print("\n" + "=" * 70)
print("FINAL COMPACT DATASET COMPLETE")
print("=" * 70)

print(f"Final shape: {final_df.shape}")
print(f"Behavioral features: {len(api_features) + len(dll_features)}")
print(f"Samples: {len(final_df):,}")

In [ ]:
# ============================================================
# FINAL SANITY CHECK - COMPACT API + DLL DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

print("=" * 70)
print("FINAL DATASET SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print(f"\nDataset shape: {df.shape}")

# ------------------------------------------------------------
# BASIC STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. BASIC STRUCTURE")
print("=" * 70)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {len(df.columns):,}")
print(f"Memory MB : {df.memory_usage(deep=True).sum() / 1024**2:.2f}")

print(f"\nSHA256 present: {'SHA256' in df.columns}")
print(f"Type present  : {'Type' in df.columns}")

behavioral_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

print(f"Behavioral features: {len(behavioral_cols)}")

# ------------------------------------------------------------
# MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. MISSING VALUES")
print("=" * 70)

missing_total = df[behavioral_cols].isna().sum().sum()

print(f"Total missing behavioral values: {missing_total:,}")

if missing_total == 0:
    print("PASS: No missing behavioral values.")
else:
    print("WARNING: Missing values detected.")

# ------------------------------------------------------------
# DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. DATA TYPES")
print("=" * 70)

print(df[behavioral_cols].dtypes.value_counts())

# ------------------------------------------------------------
# UNIQUE VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. FEATURE VALUE CHECK")
print("=" * 70)

non_binary = []

for col in behavioral_cols:
    values = df[col].dropna().unique()

    if not set(values).issubset({0, 1}):
        non_binary.append(
            (col, len(values), sorted(values)[:10])
        )

print(f"Total behavioral features : {len(behavioral_cols)}")
print(f"Binary features            : {len(behavioral_cols) - len(non_binary)}")
print(f"Non-binary features        : {len(non_binary)}")

if non_binary:
    print("\nNon-binary examples:")
    for item in non_binary[:20]:
        print(item)
else:
    print("PASS: All behavioral features are binary.")

# ------------------------------------------------------------
# ZERO VARIANCE / CONSTANT FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. CONSTANT FEATURES")
print("=" * 70)

nunique = df[behavioral_cols].nunique()

constant_features = nunique[nunique <= 1]

print(f"Constant features: {len(constant_features)}")

if len(constant_features) > 0:
    print("\nConstant features:")
    print(constant_features)
else:
    print("PASS: No constant features.")

# ------------------------------------------------------------
# FEATURE FREQUENCY / SPARSITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. FEATURE SPARSITY")
print("=" * 70)

feature_frequency = df[behavioral_cols].sum()

sparsity = (
    1 - feature_frequency / len(df)
) * 100

print(
    f"Average feature presence: "
    f"{feature_frequency.mean():.2f} samples"
)

print(
    f"Average sparsity: "
    f"{sparsity.mean():.2f}%"
)

print("\nMost common behavioral features:")

top_features = pd.DataFrame({
    "Feature": feature_frequency.index,
    "Samples_With_Feature": feature_frequency.values,
    "Percentage": (
        feature_frequency.values / len(df) * 100
    )
}).sort_values(
    "Samples_With_Feature",
    ascending=False
)

print(top_features.head(20).to_string(index=False))

# ------------------------------------------------------------
# DUPLICATE SHA256
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. SHA256 DUPLICATES")
print("=" * 70)

sha_duplicates = df["SHA256"].duplicated().sum()

print(f"Duplicate SHA256 rows: {sha_duplicates}")

# ------------------------------------------------------------
# DUPLICATE BEHAVIORAL VECTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. DUPLICATE BEHAVIORAL VECTORS")
print("=" * 70)

duplicate_vectors = df.duplicated(
    subset=behavioral_cols
).sum()

print(
    f"Duplicate behavioral vectors: "
    f"{duplicate_vectors:,}"
)

# ------------------------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. TARGET DISTRIBUTION")
print("=" * 70)

target_counts = df["Type"].value_counts().sort_index()

target_percent = (
    target_counts / len(df) * 100
).round(2)

target_summary = pd.DataFrame({
    "Samples": target_counts,
    "Percentage": target_percent
})

print(target_summary)

# ------------------------------------------------------------
# CLASS IMBALANCE
# ------------------------------------------------------------

max_class = target_counts.max()
min_class = target_counts.min()

imbalance_ratio = max_class / min_class

print(
    f"\nLargest class / smallest class ratio: "
    f"{imbalance_ratio:.2f}"
)

# ------------------------------------------------------------
# TARGET LEAKAGE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. TARGET LEAKAGE CHECK")
print("=" * 70)

# Correlation of binary features with target
# This is only a rough screening, not proof of leakage.

target_corr = (
    df[behavioral_cols]
    .corrwith(df["Type"])
    .abs()
    .sort_values(ascending=False)
)

print("Top 20 feature-target correlations:")

print(
    target_corr.head(20).to_string()
)

# ------------------------------------------------------------
# SHA256 UNIQUENESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("11. SHA256 UNIQUENESS")
print("=" * 70)

unique_sha = df["SHA256"].nunique()

print(f"Unique SHA256: {unique_sha:,}")
print(f"Total rows   : {len(df):,}")

if unique_sha == len(df):
    print("PASS: Every sample has a unique SHA256.")
else:
    print("WARNING: Duplicate SHA256 values exist.")

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SANITY CHECK SUMMARY")
print("=" * 70)

checks = {
    "Correct row count": len(df) > 29000,
    "229 columns": len(df.columns) == 229,
    "227 behavioral features": len(behavioral_cols) == 227,
    "No missing values": missing_total == 0,
    "All features binary": len(non_binary) == 0,
    "No constant features": len(constant_features) == 0,
    "Unique SHA256": unique_sha == len(df),
}

for name, result in checks.items():
    print(
        f"{'PASS' if result else 'FAIL'} : {name}"
    )

print("\n" + "=" * 70)
print("SANITY CHECK COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

print("=" * 70)
print("DUPLICATE BEHAVIORAL VECTOR ANALYSIS")
print("=" * 70)

df = pd.read_csv(DATA_PATH)

print(f"\nDataset shape: {df.shape}")

# ------------------------------------------------------------
# IDENTIFY COLUMNS
# ------------------------------------------------------------

ID_COL = "SHA256"
TARGET_COL = "Type"

FEATURE_COLS = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print(f"Behavioral features: {len(FEATURE_COLS)}")

# ------------------------------------------------------------
# CREATE BEHAVIORAL VECTOR SIGNATURE
# ------------------------------------------------------------

print("\nCreating behavioral vector signatures...")

# Convert binary features to compact strings
df["_behavior_signature"] = (
    df[FEATURE_COLS]
    .astype(np.uint8)
    .astype(str)
    .agg("".join, axis=1)
)

# ------------------------------------------------------------
# DUPLICATE GROUP ANALYSIS
# ------------------------------------------------------------

group_sizes = df["_behavior_signature"].value_counts()

duplicate_groups = group_sizes[group_sizes > 1]

print("\n" + "=" * 70)
print("DUPLICATE VECTOR SUMMARY")
print("=" * 70)

print(f"Total samples: {len(df):,}")
print(f"Unique behavioral vectors: {group_sizes.size:,}")
print(f"Duplicate behavioral groups: {len(duplicate_groups):,}")
print(
    f"Samples belonging to duplicate groups: "
    f"{duplicate_groups.sum():,}"
)

print(
    f"Samples with unique behavioral vectors: "
    f"{(group_sizes == 1).sum():,}"
)

print(
    f"Maximum samples sharing one vector: "
    f"{group_sizes.max():,}"
)

# ------------------------------------------------------------
# LABEL CONSISTENCY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING LABEL CONSISTENCY")
print("=" * 70)

label_counts = (
    df.groupby("_behavior_signature")[TARGET_COL]
    .nunique()
)

conflicting_vectors = label_counts[label_counts > 1]

consistent_duplicate_groups = (
    label_counts[
        (label_counts == 1) &
        (group_sizes > 1)
    ]
)

print(
    f"Duplicate groups with SAME label: "
    f"{len(consistent_duplicate_groups):,}"
)

print(
    f"Duplicate groups with CONFLICTING labels: "
    f"{len(conflicting_vectors):,}"
)

if len(conflicting_vectors) > 0:

    print("\nWARNING: Conflicting behavioral vectors found!")

    conflict_examples = (
        df[
            df["_behavior_signature"]
            .isin(conflicting_vectors.index)
        ]
        [[ID_COL, TARGET_COL]]
        .copy()
    )

    conflict_examples["_behavior_signature"] = (
        df.loc[
            conflict_examples.index,
            "_behavior_signature"
        ]
    )

    print("\nExample conflicts:")
    print(conflict_examples.head(20).to_string(index=False))

else:

    print("\nPASS: No behavioral vector has multiple labels.")

# ------------------------------------------------------------
# DUPLICATE VECTOR DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE GROUP SIZE DISTRIBUTION")
print("=" * 70)

print(
    duplicate_groups.describe()
)

print("\nLargest duplicate groups:")

largest_groups = duplicate_groups.head(20)

print(largest_groups.to_string())

# ------------------------------------------------------------
# CLASS DISTRIBUTION INSIDE DUPLICATE VECTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE VECTORS BY CLASS")
print("=" * 70)

duplicate_df = df[
    df["_behavior_signature"].isin(duplicate_groups.index)
]

print(
    duplicate_df[TARGET_COL]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# SAVE CONFLICT REPORT
# ------------------------------------------------------------

if len(conflicting_vectors) > 0:

    conflict_report = df[
        df["_behavior_signature"]
        .isin(conflicting_vectors.index)
    ].copy()

    conflict_report = conflict_report.drop(
        columns=["_behavior_signature"]
    )

    conflict_report.to_csv(
        "/kaggle/working/Behavioral_Vector_Label_Conflicts.csv",
        index=False
    )

    print(
        "\nConflict report saved to:"
        "\n/kaggle/working/Behavioral_Vector_Label_Conflicts.csv"
    )

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

df.drop(
    columns=["_behavior_signature"],
    inplace=True
)

print("\n" + "=" * 70)
print("DUPLICATE VECTOR ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

print("=" * 70)
print("ANALYZING CONFLICTING BEHAVIORAL VECTORS")
print("=" * 70)

df = pd.read_csv(DATA_PATH)

ID_COL = "SHA256"
TARGET_COL = "Type"

FEATURE_COLS = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print(f"\nDataset: {df.shape}")
print(f"Behavioral features: {len(FEATURE_COLS)}")

# ------------------------------------------------------------
# CREATE SIGNATURE
# ------------------------------------------------------------

print("\nCreating behavioral signatures...")

df["_signature"] = (
    df[FEATURE_COLS]
    .astype(np.uint8)
    .astype(str)
    .agg("".join, axis=1)
)

# ------------------------------------------------------------
# FIND CONFLICTING VECTORS
# ------------------------------------------------------------

label_counts = (
    df.groupby("_signature")[TARGET_COL]
    .nunique()
)

conflicting_signatures = label_counts[
    label_counts > 1
].index

print(
    f"\nConflicting behavioral vectors: "
    f"{len(conflicting_signatures):,}"
)

conflict_df = df[
    df["_signature"].isin(conflicting_signatures)
].copy()

# ------------------------------------------------------------
# CLASS COMBINATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFLICTING CLASS COMBINATIONS")
print("=" * 70)

class_combinations = (
    conflict_df
    .groupby("_signature")[TARGET_COL]
    .apply(lambda x: tuple(sorted(x.unique())))
)

combination_counts = (
    class_combinations
    .value_counts()
)

print(
    combination_counts.head(30).to_string()
)

# ------------------------------------------------------------
# NUMBER OF CLASSES PER VECTOR
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMBER OF CLASSES PER CONFLICTING VECTOR")
print("=" * 70)

classes_per_vector = (
    conflict_df
    .groupby("_signature")[TARGET_COL]
    .nunique()
)

print(
    classes_per_vector.value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# SAMPLE COUNTS PER CLASS IN CONFLICTING VECTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLES INVOLVED IN CONFLICTING VECTORS BY CLASS")
print("=" * 70)

print(
    conflict_df[TARGET_COL]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# MOST AMBIGUOUS VECTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MOST AMBIGUOUS VECTORS")
print("=" * 70)

ambiguity_table = (
    conflict_df
    .groupby("_signature")
    .agg(
        Samples=(TARGET_COL, "size"),
        Classes=(TARGET_COL, "nunique"),
        Class_List=(
            TARGET_COL,
            lambda x: ",".join(
                map(str, sorted(x.unique()))
            )
        )
    )
    .sort_values(
        ["Classes", "Samples"],
        ascending=False
    )
)

print(
    ambiguity_table.head(30).to_string()
)

# ------------------------------------------------------------
# CONFLICTING SAMPLES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFLICTING SAMPLE COUNT")
print("=" * 70)

print(
    f"Total samples involved: "
    f"{len(conflict_df):,}"
)

print(
    f"Percentage of complete dataset: "
    f"{len(conflict_df) / len(df) * 100:.2f}%"
)

# ------------------------------------------------------------
# SAVE ANALYSIS
# ------------------------------------------------------------

conflict_report = (
    conflict_df[
        [ID_COL, TARGET_COL]
    ]
    .copy()
)

conflict_report.to_csv(
    "/kaggle/working/Conflicting_Vector_Samples.csv",
    index=False
)

ambiguity_table.to_csv(
    "/kaggle/working/Conflicting_Vector_Summary.csv"
)

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

del df
del conflict_df

print("\n" + "=" * 70)
print("CONFLICT ANALYSIS COMPLETE")
print("=" * 70)

print("\nSaved:")
print("/kaggle/working/Conflicting_Vector_Samples.csv")
print("/kaggle/working/Conflicting_Vector_Summary.csv")

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# ============================================================
# DOMINANT BEHAVIORAL VECTOR INVESTIGATION
# ============================================================

print("=" * 70)
print("DOMINANT BEHAVIORAL VECTOR INVESTIGATION")
print("=" * 70)

DATA_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

# ------------------------------------------------------------
# 2. IDENTIFY COLUMNS
# ------------------------------------------------------------

ID_COL = "SHA256"
TARGET_COL = "Type"

feature_cols = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print(f"Behavioral features: {len(feature_cols)}")

# ------------------------------------------------------------
# 3. CREATE BEHAVIORAL SIGNATURE
# ------------------------------------------------------------

print("\nCreating behavioral signatures...")

X = df[feature_cols].astype(np.uint8)

# Convert each row into a compact binary signature
signatures = X.astype(str).agg("".join, axis=1)

df["_behavior_signature"] = signatures

# ------------------------------------------------------------
# 4. FIND LARGEST VECTOR
# ------------------------------------------------------------

print("\nFinding largest behavioral vector...")

signature_counts = df["_behavior_signature"].value_counts()

dominant_signature = signature_counts.index[0]
dominant_count = signature_counts.iloc[0]

print("\n" + "=" * 70)
print("DOMINANT VECTOR")
print("=" * 70)

print(f"Samples sharing dominant vector: {dominant_count:,}")
print(f"Percentage of dataset: {dominant_count / len(df) * 100:.2f}%")

# ------------------------------------------------------------
# 5. GET SAMPLES USING DOMINANT VECTOR
# ------------------------------------------------------------

dominant_df = df[
    df["_behavior_signature"] == dominant_signature
].copy()

print("\nDominant vector samples:")
print(len(dominant_df))

# ------------------------------------------------------------
# 6. ACTIVE FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTIVE FEATURES IN DOMINANT VECTOR")
print("=" * 70)

active_features = [
    feature_cols[i]
    for i, value in enumerate(dominant_signature)
    if value == "1"
]

inactive_features = [
    feature_cols[i]
    for i, value in enumerate(dominant_signature)
    if value == "0"
]

print(f"\nActive features   : {len(active_features)}")
print(f"Inactive features : {len(inactive_features)}")

print("\nActive features:")

if active_features:
    for feature in active_features:
        print(f"  {feature}")
else:
    print("  NONE")

# ------------------------------------------------------------
# 7. FEATURE SPARSITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DOMINANT VECTOR SPARSITY")
print("=" * 70)

print(f"Total behavioral features : {len(feature_cols)}")
print(f"Active features           : {len(active_features)}")
print(f"Inactive features         : {len(inactive_features)}")

sparsity = (
    len(inactive_features) /
    len(feature_cols) *
    100
)

print(f"Sparsity                   : {sparsity:.2f}%")
print(f"Feature presence           : {100 - sparsity:.2f}%")

# ------------------------------------------------------------
# 8. TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DOMINANT VECTOR TARGET DISTRIBUTION")
print("=" * 70)

dominant_target = dominant_df[TARGET_COL].value_counts().sort_index()

dominant_distribution = pd.DataFrame({
    "Samples": dominant_target,
    "Percentage": (
        dominant_target /
        dominant_count *
        100
    ).round(2)
})

print(dominant_distribution)

# ------------------------------------------------------------
# 9. COMPARE WITH COMPLETE DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DOMINANT VECTOR VS COMPLETE DATASET")
print("=" * 70)

overall_target = df[TARGET_COL].value_counts().sort_index()

comparison = pd.DataFrame({
    "Dominant_Vector_%": (
        dominant_target /
        dominant_count *
        100
    ).round(2),

    "Complete_Dataset_%": (
        overall_target /
        len(df) *
        100
    ).round(2)
}).fillna(0)

print(comparison)

# ------------------------------------------------------------
# 10. CLASS ENTROPY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS AMBIGUITY")
print("=" * 70)

probabilities = (
    dominant_target /
    dominant_count
).values

entropy = -np.sum(
    probabilities *
    np.log2(probabilities)
)

print(f"Number of classes present : {len(dominant_target)}")
print(f"Class entropy             : {entropy:.4f} bits")

if len(dominant_target) == 1:
    print("Interpretation: PURE VECTOR")
elif len(dominant_target) == 2:
    print("Interpretation: TWO-CLASS CONFLICT")
else:
    print("Interpretation: MULTI-CLASS CONFLICT")

# ------------------------------------------------------------
# 11. SAMPLE IDS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXAMPLE SHA256 VALUES")
print("=" * 70)

print(
    dominant_df[
        [ID_COL, TARGET_COL]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 12. SAVE DOMINANT VECTOR SAMPLES
# ------------------------------------------------------------

OUTPUT_PATH = (
    "/kaggle/working/"
    "Dominant_Behavioral_Vector_Samples.csv"
)

dominant_df.drop(
    columns=["_behavior_signature"],
    inplace=True
)

dominant_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved dominant vector samples:")
print(OUTPUT_PATH)

# ------------------------------------------------------------
# 13. SAVE ACTIVE FEATURES
# ------------------------------------------------------------

feature_report = pd.DataFrame({
    "Feature": feature_cols,
    "Active_In_Dominant_Vector": [
        int(x) for x in dominant_signature
    ]
})

FEATURE_OUTPUT = (
    "/kaggle/working/"
    "Dominant_Vector_Feature_Analysis.csv"
)

feature_report.to_csv(
    FEATURE_OUTPUT,
    index=False
)

print("\nSaved feature analysis:")
print(FEATURE_OUTPUT)

# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DOMINANT VECTOR INVESTIGATION COMPLETE")
print("=" * 70)

print(f"""
Dataset samples              : {len(df):,}
Behavioral features          : {len(feature_cols)}
Dominant vector samples      : {dominant_count:,}
Dataset percentage           : {dominant_count / len(df) * 100:.2f}%

Active features              : {len(active_features)}
Inactive features            : {len(inactive_features)}
Dominant vector sparsity     : {sparsity:.2f}%

Classes represented          : {len(dominant_target)}
Class entropy                : {entropy:.4f}

Output files:
- {OUTPUT_PATH}
- {FEATURE_OUTPUT}
""")

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

print("=" * 70)
print("DOMINANT VECTOR SOURCE DATASET INVESTIGATION")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

FINAL_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

# CHANGE THESE ONLY IF YOUR ORIGINAL PATHS ARE DIFFERENT
API_PATH = "/kaggle/input/YOUR_API_DATASET.csv"
DLL_PATH = "/kaggle/input/YOUR_DLL_DATASET.csv"

# ============================================================
# LOAD FINAL DATASET
# ============================================================

df = pd.read_csv(FINAL_PATH)

behavioral_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

print(f"\nFinal dataset: {df.shape}")
print(f"Behavioral features: {len(behavioral_cols)}")

# ============================================================
# CREATE SIGNATURE
# ============================================================

df["_behavior_signature"] = (
    df[behavioral_cols]
    .astype(np.uint8)
    .astype(str)
    .agg("".join, axis=1)
)

signature_counts = df["_behavior_signature"].value_counts()

dominant_signature = signature_counts.index[0]
dominant_count = signature_counts.iloc[0]

print("\n" + "=" * 70)
print("DOMINANT VECTOR")
print("=" * 70)

print(f"Samples: {dominant_count}")
print(f"Percentage: {dominant_count / len(df) * 100:.2f}%")

dominant_df = df[
    df["_behavior_signature"] == dominant_signature
].copy()

# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("DOMINANT VECTOR CLASS DISTRIBUTION")
print("=" * 70)

print(
    dominant_df["Type"]
    .value_counts()
    .sort_index()
)

# ============================================================
# SHA256 LIST
# ============================================================

dominant_hashes = set(
    dominant_df["SHA256"].astype(str)
)

print(f"\nDominant SHA256 values: {len(dominant_hashes):,}")

# ============================================================
# IMPORTANT:
# Inspect actual columns of source datasets
# ============================================================

print("\n" + "=" * 70)
print("SOURCE DATASET INSPECTION")
print("=" * 70)

try:
    api_preview = pd.read_csv(API_PATH, nrows=5)
    print("\nAPI columns:")
    print(api_preview.columns.tolist())

except Exception as e:
    print("\nCould not load API dataset:")
    print(e)

try:
    dll_preview = pd.read_csv(DLL_PATH, nrows=5)
    print("\nDLL columns:")
    print(dll_preview.columns.tolist())

except Exception as e:
    print("\nCould not load DLL dataset:")
    print(e)

print("\n" + "=" * 70)
print("INVESTIGATION SETUP COMPLETE")
print("=" * 70)

print("""
NEXT:

We need the exact original API_PATH and DLL_PATH.

Do NOT delete any samples yet.

The purpose of this investigation is to determine whether the
dominant behavioral vector was caused by:

1. Source-data sparsity
2. Feature extraction
3. Feature selection
4. Dataset merging/alignment
5. Loss of important behavioral features
""")

In [ ]:
import os

print("=" * 70)
print("KAGGLE INPUT FILES")
print("=" * 70)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd
import numpy as np

print("=" * 70)
print("TRACING DOMINANT VECTOR TO ORIGINAL API / DLL DATA")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

FINAL_PATH = "/kaggle/working/Compact_API_DLL_Dataset.csv"

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

# ============================================================
# LOAD FINAL DATASET
# ============================================================

df = pd.read_csv(FINAL_PATH)

behavioral_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

print(f"\nFinal dataset: {df.shape}")
print(f"Behavioral features: {len(behavioral_cols)}")

# ============================================================
# FIND DOMINANT VECTOR
# ============================================================

print("\nCreating behavioral signatures...")

df["_signature"] = (
    df[behavioral_cols]
    .astype(np.uint8)
    .astype(str)
    .agg("".join, axis=1)
)

signature_counts = df["_signature"].value_counts()

dominant_signature = signature_counts.index[0]

dominant_df = df[
    df["_signature"] == dominant_signature
].copy()

dominant_hashes = set(
    dominant_df["SHA256"].astype(str)
)

print("\n" + "=" * 70)
print("DOMINANT VECTOR")
print("=" * 70)

print(f"Samples: {len(dominant_df):,}")
print(f"Percentage: {len(dominant_df) / len(df) * 100:.2f}%")

print("\nClass distribution:")
print(
    dominant_df["Type"]
    .value_counts()
    .sort_index()
)

# ============================================================
# LOAD SOURCE DATASETS
# ============================================================

print("\n" + "=" * 70)
print("LOADING ORIGINAL API DATASET")
print("=" * 70)

api_df = pd.read_csv(API_PATH)

print(f"API shape: {api_df.shape}")
print("\nAPI columns:")
print(api_df.columns.tolist())

print("\n" + "=" * 70)
print("LOADING ORIGINAL DLL DATASET")
print("=" * 70)

dll_df = pd.read_csv(DLL_PATH)

print(f"DLL shape: {dll_df.shape}")
print("\nDLL columns:")
print(dll_df.columns.tolist())

# ============================================================
# NORMALIZE SHA256
# ============================================================

print("\n" + "=" * 70)
print("SHA256 MATCHING")
print("=" * 70)

api_df["SHA256"] = api_df["SHA256"].astype(str)
dll_df["SHA256"] = dll_df["SHA256"].astype(str)

api_hashes = set(api_df["SHA256"])
dll_hashes = set(dll_df["SHA256"])

api_matches = dominant_hashes.intersection(api_hashes)
dll_matches = dominant_hashes.intersection(dll_hashes)

print(f"Dominant hashes: {len(dominant_hashes):,}")

print(f"Found in original API dataset: {len(api_matches):,}")
print(f"Found in original DLL dataset: {len(dll_matches):,}")

print(
    f"Missing from API dataset: "
    f"{len(dominant_hashes - api_hashes):,}"
)

print(
    f"Missing from DLL dataset: "
    f"{len(dominant_hashes - dll_hashes):,}"
)

# ============================================================
# ORIGINAL API BEHAVIOR
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL API BEHAVIOR OF DOMINANT VECTOR")
print("=" * 70)

dominant_api = api_df[
    api_df["SHA256"].isin(dominant_hashes)
].copy()

print(f"API samples found: {len(dominant_api):,}")

# Identify API feature columns
api_meta = {
    "SHA256",
    "Type"
}

api_features = [
    c for c in api_df.columns
    if c not in api_meta
]

print(f"Original API features: {len(api_features):,}")

# Count feature presence
if len(dominant_api) > 0 and len(api_features) > 0:

    api_feature_counts = (
        dominant_api[api_features]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum()
        .sort_values(ascending=False)
    )

    api_feature_counts = api_feature_counts[
        api_feature_counts > 0
    ]

    print("\nMost common ORIGINAL API features:")
    print(
        api_feature_counts.head(30)
    )

# ============================================================
# ORIGINAL DLL BEHAVIOR
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL DLL BEHAVIOR OF DOMINANT VECTOR")
print("=" * 70)

dominant_dll = dll_df[
    dll_df["SHA256"].isin(dominant_hashes)
].copy()

print(f"DLL samples found: {len(dominant_dll):,}")

dll_meta = {
    "SHA256",
    "Type"
}

dll_features = [
    c for c in dll_df.columns
    if c not in dll_meta
]

print(f"Original DLL features: {len(dll_features):,}")

if len(dominant_dll) > 0 and len(dll_features) > 0:

    dll_feature_counts = (
        dominant_dll[dll_features]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum()
        .sort_values(ascending=False)
    )

    dll_feature_counts = dll_feature_counts[
        dll_feature_counts > 0
    ]

    print("\nMost common ORIGINAL DLL features:")
    print(
        dll_feature_counts.head(30)
    )

# ============================================================
# SOURCE LABEL CHECK
# ============================================================

print("\n" + "=" * 70)
print("SOURCE LABEL DISTRIBUTION")
print("=" * 70)

if "Type" in dominant_api.columns:
    print("\nAPI:")
    print(
        dominant_api["Type"]
        .value_counts()
        .sort_index()
    )

if "Type" in dominant_dll.columns:
    print("\nDLL:")
    print(
        dominant_dll["Type"]
        .value_counts()
        .sort_index()
    )

# ============================================================
# SAVE INVESTIGATION RESULTS
# ============================================================

dominant_api.to_csv(
    "/kaggle/working/Dominant_Vector_Original_API.csv",
    index=False
)

dominant_dll.to_csv(
    "/kaggle/working/Dominant_Vector_Original_DLL.csv",
    index=False
)

print("\n" + "=" * 70)
print("TRACE COMPLETE")
print("=" * 70)

print("""
Saved:

/kaggle/working/Dominant_Vector_Original_API.csv
/kaggle/working/Dominant_Vector_Original_DLL.csv

NEXT:
We will compare the ORIGINAL behavioral richness against the
current 227-feature representation.

Do NOT delete samples yet.
Do NOT rebuild the dataset yet.
""")

In [ ]:
import os

print("=" * 70)
print("CHECKING KAGGLE WORKING DIRECTORY")
print("=" * 70)

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
print("\n" + "=" * 70)
print("CHECKING KAGGLE INPUT DIRECTORY")
print("=" * 70)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import os
import glob

print("=" * 70)
print("SEARCHING FOR PREVIOUS FEATURE-SELECTION OUTPUTS")
print("=" * 70)

patterns = [
    "**/API_Top200_MI.csv",
    "**/API_MI_Selected.csv",
    "**/DLL_Candidate_Features.csv",
    "**/DLL_Candidate_Features_MI.csv",
    "**/Compact_API_DLL_Dataset.csv",
]

found = []

for pattern in patterns:
    matches = glob.glob("/kaggle/**/*", recursive=True)
    break

for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if f in [
            "API_Top200_MI.csv",
            "API_MI_Selected.csv",
            "DLL_Candidate_Features.csv",
            "DLL_Candidate_Features_MI.csv",
            "Compact_API_DLL_Dataset.csv",
        ]:
            path = os.path.join(root, f)
            found.append(path)
            print(path)

print("\n" + "=" * 70)
if found:
    print(f"FOUND {len(found)} PREVIOUS OUTPUT FILE(S)")
else:
    print("NO PREVIOUS OUTPUT FILES FOUND")
    print("We will reconstruct them from the original datasets.")
    

In [ ]:
import pandas as pd
import os

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

print("=" * 70)
print("ORIGINAL DATASET INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# API
# ------------------------------------------------------------

print("\nAPI DATASET")
print("-" * 70)

api_sample = pd.read_csv(API_PATH, nrows=5)

print("Shape preview:", api_sample.shape)
print("Columns:", api_sample.columns.tolist())
print("\nFirst 5 rows:")
print(api_sample.head())

# ------------------------------------------------------------
# DLL
# ------------------------------------------------------------

print("\nDLL DATASET")
print("-" * 70)

dll_sample = pd.read_csv(DLL_PATH, nrows=5)

print("Shape preview:", dll_sample.shape)
print("Columns:", dll_sample.columns.tolist())
print("\nFirst 5 rows:")
print(dll_sample.head())

# ------------------------------------------------------------
# FILE SIZES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILE INFORMATION")
print("=" * 70)

for path in [API_PATH, DLL_PATH]:
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{os.path.basename(path):30s} {size_mb:.2f} MB")

print("\nInspection complete.")

In [ ]:
import os
import pandas as pd

BASE = "/kaggle/input/datasets/joebeachcapital/windows-malwares"

API_PATH = os.path.join(BASE, "API_Functions.csv")
DLL_PATH = os.path.join(BASE, "DLLs_Imported.csv")
PE_SECTION_PATH = os.path.join(BASE, "PE_Section.csv")
PE_HEADER_PATH = os.path.join(BASE, "PE_Header.csv")

print("=" * 70)
print("INSPECTING ORIGINAL WINDOWS MALWARE DATASETS")
print("=" * 70)

for path in [API_PATH, DLL_PATH, PE_SECTION_PATH, PE_HEADER_PATH]:
    print(f"\n{os.path.basename(path)}")
    print(f"Exists: {os.path.exists(path)}")
    if os.path.exists(path):
        print(f"Size : {os.path.getsize(path) / (1024**2):.2f} MB")

print("\n" + "=" * 70)
print("API DATASET SCHEMA")
print("=" * 70)

api_sample = pd.read_csv(API_PATH, nrows=5)

print("Shape sample:", api_sample.shape)
print("Columns:")
for col in api_sample.columns:
    print(" ", col)

print("\nFirst 5 rows:")
display(api_sample.head())

print("\n" + "=" * 70)
print("DLL DATASET SCHEMA")
print("=" * 70)

dll_sample = pd.read_csv(DLL_PATH, nrows=5)

print("Shape sample:", dll_sample.shape)
print("Columns:")
for col in dll_sample.columns:
    print(" ", col)

print("\nFirst 5 rows:")
display(dll_sample.head())

print("\n" + "=" * 70)
print("PE SECTION SCHEMA")
print("=" * 70)

pe_section_sample = pd.read_csv(PE_SECTION_PATH, nrows=5)

print("Columns:")
for col in pe_section_sample.columns:
    print(" ", col)

print("\nFirst 5 rows:")
display(pe_section_sample.head())

print("\n" + "=" * 70)
print("PE HEADER SCHEMA")
print("=" * 70)

pe_header_sample = pd.read_csv(PE_HEADER_PATH, nrows=5)

print("Columns:")
for col in pe_header_sample.columns:
    print(" ", col)

print("\nFirst 5 rows:")
display(pe_header_sample.head())

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# PATHS
# ============================================================

BASE = "/kaggle/input/datasets/joebeachcapital/windows-malwares"

API_PATH = os.path.join(BASE, "API_Functions.csv")
DLL_PATH = os.path.join(BASE, "DLLs_Imported.csv")

# ============================================================
# TARGET DOMINANT SAMPLES
# ============================================================

DOMINANT_SHA_PATH = "/kaggle/working/Dominant_Behavioral_Vector_Samples.csv"

print("=" * 70)
print("TRACING DOMINANT VECTOR IN ORIGINAL DATA")
print("=" * 70)

# ------------------------------------------------------------
# Load SHA256 values belonging to dominant vector
# ------------------------------------------------------------

dominant_df = pd.read_csv(DOMINANT_SHA_PATH)

dominant_hashes = set(
    dominant_df["SHA256"].astype(str).str.lower()
)

print("Dominant vector SHA256 samples:", len(dominant_hashes))

# ============================================================
# STEP 1 — STREAM ORIGINAL API DATA
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: SEARCHING ORIGINAL API DATA")
print("=" * 70)

api_matches = []

for chunk in pd.read_csv(
    API_PATH,
    chunksize=2500,
    low_memory=False
):
    
    chunk["SHA256"] = chunk["SHA256"].astype(str).str.lower()
    
    matched = chunk[
        chunk["SHA256"].isin(dominant_hashes)
    ]
    
    if len(matched) > 0:
        api_matches.append(matched)
    
    print(
        f"Processed: {len(chunk):,} | "
        f"Matched so far: {sum(len(x) for x in api_matches):,}"
    )

api_dom = pd.concat(
    api_matches,
    ignore_index=True
)

print("\nOriginal API dominant samples found:", len(api_dom))

# ============================================================
# STEP 2 — STREAM ORIGINAL DLL DATA
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: SEARCHING ORIGINAL DLL DATA")
print("=" * 70)

dll_matches = []

for chunk in pd.read_csv(
    DLL_PATH,
    chunksize=5000,
    low_memory=False
):
    
    chunk["SHA256"] = chunk["SHA256"].astype(str).str.lower()
    
    matched = chunk[
        chunk["SHA256"].isin(dominant_hashes)
    ]
    
    if len(matched) > 0:
        dll_matches.append(matched)

dll_dom = pd.concat(
    dll_matches,
    ignore_index=True
)

print("Original DLL dominant samples found:", len(dll_dom))

# ============================================================
# STEP 3 — VERIFY ALIGNMENT
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: VERIFYING DOMINANT SAMPLE ALIGNMENT")
print("=" * 70)

api_hashes = set(api_dom["SHA256"])
dll_hashes = set(dll_dom["SHA256"])

print("Dominant SHA256 count :", len(dominant_hashes))
print("API matches           :", len(api_hashes))
print("DLL matches           :", len(dll_hashes))
print("API ∩ DLL             :", len(api_hashes & dll_hashes))

print("Missing from API:", len(dominant_hashes - api_hashes))
print("Missing from DLL:", len(dominant_hashes - dll_hashes))

# ============================================================
# STEP 4 — ORIGINAL API FEATURE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: ORIGINAL API BEHAVIOR")
print("=" * 70)

api_feature_cols = [
    c for c in api_dom.columns
    if c not in ["SHA256", "Type"]
]

api_values = (
    api_dom[api_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

api_presence = api_values.sum(axis=1)

print("Original API features:", len(api_feature_cols))

print("\nAPI active-feature statistics:")
print(api_presence.describe())

print("\nAPI active features per dominant sample:")
print(
    api_presence.value_counts()
    .sort_index()
    .head(30)
)

# ============================================================
# STEP 5 — ORIGINAL DLL FEATURE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: ORIGINAL DLL BEHAVIOR")
print("=" * 70)

dll_feature_cols = [
    c for c in dll_dom.columns
    if c not in ["SHA256", "Type"]
]

dll_values = (
    dll_dom[dll_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

dll_presence = dll_values.sum(axis=1)

print("Original DLL features:", len(dll_feature_cols))

print("\nDLL active-feature statistics:")
print(dll_presence.describe())

# ============================================================
# STEP 6 — MOST COMMON ORIGINAL API FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 6: MOST COMMON API FEATURES")
print("=" * 70)

api_counts = api_values.sum(axis=0).sort_values(ascending=False)

print(
    api_counts.head(30)
)

# ============================================================
# STEP 7 — MOST COMMON ORIGINAL DLL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 7: MOST COMMON DLL FEATURES")
print("=" * 70)

dll_counts = dll_values.sum(axis=0).sort_values(ascending=False)

print(
    dll_counts.head(30)
)

# ============================================================
# STEP 8 — LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("STEP 8: LABEL DISTRIBUTION")
print("=" * 70)

print(
    api_dom["Type"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 70)
print("ORIGINAL DOMINANT VECTOR TRACE COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# PATHS
# ============================================================

BASE = "/kaggle/input/datasets/joebeachcapital/windows-malwares"

API_PATH = os.path.join(BASE, "API_Functions.csv")
DLL_PATH = os.path.join(BASE, "DLLs_Imported.csv")

print("=" * 70)
print("RECONSTRUCTING DOMINANT VECTOR FROM ORIGINAL DATA")
print("=" * 70)

# ============================================================
# COMPACT FEATURES
# ============================================================

# These are the exact features that defined the dominant vector
# in our 227-feature compact dataset.

API_SELECTED = [
    # We need the exact 200 API features from the previous
    # feature-selection stage.
]

DLL_SELECTED = [
    # We need the exact 27 DLL features from the previous
    # feature-selection stage.
]

print("API selected features:", len(API_SELECTED))
print("DLL selected features:", len(DLL_SELECTED))

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

# ============================================================
# PATHS
# ============================================================

BASE = "/kaggle/input/datasets/joebeachcapital/windows-malwares"

API_PATH = os.path.join(BASE, "API_Functions.csv")
DLL_PATH = os.path.join(BASE, "DLLs_Imported.csv")

WORK = "/kaggle/working"

print("=" * 70)
print("RECONSTRUCTING API FEATURE SELECTION")
print("=" * 70)

# ============================================================
# PARAMETERS
# ============================================================

MIN_FREQUENCY = 5
MI_THRESHOLD = 0.01
TOP_N_API = 200

print("Minimum API frequency:", MIN_FREQUENCY)
print("MI threshold:", MI_THRESHOLD)
print("Top APIs:", TOP_N_API)

# ============================================================
# LOAD API HEADER ONLY
# ============================================================

api_header = pd.read_csv(
    API_PATH,
    nrows=0
)

api_columns = list(api_header.columns)

API_FEATURES = [
    c for c in api_columns
    if c not in ["SHA256", "Type"]
]

print("\nTotal original API features:", len(API_FEATURES))

# ============================================================
# STREAM DATA TO CALCULATE FEATURE FREQUENCY
# ============================================================

print("\nCalculating API frequencies...")

frequency = pd.Series(
    0,
    index=API_FEATURES,
    dtype="int64"
)

total_rows = 0

for chunk in pd.read_csv(
    API_PATH,
    usecols=API_FEATURES,
    chunksize=1000,
    dtype=np.uint8,
    low_memory=False
):

    frequency += chunk.sum(axis=0)

    total_rows += len(chunk)

    if total_rows % 5000 == 0:
        print(
            f"Processed: {total_rows:,} | "
            f"Qualified so far: {(frequency >= MIN_FREQUENCY).sum():,}"
        )

    del chunk
    gc.collect()

print("\n" + "=" * 70)
print("API FREQUENCY ANALYSIS COMPLETE")
print("=" * 70)

print("Total API rows:", total_rows)
print("Original API features:", len(API_FEATURES))

qualified_api = frequency[
    frequency >= MIN_FREQUENCY
].index.tolist()

print("Frequency-qualified APIs:", len(qualified_api))

# Save frequency results
freq_df = pd.DataFrame({
    "API": frequency.index,
    "Frequency": frequency.values
})

freq_df = freq_df.sort_values(
    "Frequency",
    ascending=False
)

freq_df.to_csv(
    os.path.join(WORK, "API_Feature_Frequencies_Reconstructed.csv"),
    index=False
)

# Save qualified list
pd.DataFrame({
    "API": qualified_api
}).to_csv(
    os.path.join(WORK, "API_Frequency_Selected_Reconstructed.csv"),
    index=False
)

print("\nSaved:")
print("/kaggle/working/API_Feature_Frequencies_Reconstructed.csv")
print("/kaggle/working/API_Frequency_Selected_Reconstructed.csv")

print("\nTop 30 frequency APIs:")
print(freq_df.head(30).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

FREQ_PATH = "/kaggle/working/API_Feature_Frequencies_Reconstructed.csv"

freq_df = pd.read_csv(FREQ_PATH)

freq = freq_df["Frequency"]

print("=" * 70)
print("API FREQUENCY DISTRIBUTION")
print("=" * 70)

print("Total API features:", len(freq))

print("\nFrequency statistics:")
print(freq.describe(percentiles=[
    0.50, 0.60, 0.70, 0.75, 0.80,
    0.85, 0.90, 0.95, 0.99
]))

print("\n" + "=" * 70)
print("CANDIDATE FREQUENCY CUTOFFS")
print("=" * 70)

for cutoff in [
    5, 10, 15, 20, 25, 30, 40, 50,
    75, 100, 150, 200, 250, 300,
    400, 500, 750, 1000
]:
    count = (freq >= cutoff).sum()
    print(f"Frequency >= {cutoff:<4}: {count:>5} APIs")

print("\n" + "=" * 70)
print("CUTOFFS NEAR 3079")
print("=" * 70)

# Find the frequencies around the 3079th-ranked feature
sorted_freq = np.sort(freq.values)[::-1]

for rank in [3000, 3050, 3070, 3079, 3080, 3090, 3100]:
    if rank <= len(sorted_freq):
        print(
            f"Rank {rank:4}: "
            f"frequency = {sorted_freq[rank-1]}"
        )

print("\n" + "=" * 70)
print("DISTRIBUTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
import os
import gc

# ============================================================
# PATHS
# ============================================================

BASE = "/kaggle/input/datasets/joebeachcapital/windows-malwares"

API_PATH = os.path.join(BASE, "API_Functions.csv")

WORK = "/kaggle/working"

# ============================================================
# PARAMETERS — RECONSTRUCTED FROM PREVIOUS RESULTS
# ============================================================

FREQUENCY_THRESHOLD = 20
MI_THRESHOLD = 0.01
TOP_N = 200

print("=" * 70)
print("RECONSTRUCTING API MI FEATURE SELECTION")
print("=" * 70)

# ============================================================
# LOAD FREQUENCY RESULTS
# ============================================================

freq_df = pd.read_csv(
    os.path.join(
        WORK,
        "API_Feature_Frequencies_Reconstructed.csv"
    )
)

qualified_api = freq_df.loc[
    freq_df["Frequency"] >= FREQUENCY_THRESHOLD,
    "API"
].tolist()

print("Frequency threshold:", FREQUENCY_THRESHOLD)
print("Frequency-qualified APIs:", len(qualified_api))

# ============================================================
# LOAD LABELS + SELECTED FEATURES
# ============================================================

print("\nLoading labels and selected API features...")

usecols = ["Type"] + qualified_api

parts = []

for chunk in pd.read_csv(
    API_PATH,
    usecols=usecols,
    chunksize=2500,
    low_memory=False
):
    parts.append(chunk)

api_selected_df = pd.concat(
    parts,
    ignore_index=True
)

del parts
gc.collect()

print("Feature matrix shape:", api_selected_df.shape)

# ============================================================
# EXTRACT X / y
# ============================================================

y = api_selected_df["Type"].values

X = api_selected_df[
    qualified_api
].astype(np.uint8).values

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("y shape:", y.shape)

del api_selected_df
gc.collect()

# ============================================================
# MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING MUTUAL INFORMATION")
print("=" * 70)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("\nThis may take some time...")

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

# ============================================================
# CREATE MI TABLE
# ============================================================

mi_df = pd.DataFrame({
    "API": qualified_api,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# MI STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("MUTUAL INFORMATION STATISTICS")
print("=" * 70)

print(
    mi_df["MI_Score"].describe()
)

# ============================================================
# VERIFY MI >= 0.01
# ============================================================

mi_selected = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

print("\n" + "=" * 70)
print("MI FILTER RESULT")
print("=" * 70)

print("Before MI filtering:", len(mi_df))
print("After MI filtering :", len(mi_selected))
print(
    "Percentage retained :",
    round(
        len(mi_selected) / len(mi_df) * 100,
        2
    ),
    "%"
)

# ============================================================
# TOP 200
# ============================================================

top200 = mi_selected.head(TOP_N).copy()

print("\n" + "=" * 70)
print("TOP 200 API FEATURES")
print("=" * 70)

print("Selected:", len(top200))

print("\nTop 30:")
print(
    top200.head(30).to_string(index=False)
)

# ============================================================
# SAVE
# ============================================================

mi_df.to_csv(
    os.path.join(
        WORK,
        "API_Candidate_Features_MI_Reconstructed.csv"
    ),
    index=False
)

mi_selected.to_csv(
    os.path.join(
        WORK,
        "API_MI_Selected_Reconstructed.csv"
    ),
    index=False
)

top200.to_csv(
    os.path.join(
        WORK,
        "API_Top200_MI_Reconstructed.csv"
    ),
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    "/kaggle/working/API_Candidate_Features_MI_Reconstructed.csv"
)

print(
    "/kaggle/working/API_MI_Selected_Reconstructed.csv"
)

print(
    "/kaggle/working/API_Top200_MI_Reconstructed.csv"
)

print("\n" + "=" * 70)
print("API MI RECONSTRUCTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# PATH
# ============================================================

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

WORK = "/kaggle/working"

print("=" * 70)
print("RECONSTRUCTING DLL FEATURE SELECTION")
print("=" * 70)

# ============================================================
# LOAD DLL DATASET
# ============================================================

dll_df = pd.read_csv(
    DLL_PATH,
    low_memory=False
)

print("DLL dataset shape:", dll_df.shape)

# ============================================================
# BASIC INFORMATION
# ============================================================

dll_features = [
    c for c in dll_df.columns
    if c not in ["SHA256", "Type"]
]

print("Original DLL features:", len(dll_features))
print("Samples:", len(dll_df))

# ============================================================
# FEATURE TYPES
# ============================================================

dll_numeric = dll_df[dll_features].apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

# ============================================================
# FREQUENCY ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("DLL FEATURE FREQUENCY ANALYSIS")
print("=" * 70)

dll_frequency = dll_numeric.sum(axis=0)

print("\nFrequency statistics:")
print(dll_frequency.describe())

print("\n" + "=" * 70)
print("DLL FREQUENCY CUTOFFS")
print("=" * 70)

for cutoff in [
    1, 2, 3, 5, 10, 15, 20,
    25, 30, 40, 50, 75,
    100, 150, 200, 300,
    500, 1000, 2000
]:
    count = (dll_frequency >= cutoff).sum()
    print(
        f"Frequency >= {cutoff:<4}: "
        f"{count:>4} DLLs"
    )

# ============================================================
# UNIQUE VALUES / CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("DLL FEATURE QUALITY")
print("=" * 70)

constant_dll = [
    c for c in dll_features
    if dll_numeric[c].nunique() <= 1
]

print("Constant DLL features:", len(constant_dll))

# ============================================================
# MOST COMMON DLLs
# ============================================================

dll_freq_df = pd.DataFrame({
    "DLL": dll_frequency.index,
    "Frequency": dll_frequency.values
}).sort_values(
    "Frequency",
    ascending=False
)

print("\nTop 50 DLL features:")
print(
    dll_freq_df.head(50).to_string(index=False)
)

# ============================================================
# SAVE
# ============================================================

dll_freq_df.to_csv(
    os.path.join(
        WORK,
        "DLL_Feature_Frequencies_Reconstructed.csv"
    ),
    index=False
)

print("\nSaved:")
print(
    "/kaggle/working/DLL_Feature_Frequencies_Reconstructed.csv"
)

print("\n" + "=" * 70)
print("DLL FREQUENCY ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np

FREQ_PATH = "/kaggle/working/DLL_Feature_Frequencies_Reconstructed.csv"

freq_df = pd.read_csv(FREQ_PATH)

freq = freq_df["Frequency"]

print("=" * 70)
print("DLL FREQUENCY DISTRIBUTION — RECONSTRUCTION")
print("=" * 70)

print("Total DLL features:", len(freq))

print("\nFrequency statistics:")
print(
    freq.describe(
        percentiles=[
            0.50, 0.60, 0.70, 0.75,
            0.80, 0.85, 0.90,
            0.95, 0.99
        ]
    )
)

print("\n" + "=" * 70)
print("CANDIDATE FREQUENCY CUTOFFS")
print("=" * 70)

for cutoff in [
    5, 10, 15, 20, 25, 30,
    40, 50, 60, 75, 100,
    125, 150, 175, 200,
    250, 300, 400, 500
]:
    count = (freq >= cutoff).sum()

    print(
        f"Frequency >= {cutoff:<4}: "
        f"{count:>4} DLLs"
    )

print("\n" + "=" * 70)
print("RANKS NEAR 27 FEATURES")
print("=" * 70)

sorted_freq = np.sort(
    freq.values
)[::-1]

for rank in [
    20, 22, 24, 25,
    26, 27, 28, 29,
    30, 32, 35
]:
    if rank <= len(sorted_freq):
        print(
            f"Rank {rank:3}: "
            f"frequency = {sorted_freq[rank - 1]}"
        )

print("\n" + "=" * 70)
print("DLL FREQUENCY DISTRIBUTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# CONFIG
# ============================================================

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

# Start with the cutoff that gives a candidate pool close to
# the previous 27-feature final result.
FREQ_THRESHOLD = 250

# Same MI threshold used in API reconstruction
MI_THRESHOLD = 0.01

# Previous final DLL feature count
TOP_N = 27

# ============================================================
# LOAD DATA
# ============================================================

print("=" * 70)
print("RECONSTRUCTING DLL MI FEATURE SELECTION")
print("=" * 70)

df = pd.read_csv(DLL_PATH)

print("DLL dataset shape:", df.shape)

feature_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

print("Original DLL features:", len(feature_cols))
print("Samples:", len(df))

# ============================================================
# FREQUENCY FILTER
# ============================================================

print("\n" + "=" * 70)
print("APPLYING DLL FREQUENCY FILTER")
print("=" * 70)

X_full = df[feature_cols].fillna(0)

frequencies = X_full.sum(axis=0)

selected_frequency = frequencies[
    frequencies >= FREQ_THRESHOLD
].index.tolist()

print("Frequency threshold:", FREQ_THRESHOLD)
print("Frequency-qualified DLLs:", len(selected_frequency))

# ============================================================
# BUILD FEATURE MATRIX
# ============================================================

X = X_full[selected_frequency].astype(np.uint8)
y = df["Type"].astype(np.int64)

print("\nFeature matrix shape:", X.shape)
print("X shape:", X.shape)
print("X dtype:", X.dtypes.unique())
print("y shape:", y.shape)

# ============================================================
# MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING DLL MUTUAL INFORMATION")
print("=" * 70)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("\nThis may take some time...")

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

mi_df = pd.DataFrame({
    "DLL": selected_frequency,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# MI STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("DLL MUTUAL INFORMATION STATISTICS")
print("=" * 70)

print(mi_df["MI_Score"].describe())

# ============================================================
# MI FILTER
# ============================================================

mi_filtered = mi_df[
    mi_df["MI_Score"] >= MI_THRESHOLD
].copy()

print("\n" + "=" * 70)
print("DLL MI FILTER RESULT")
print("=" * 70)

print("Before MI filtering:", len(mi_df))
print("After MI filtering :", len(mi_filtered))

if len(mi_df) > 0:
    print(
        "Percentage retained:",
        round(100 * len(mi_filtered) / len(mi_df), 2),
        "%"
    )

# ============================================================
# TOP 27
# ============================================================

dll_top27 = mi_filtered.head(TOP_N).copy()

print("\n" + "=" * 70)
print("TOP 27 DLL FEATURES")
print("=" * 70)

print("Selected:", len(dll_top27))
print()

print(
    dll_top27.to_string(index=False)
)

# ============================================================
# SAVE OUTPUTS
# ============================================================

FREQ_OUTPUT = (
    "/kaggle/working/"
    "DLL_MI_Candidate_Features_Reconstructed.csv"
)

TOP_OUTPUT = (
    "/kaggle/working/"
    "DLL_Top27_MI_Reconstructed.csv"
)

mi_df.to_csv(
    FREQ_OUTPUT,
    index=False
)

dll_top27.to_csv(
    TOP_OUTPUT,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(FREQ_OUTPUT)
print(TOP_OUTPUT)

print("\n" + "=" * 70)
print("DLL MI RECONSTRUCTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import os

print("=" * 70)
print("IDENTIFYING THE MISSING 27TH DLL FEATURE")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

# ------------------------------------------------------------
# LOAD RECONSTRUCTED MI FEATURES
# ------------------------------------------------------------

mi_path = "/kaggle/working/DLL_MI_Candidate_Features_Reconstructed.csv"

mi_df = pd.read_csv(mi_path)

# Features surviving MI >= 0.01
reconstructed_26 = mi_df[
    mi_df["MI_Score"] >= 0.01
]["DLL"].tolist()

print("\nReconstructed MI-selected DLLs:", len(reconstructed_26))

# ------------------------------------------------------------
# LOAD ORIGINAL DLL DATASET
# ------------------------------------------------------------

dll_df = pd.read_csv(DLL_PATH, nrows=5)

original_dll_features = [
    c for c in dll_df.columns
    if c not in ["SHA256", "Type"]
]

print("Original DLL features:", len(original_dll_features))

# ------------------------------------------------------------
# LOAD FINAL DATASET IF AVAILABLE
# ------------------------------------------------------------

possible_final_paths = [
    "/kaggle/working/Compact_API_DLL_Dataset.csv",
    "/kaggle/input/Compact_API_DLL_Dataset.csv",
]

final_path = None

for path in possible_final_paths:
    if os.path.exists(path):
        final_path = path
        break

if final_path is None:

    print("\nFINAL DATASET NOT FOUND.")
    print("We will inspect the reconstructed candidates only.")

else:

    print("\nFinal dataset found:")
    print(final_path)

    final_df = pd.read_csv(final_path, nrows=5)

    final_behavioral = [
        c for c in final_df.columns
        if c not in ["SHA256", "Type"]
    ]

    # --------------------------------------------------------
    # IDENTIFY DLL FEATURES IN FINAL DATASET
    # --------------------------------------------------------

    final_dll_features = [
        c for c in final_behavioral
        if c.endswith(".dll") or c.endswith(".drv") or c.endswith(".ocx")
    ]

    print("\nDLL-like features in final dataset:",
          len(final_dll_features))

    # --------------------------------------------------------
    # COMPARE
    # --------------------------------------------------------

    reconstructed_set = set(reconstructed_26)
    final_dll_set = set(final_dll_features)

    missing_from_final = reconstructed_set - final_dll_set
    extra_in_final = final_dll_set - reconstructed_set

    print("\n" + "=" * 70)
    print("COMPARISON")
    print("=" * 70)

    print("\nReconstructed DLLs present in final:",
          len(reconstructed_set & final_dll_set))

    print("Reconstructed DLLs missing from final:",
          len(missing_from_final))

    if missing_from_final:
        print("\nMissing reconstructed DLLs:")
        for x in sorted(missing_from_final):
            print(" ", x)

    print("\nFinal DLLs not in reconstructed MI set:",
          len(extra_in_final))

    if extra_in_final:
        print("\nExtra final DLL features:")
        for x in sorted(extra_in_final):
            print(" ", x)

# ------------------------------------------------------------
# SHOW FREQUENCY + MI FOR ALL 29 CANDIDATES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL 29 FREQUENCY-QUALIFIED DLLs")
print("=" * 70)

candidate_df = mi_df[
    mi_df["DLL"].isin(
        mi_df["DLL"].head(29)
    )
].copy()

candidate_df["Selected_MI_0.01"] = (
    candidate_df["MI_Score"] >= 0.01
)

print(
    candidate_df[
        ["DLL", "MI_Score", "Selected_MI_0.01"]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("INVESTIGATION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# DLL FEATURE SELECTION RECONSTRUCTION — CUTOFF COMPARISON
# ============================================================

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

print("=" * 70)
print("SYSTEMATIC DLL FEATURE-SELECTION RECONSTRUCTION")
print("=" * 70)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(DLL_PATH)

feature_cols = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

X_full = df[feature_cols].fillna(0).astype(np.uint8)
y = df["Type"].astype(np.int64)

print("Dataset:", df.shape)
print("Original DLL features:", len(feature_cols))

# ------------------------------------------------------------
# TEST DIFFERENT FREQUENCY CUTOFFS
# ------------------------------------------------------------

cutoffs = [
    200,
    225,
    250,
    275,
    300,
    350,
    400
]

results = []

for cutoff in cutoffs:

    print("\n" + "-" * 70)
    print("Frequency cutoff:", cutoff)
    print("-" * 70)

    frequencies = X_full.sum(axis=0)

    candidates = frequencies[
        frequencies >= cutoff
    ].index.tolist()

    print("Candidate DLLs:", len(candidates))

    if len(candidates) < 27:
        print("Not enough candidates for top-27 selection.")
        continue

    X = X_full[candidates]

    mi_scores = mutual_info_classif(
        X,
        y,
        discrete_features=True,
        random_state=42
    )

    temp = pd.DataFrame({
        "DLL": candidates,
        "Frequency": frequencies[candidates].values,
        "MI_Score": mi_scores
    })

    temp = temp.sort_values(
        "MI_Score",
        ascending=False
    ).reset_index(drop=True)

    top27 = temp.head(27)

    mi_threshold_count = (
        temp["MI_Score"] >= 0.01
    ).sum()

    results.append({
        "Frequency_Cutoff": cutoff,
        "Candidates": len(candidates),
        "MI_GE_0.01": mi_threshold_count,
        "Top27_Min_MI": top27["MI_Score"].min()
    })

    print(
        "MI >= 0.01:",
        mi_threshold_count
    )

    print(
        "27th-ranked MI:",
        round(top27["MI_Score"].iloc[-1], 6)
    )

    print("\nTop 27:")

    print(
        top27[
            ["DLL", "Frequency", "MI_Score"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CUTOFF COMPARISON SUMMARY")
print("=" * 70)

results_df = pd.DataFrame(results)

print(
    results_df.to_string(index=False)
)

results_df.to_csv(
    "/kaggle/working/DLL_Selection_Cutoff_Comparison.csv",
    index=False
)

print("\nSaved:")
print(
    "/kaggle/working/DLL_Selection_Cutoff_Comparison.csv"
)

print("\n" + "=" * 70)
print("RECONSTRUCTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# PE HEADER FEATURE ANALYSIS
# ============================================================

PE_HEADER_PATH = (
    "/kaggle/input/datasets/joebeachcapital/"
    "windows-malwares/PE_Header.csv"
)

OUTPUT_PATH = (
    "/kaggle/working/PE_Header_Feature_Analysis.csv"
)

print("=" * 70)
print("PE HEADER FEATURE ANALYSIS")
print("=" * 70)

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(PE_HEADER_PATH)

print("\nDataset shape:", df.shape)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# ============================================================
# 2. IDENTIFY FEATURES
# ============================================================

identifier_cols = ["SHA256", "Type"]

feature_cols = [
    c for c in df.columns
    if c not in identifier_cols
]

print("\n" + "=" * 70)
print("FEATURE STRUCTURE")
print("=" * 70)

print("Identifier columns:", identifier_cols)
print("PE Header features:", len(feature_cols))

print("\nFeature names:")

for i, col in enumerate(feature_cols, 1):
    print(f"{i:3d}. {col}")

# ============================================================
# 3. DATA TYPES
# ============================================================

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df[feature_cols].dtypes.value_counts())

# ============================================================
# 4. MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df[feature_cols].isna().sum()

print("Total missing values:", int(missing.sum()))
print("Features containing missing values:",
      int((missing > 0).sum()))

if (missing > 0).any():
    print("\nFeatures with missing values:")
    print(
        missing[missing > 0]
        .sort_values(ascending=False)
        .to_string()
    )
else:
    print("PASS: No missing PE Header values.")

# ============================================================
# 5. DUPLICATE SHA256
# ============================================================

print("\n" + "=" * 70)
print("SHA256 DUPLICATES")
print("=" * 70)

sha_duplicates = df["SHA256"].duplicated().sum()

print("Duplicate SHA256 rows:", int(sha_duplicates))
print("Unique SHA256:", df["SHA256"].nunique())

# ============================================================
# 6. CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CONSTANT FEATURES")
print("=" * 70)

nunique = df[feature_cols].nunique(dropna=False)

constant_features = nunique[
    nunique <= 1
].index.tolist()

print("Constant features:", len(constant_features))

if constant_features:
    print("\nConstant features:")
    for col in constant_features:
        print(" ", col)
else:
    print("PASS: No constant PE Header features.")

# ============================================================
# 7. UNIQUE VALUE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CARDINALITY")
print("=" * 70)

cardinality_df = pd.DataFrame({
    "Feature": feature_cols,
    "Unique_Values": [
        nunique[c] for c in feature_cols
    ]
})

print(
    cardinality_df
    .sort_values("Unique_Values")
    .to_string(index=False)
)

# ============================================================
# 8. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)

numeric_df = df[feature_cols].select_dtypes(
    include=[np.number]
)

print("Numeric features:", numeric_df.shape[1])

stats = numeric_df.describe().T

stats["Missing"] = numeric_df.isna().sum()
stats["Unique"] = numeric_df.nunique()

print(
    stats[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "Unique",
            "Missing"
        ]
    ].to_string()
)

# ============================================================
# 9. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

target_distribution = (
    df["Type"]
    .value_counts()
    .sort_index()
)

target_percentage = (
    df["Type"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

target_df = pd.DataFrame({
    "Samples": target_distribution,
    "Percentage": target_percentage.round(2)
})

print(target_df)

# ============================================================
# 10. FEATURE-TARGET MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING PE HEADER MUTUAL INFORMATION")
print("=" * 70)

# Remove constant features before MI
mi_features = [
    c for c in feature_cols
    if c not in constant_features
]

X = df[mi_features].fillna(0)

# Ensure numeric representation
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

y = df["Type"].astype(np.int64)

print("Features evaluated:", len(mi_features))
print("Samples:", len(X))

# PE header fields are discrete/integer-valued structural fields.
# Treat them as discrete for MI estimation.
mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": mi_features,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# 11. TOP FEATURES
# ============================================================

print("\n" + "=" * 70)
print("TOP PE HEADER FEATURES BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(30).to_string(index=False)
)

# ============================================================
# 12. CORRELATION WITH TARGET
# ============================================================

print("\n" + "=" * 70)
print("FEATURE-TARGET CORRELATION")
print("=" * 70)

correlations = []

for col in mi_features:

    try:
        corr = df[col].corr(
            df["Type"],
            method="spearman"
        )
    except Exception:
        corr = np.nan

    correlations.append({
        "Feature": col,
        "Spearman_Correlation": corr
    })

corr_df = pd.DataFrame(correlations)

corr_df["Abs_Correlation"] = (
    corr_df["Spearman_Correlation"]
    .abs()
)

corr_df = corr_df.sort_values(
    "Abs_Correlation",
    ascending=False
)

print(
    corr_df.head(30).to_string(index=False)
)

# ============================================================
# 13. COMBINED FEATURE ANALYSIS
# ============================================================

analysis_df = pd.DataFrame({
    "Feature": mi_features,
    "Unique_Values": [
        nunique[c] for c in mi_features
    ],
    "Missing": [
        missing[c] for c in mi_features
    ],
    "MI_Score": [
        mi_df.set_index("Feature")
        .loc[c, "MI_Score"]
        for c in mi_features
    ]
})

analysis_df = analysis_df.merge(
    corr_df[
        [
            "Feature",
            "Spearman_Correlation",
            "Abs_Correlation"
        ]
    ],
    on="Feature",
    how="left"
)

analysis_df = analysis_df.sort_values(
    "MI_Score",
    ascending=False
).reset_index(drop=True)

# ============================================================
# 14. SAVE
# ============================================================

analysis_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("OUTPUT")
print("=" * 70)

print("Saved:")
print(OUTPUT_PATH)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PE HEADER ANALYSIS SUMMARY")
print("=" * 70)

print("Samples:", len(df))
print("PE Header features:", len(feature_cols))
print("Constant features:", len(constant_features))
print("Missing values:", int(missing.sum()))
print("Duplicate SHA256 rows:", int(sha_duplicates))

print("\nTop 10 MI features:")

print(
    mi_df.head(10).to_string(index=False)
)

print("\n" + "=" * 70)
print("PE HEADER ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# PE SECTION FEATURE ANALYSIS
# ============================================================

PE_SECTION_PATH = (
    "/kaggle/input/datasets/joebeachcapital/"
    "windows-malwares/PE_Section.csv"
)

OUTPUT_PATH = (
    "/kaggle/working/PE_Section_Feature_Analysis.csv"
)

print("=" * 70)
print("PE SECTION FEATURE ANALYSIS")
print("=" * 70)

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(PE_SECTION_PATH)

print("\nDataset shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

# ============================================================
# 2. IDENTIFY FEATURES
# ============================================================

identifier_cols = ["SHA256", "Type"]

feature_cols = [
    c for c in df.columns
    if c not in identifier_cols
]

print("\n" + "=" * 70)
print("FEATURE STRUCTURE")
print("=" * 70)

print("Identifier columns:", identifier_cols)
print("PE Section features:", len(feature_cols))

print("\nFeature names:")

for i, col in enumerate(feature_cols, 1):
    print(f"{i:3d}. {col}")

# ============================================================
# 3. DATA TYPES
# ============================================================

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df[feature_cols].dtypes.value_counts())

# ============================================================
# 4. MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df[feature_cols].isna().sum()

print("Total missing values:", int(missing.sum()))
print(
    "Features containing missing values:",
    int((missing > 0).sum())
)

if (missing > 0).any():
    print("\nFeatures with missing values:")
    print(
        missing[missing > 0]
        .sort_values(ascending=False)
        .to_string()
    )
else:
    print("PASS: No missing PE Section values.")

# ============================================================
# 5. SHA256 DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("SHA256 DUPLICATES")
print("=" * 70)

sha_duplicates = df["SHA256"].duplicated().sum()

print("Duplicate SHA256 rows:", int(sha_duplicates))
print("Unique SHA256:", df["SHA256"].nunique())

# ============================================================
# 6. CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CONSTANT FEATURES")
print("=" * 70)

nunique = df[feature_cols].nunique(dropna=False)

constant_features = nunique[
    nunique <= 1
].index.tolist()

print("Constant features:", len(constant_features))

if constant_features:
    print("\nConstant features:")
    for col in constant_features:
        print(" ", col)
else:
    print("PASS: No constant PE Section features.")

# ============================================================
# 7. FEATURE CARDINALITY
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CARDINALITY")
print("=" * 70)

cardinality_df = pd.DataFrame({
    "Feature": feature_cols,
    "Unique_Values": [
        nunique[c] for c in feature_cols
    ]
})

print(
    cardinality_df
    .sort_values("Unique_Values")
    .to_string(index=False)
)

# ============================================================
# 8. DESCRIPTIVE STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)

numeric_df = df[feature_cols].select_dtypes(
    include=[np.number]
)

print("Numeric features:", numeric_df.shape[1])

stats = numeric_df.describe().T

stats["Missing"] = numeric_df.isna().sum()
stats["Unique"] = numeric_df.nunique()

print(
    stats[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "Unique",
            "Missing"
        ]
    ].to_string()
)

# ============================================================
# 9. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

target_distribution = (
    df["Type"]
    .value_counts()
    .sort_index()
)

target_percentage = (
    df["Type"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

target_df = pd.DataFrame({
    "Samples": target_distribution,
    "Percentage": target_percentage.round(2)
})

print(target_df)

# ============================================================
# 10. MUTUAL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING PE SECTION MUTUAL INFORMATION")
print("=" * 70)

mi_features = [
    c for c in feature_cols
    if c not in constant_features
]

X = df[mi_features].apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

y = df["Type"].astype(np.int64)

print("Features evaluated:", len(mi_features))
print("Samples:", len(X))

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=True,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": mi_features,
    "MI_Score": mi_scores
})

mi_df = (
    mi_df
    .sort_values("MI_Score", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# 11. TOP MI FEATURES
# ============================================================

print("\n" + "=" * 70)
print("TOP PE SECTION FEATURES BY MUTUAL INFORMATION")
print("=" * 70)

print(
    mi_df.head(30).to_string(index=False)
)

# ============================================================
# 12. SPEARMAN CORRELATION
# ============================================================

print("\n" + "=" * 70)
print("FEATURE-TARGET CORRELATION")
print("=" * 70)

correlations = []

for col in mi_features:

    try:
        corr = df[col].corr(
            df["Type"],
            method="spearman"
        )
    except Exception:
        corr = np.nan

    correlations.append({
        "Feature": col,
        "Spearman_Correlation": corr
    })

corr_df = pd.DataFrame(correlations)

corr_df["Abs_Correlation"] = (
    corr_df["Spearman_Correlation"].abs()
)

corr_df = corr_df.sort_values(
    "Abs_Correlation",
    ascending=False
)

print(
    corr_df.head(30).to_string(index=False)
)

# ============================================================
# 13. COMBINED ANALYSIS
# ============================================================

analysis_df = pd.DataFrame({
    "Feature": mi_features,
    "Unique_Values": [
        nunique[c] for c in mi_features
    ],
    "Missing": [
        missing[c] for c in mi_features
    ],
    "MI_Score": [
        mi_df.set_index("Feature")
        .loc[c, "MI_Score"]
        for c in mi_features
    ]
})

analysis_df = analysis_df.merge(
    corr_df[
        [
            "Feature",
            "Spearman_Correlation",
            "Abs_Correlation"
        ]
    ],
    on="Feature",
    how="left"
)

analysis_df = (
    analysis_df
    .sort_values("MI_Score", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# 14. SAVE
# ============================================================

analysis_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n" + "=" * 70)
print("OUTPUT")
print("=" * 70)

print("Saved:")
print(OUTPUT_PATH)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PE SECTION ANALYSIS SUMMARY")
print("=" * 70)

print("Samples:", len(df))
print("PE Section features:", len(feature_cols))
print("Constant features:", len(constant_features))
print("Missing values:", int(missing.sum()))
print("Duplicate SHA256 rows:", int(sha_duplicates))

print("\nTop 10 MI features:")

print(
    mi_df.head(10).to_string(index=False)
)

print("\n" + "=" * 70)
print("PE SECTION ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np

print("=" * 70)
print("ALIGNING ALL FOUR ORIGINAL DATASETS")
print("=" * 70)

# ============================================================
# PATHS
# CHANGE ONLY IF YOUR ORIGINAL PATHS ARE DIFFERENT
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"
PE_HEADER_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"
PE_SECTION_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"

# ============================================================
# LOAD
# ============================================================

api = pd.read_csv(API_PATH)
dll = pd.read_csv(DLL_PATH)
pe_header = pd.read_csv(PE_HEADER_PATH)
pe_section = pd.read_csv(PE_SECTION_PATH)

datasets = {
    "API": api,
    "DLL": dll,
    "PE_Header": pe_header,
    "PE_Section": pe_section
}

# ============================================================
# BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET SIZES")
print("=" * 70)

for name, df in datasets.items():
    print(
        f"{name:12s}: "
        f"rows={len(df):6d} | "
        f"SHA256={df['SHA256'].nunique():6d}"
    )

# ============================================================
# SHA256 SETS
# ============================================================

sha_sets = {
    name: set(df["SHA256"].astype(str))
    for name, df in datasets.items()
}

print("\n" + "=" * 70)
print("PAIRWISE SHA256 OVERLAP")
print("=" * 70)

names = list(sha_sets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        a = names[i]
        b = names[j]

        intersection = (
            sha_sets[a] &
            sha_sets[b]
        )

        print(
            f"{a:12s} ∩ {b:12s}: "
            f"{len(intersection):6d}"
        )

# ============================================================
# COMMON TO ALL FOUR
# ============================================================

common_sha = set.intersection(
    *sha_sets.values()
)

print("\n" + "=" * 70)
print("COMMON SHA256 ACROSS ALL FOUR DATASETS")
print("=" * 70)

print(
    "Common samples:",
    len(common_sha)
)

# ============================================================
# UNION
# ============================================================

union_sha = set.union(
    *sha_sets.values()
)

print("Union samples:", len(union_sha))

# ============================================================
# MISSING FROM EACH DATASET
# ============================================================

print("\n" + "=" * 70)
print("SAMPLES MISSING FROM EACH DATASET")
print("=" * 70)

for name, sha in sha_sets.items():

    missing_from_dataset = (
        union_sha - sha
    )

    print(
        f"{name:12s}: "
        f"{len(missing_from_dataset):6d}"
    )

# ============================================================
# LABEL CONSISTENCY ON COMMON SAMPLES
# ============================================================

print("\n" + "=" * 70)
print("CHECKING TYPE LABEL CONSISTENCY")
print("=" * 70)

label_tables = []

for name, df in datasets.items():

    temp = df[
        df["SHA256"].astype(str).isin(common_sha)
    ][
        ["SHA256", "Type"]
    ].copy()

    temp["SHA256"] = temp["SHA256"].astype(str)

    temp = temp.rename(
        columns={"Type": f"Type_{name}"}
    )

    label_tables.append(temp)

labels = label_tables[0]

for temp in label_tables[1:]:

    labels = labels.merge(
        temp,
        on="SHA256",
        how="inner"
    )

type_cols = [
    c for c in labels.columns
    if c.startswith("Type_")
]

print("Common samples checked:", len(labels))

# Number of distinct labels per SHA256
labels["Unique_Types"] = (
    labels[type_cols]
    .nunique(axis=1)
)

inconsistent = labels[
    labels["Unique_Types"] > 1
]

print(
    "Label-inconsistent samples:",
    len(inconsistent)
)

if len(inconsistent) > 0:
    print("\nFirst inconsistent samples:")
    print(
        inconsistent
        .head(20)
        .to_string(index=False)
    )
else:
    print(
        "PASS: Type labels are consistent "
        "across all four datasets."
    )

# ============================================================
# TARGET DISTRIBUTION ON COMMON SET
# ============================================================

print("\n" + "=" * 70)
print("COMMON DATASET TARGET DISTRIBUTION")
print("=" * 70)

if len(labels) > 0:

    common_type = labels[
        "Type_API"
    ]

    target_counts = (
        common_type
        .value_counts()
        .sort_index()
    )

    target_pct = (
        common_type
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )

    target_df = pd.DataFrame({
        "Samples": target_counts,
        "Percentage": target_pct.round(2)
    })

    print(target_df)

# ============================================================
# SAVE COMMON SHA256 LIST
# ============================================================

common_sha_df = pd.DataFrame({
    "SHA256": sorted(common_sha)
})

COMMON_SHA_PATH = (
    "/kaggle/working/"
    "Common_SHA256_All_Four_Datasets.csv"
)

common_sha_df.to_csv(
    COMMON_SHA_PATH,
    index=False
)

print("\n" + "=" * 70)
print("OUTPUT")
print("=" * 70)

print(
    "Saved:",
    COMMON_SHA_PATH
)

print("\n" + "=" * 70)
print("ALIGNMENT ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import os
import gc

print("=" * 70)
print("MEMORY-EFFICIENT DATASET ALIGNMENT")
print("=" * 70)

# ============================================================
# USE THE SAME PATHS AS YOUR PREVIOUS CELLS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"
PE_HEADER_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"
PE_SECTION_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"

paths = {
    "API": API_PATH,
    "DLL": DLL_PATH,
    "PE_Header": PE_HEADER_PATH,
    "PE_Section": PE_SECTION_PATH
}

# ============================================================
# READ ONLY SHA256 + TYPE
# ============================================================

metadata = {}

for name, path in paths.items():

    print(f"\nLoading metadata: {name}")

    df = pd.read_csv(
        path,
        usecols=["SHA256", "Type"],
        dtype={
            "SHA256": "string",
            "Type": "int8"
        }
    )

    # Remove duplicate SHA256 rows if any
    df = df.drop_duplicates(
        subset="SHA256",
        keep="first"
    )

    metadata[name] = df

    print(
        f"{name:12s} | "
        f"Rows: {len(df):6d} | "
        f"Unique SHA256: {df['SHA256'].nunique():6d}"
    )

    gc.collect()

# ============================================================
# SHA256 SETS
# ============================================================

sha_sets = {
    name: set(df["SHA256"].dropna())
    for name, df in metadata.items()
}

# ============================================================
# COMMON SHA256
# ============================================================

common_sha = set.intersection(
    *sha_sets.values()
)

union_sha = set.union(
    *sha_sets.values()
)

print("\n" + "=" * 70)
print("SHA256 ALIGNMENT")
print("=" * 70)

print(
    f"Union of all datasets : {len(union_sha):,}"
)

print(
    f"Common to all 4       : {len(common_sha):,}"
)

# ============================================================
# MISSING FROM EACH
# ============================================================

print("\n" + "=" * 70)
print("MISSING SAMPLES")
print("=" * 70)

for name, sha in sha_sets.items():

    missing = union_sha - sha

    print(
        f"{name:12s}: "
        f"{len(missing):,} missing"
    )

# ============================================================
# PAIRWISE OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("PAIRWISE OVERLAP")
print("=" * 70)

names = list(sha_sets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        a = names[i]
        b = names[j]

        overlap = len(
            sha_sets[a] & sha_sets[b]
        )

        print(
            f"{a:12s} ∩ {b:12s}: "
            f"{overlap:,}"
        )

# ============================================================
# LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 70)
print("CHECKING TYPE CONSISTENCY")
print("=" * 70)

common_labels = None

for name, df in metadata.items():

    temp = (
        df[df["SHA256"].isin(common_sha)]
        [["SHA256", "Type"]]
        .copy()
    )

    temp = temp.rename(
        columns={"Type": f"Type_{name}"}
    )

    if common_labels is None:
        common_labels = temp
    else:
        common_labels = common_labels.merge(
            temp,
            on="SHA256",
            how="inner"
        )

type_cols = [
    c for c in common_labels.columns
    if c.startswith("Type_")
]

common_labels["Unique_Types"] = (
    common_labels[type_cols]
    .nunique(axis=1)
)

inconsistent = common_labels[
    common_labels["Unique_Types"] > 1
]

print(
    f"Common samples checked : {len(common_labels):,}"
)

print(
    f"Label inconsistencies  : {len(inconsistent):,}"
)

if len(inconsistent) == 0:
    print(
        "PASS: All four datasets have "
        "consistent Type labels."
    )
else:
    print("\nWARNING: Label mismatch detected.")
    print(inconsistent.head(20))

# ============================================================
# TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("COMMON DATASET TARGET DISTRIBUTION")
print("=" * 70)

target = common_labels["Type_API"]

distribution = pd.DataFrame({
    "Samples": target.value_counts().sort_index(),
    "Percentage": (
        target.value_counts(
            normalize=True
        ).sort_index() * 100
    ).round(2)
})

print(distribution)

# ============================================================
# SAVE COMMON SHA256
# ============================================================

COMMON_SHA_PATH = (
    "/kaggle/working/"
    "Common_SHA256_All_Four_Datasets.csv"
)

pd.DataFrame({
    "SHA256": sorted(common_sha)
}).to_csv(
    COMMON_SHA_PATH,
    index=False
)

print("\n" + "=" * 70)
print("OUTPUT")
print("=" * 70)

print("Saved:")
print(COMMON_SHA_PATH)

print("\n" + "=" * 70)
print("ALIGNMENT COMPLETE")
print("=" * 70)

# Free memory
del common_labels
gc.collect()

In [ ]:
import pandas as pd
import numpy as np
import gc

from sklearn.feature_selection import mutual_info_classif

print("=" * 70)
print("RECONSTRUCTING PE FEATURE SELECTION ON COMMON DATASET")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

PE_HEADER_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"
PE_SECTION_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"

COMMON_SHA_PATH = (
    "/kaggle/working/"
    "Common_SHA256_All_Four_Datasets.csv"
)

# ============================================================
# LOAD COMMON SHA256
# ============================================================

common_sha = pd.read_csv(
    COMMON_SHA_PATH,
    dtype={"SHA256": "string"}
)

common_set = set(common_sha["SHA256"])

print(
    f"Common samples available: {len(common_set):,}"
)

# ============================================================
# FUNCTION
# ============================================================

def analyze_feature_family(path, family_name):

    print("\n" + "=" * 70)
    print(f"{family_name.upper()} FEATURE SELECTION")
    print("=" * 70)

    # --------------------------------------------------------
    # Read dataset
    # --------------------------------------------------------

    df = pd.read_csv(
        path
    )

    print(
        f"Original dataset shape: {df.shape}"
    )

    # --------------------------------------------------------
    # Align to common SHA256
    # --------------------------------------------------------

    df["SHA256"] = df["SHA256"].astype("string")

    df = df[
        df["SHA256"].isin(common_set)
    ].copy()

    print(
        f"Aligned dataset shape: {df.shape}"
    )

    # --------------------------------------------------------
    # Feature columns
    # --------------------------------------------------------

    feature_cols = [
        c for c in df.columns
        if c not in ["SHA256", "Type"]
    ]

    print(
        f"Original features: {len(feature_cols)}"
    )

    # --------------------------------------------------------
    # Remove constant features
    # --------------------------------------------------------

    nunique = df[feature_cols].nunique()

    constant_features = (
        nunique[nunique <= 1]
        .index
        .tolist()
    )

    usable_features = [
        c for c in feature_cols
        if c not in constant_features
    ]

    print(
        f"Constant features: {len(constant_features)}"
    )

    if constant_features:
        print("\nRemoved constants:")
        for c in constant_features:
            print(" ", c)

    print(
        f"Usable features: {len(usable_features)}"
    )

    # --------------------------------------------------------
    # Matrix
    # --------------------------------------------------------

    X = df[usable_features].copy()

    y = df["Type"].astype("int8").values

    # Make numeric
    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Missing safety
    if X.isna().sum().sum() > 0:

        print(
            "WARNING: Missing values detected. "
            "Replacing with 0."
        )

        X = X.fillna(0)

    X_values = X.values

    print(
        f"\nCalculating MI:"
        f"\nFeatures: {X_values.shape[1]:,}"
        f"\nSamples: {X_values.shape[0]:,}"
    )

    # --------------------------------------------------------
    # Mutual Information
    # --------------------------------------------------------

    mi = mutual_info_classif(
        X_values,
        y,
        discrete_features=False,
        random_state=42
    )

    result = pd.DataFrame({
        "Feature": usable_features,
        "MI_Score": mi
    })

    result = result.sort_values(
        "MI_Score",
        ascending=False
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("MI STATISTICS")
    print("-" * 70)

    print(
        result["MI_Score"].describe()
    )

    # --------------------------------------------------------
    # Candidate thresholds
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("MI THRESHOLD ANALYSIS")
    print("-" * 70)

    thresholds = [
        0.001,
        0.005,
        0.010,
        0.015,
        0.020,
        0.025,
        0.030,
        0.040,
        0.050,
        0.075,
        0.100
    ]

    threshold_rows = []

    for threshold in thresholds:

        count = (
            result["MI_Score"] >= threshold
        ).sum()

        threshold_rows.append({
            "MI_Threshold": threshold,
            "Features_Retained": count,
            "Percentage_Retained":
                round(
                    count /
                    len(result) *
                    100,
                    2
                )
        })

        print(
            f"MI >= {threshold:<6}: "
            f"{count:4d} features"
        )

    threshold_df = pd.DataFrame(
        threshold_rows
    )

    # --------------------------------------------------------
    # Top ranks
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("TOP 30 FEATURES")
    print("-" * 70)

    print(
        result.head(30)
        .to_string(index=False)
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = (
        f"/kaggle/working/"
        f"{family_name}_MI_Common_29489.csv"
    )

    result.to_csv(
        output_path,
        index=False
    )

    threshold_path = (
        f"/kaggle/working/"
        f"{family_name}_MI_Thresholds_Common_29489.csv"
    )

    threshold_df.to_csv(
        threshold_path,
        index=False
    )

    print("\nSaved:")
    print(output_path)
    print(threshold_path)

    # --------------------------------------------------------
    # Free memory
    # --------------------------------------------------------

    del df
    del X
    del X_values
    gc.collect()

    return result, threshold_df


# ============================================================
# PE HEADER
# ============================================================

header_mi, header_thresholds = analyze_feature_family(
    PE_HEADER_PATH,
    "PE_Header"
)

# ============================================================
# PE SECTION
# ============================================================

section_mi, section_thresholds = analyze_feature_family(
    PE_SECTION_PATH,
    "PE_Section"
)

print("\n" + "=" * 70)
print("COMMON-DATASET PE FEATURE SELECTION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
import gc
import os

print("=" * 70)
print("BUILDING FINAL ALIGNED MASTER DATASET")
print("=" * 70)

# ============================================================
# PATHS
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"
DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"
PE_HEADER_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"
PE_SECTION_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"

COMMON_SHA_PATH = (
    "/kaggle/working/"
    "Common_SHA256_All_Four_Datasets.csv"
)

# Previously reconstructed selection files
API_SELECTION_PATH = (
    "/kaggle/working/"
    "API_Top200_MI_Reconstructed.csv"
)

DLL_SELECTION_PATH = (
    "/kaggle/working/"
    "DLL_Top27_MI_Reconstructed.csv"
)

PE_HEADER_SELECTION_PATH = (
    "/kaggle/working/"
    "PE_Header_MI_Common_29489.csv"
)

PE_SECTION_SELECTION_PATH = (
    "/kaggle/working/"
    "PE_Section_MI_Common_29489.csv"
)

# ============================================================
# LOAD COMMON SHA256
# ============================================================

common_df = pd.read_csv(
    COMMON_SHA_PATH,
    dtype={"SHA256": "string"}
)

common_sha = set(
    common_df["SHA256"]
)

print(
    f"\nCommon SHA256 samples: {len(common_sha):,}"
)

# ============================================================
# LOAD FEATURE SELECTION LISTS
# ============================================================

print("\n" + "=" * 70)
print("LOADING FEATURE SELECTIONS")
print("=" * 70)

# ---------------- API ----------------

api_selected_df = pd.read_csv(
    API_SELECTION_PATH
)

api_features = (
    api_selected_df["API"]
    .head(200)
    .tolist()
)

print(
    f"API selected features: {len(api_features)}"
)

# ---------------- DLL ----------------

dll_selected_df = pd.read_csv(
    DLL_SELECTION_PATH
)

dll_features = (
    dll_selected_df["DLL"]
    .head(27)
    .tolist()
)

# The reconstructed MI process retained 26,
# but our intended 27th DLL was shell32.dll.
if "shell32.dll" not in dll_features:
    dll_features.append("shell32.dll")

# Keep exactly 27
dll_features = dll_features[:27]

print(
    f"DLL selected features: {len(dll_features)}"
)

# ---------------- PE HEADER ----------------

header_mi = pd.read_csv(
    PE_HEADER_SELECTION_PATH
)

header_features = (
    header_mi[
        header_mi["MI_Score"] >= 0.01
    ]["Feature"]
    .tolist()
)

print(
    f"PE Header selected features: "
    f"{len(header_features)}"
)

# ---------------- PE SECTION ----------------

section_mi = pd.read_csv(
    PE_SECTION_SELECTION_PATH
)

section_features = (
    section_mi[
        section_mi["MI_Score"] >= 0.01
    ]["Feature"]
    .tolist()
)

print(
    f"PE Section selected features: "
    f"{len(section_features)}"
)

# ============================================================
# VERIFY EXPECTED DIMENSIONS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE COUNT VERIFICATION")
print("=" * 70)

print(f"API        : {len(api_features)}")
print(f"DLL        : {len(dll_features)}")
print(f"PE Header  : {len(header_features)}")
print(f"PE Section : {len(section_features)}")

total_features = (
    len(api_features)
    + len(dll_features)
    + len(header_features)
    + len(section_features)
)

print("-" * 70)
print(f"TOTAL FEATURES: {total_features}")

assert len(api_features) == 200
assert len(dll_features) == 27
assert len(header_features) == 46
assert len(section_features) == 45

assert total_features == 318

print("PASS: Expected 318 features confirmed.")

# ============================================================
# FUNCTION TO LOAD SELECTED FEATURES
# ============================================================

def load_selected_dataset(
    path,
    selected_features,
    dataset_name
):

    print("\n" + "=" * 70)
    print(f"LOADING {dataset_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Only load selected columns
    # --------------------------------------------------------

    usecols = [
        "SHA256",
        "Type"
    ] + selected_features

    df = pd.read_csv(
        path,
        usecols=usecols
    )

    df["SHA256"] = df["SHA256"].astype("string")

    print(
        f"Loaded shape: {df.shape}"
    )

    # --------------------------------------------------------
    # Keep common samples only
    # --------------------------------------------------------

    df = df[
        df["SHA256"].isin(common_sha)
    ].copy()

    print(
        f"After SHA alignment: {df.shape}"
    )

    # --------------------------------------------------------
    # Check duplicate SHA256
    # --------------------------------------------------------

    duplicates = df["SHA256"].duplicated().sum()

    print(
        f"Duplicate SHA256: {duplicates}"
    )

    assert duplicates == 0

    # --------------------------------------------------------
    # Verify features exist
    # --------------------------------------------------------

    missing_features = [
        f for f in selected_features
        if f not in df.columns
    ]

    if missing_features:

        print(
            "\nMISSING FEATURES:"
        )

        for f in missing_features:
            print(" ", f)

        raise ValueError(
            f"{dataset_name}: selected features missing."
        )

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return df


# ============================================================
# LOAD API
# ============================================================

api_final = load_selected_dataset(
    API_PATH,
    api_features,
    "API"
)

gc.collect()

# ============================================================
# LOAD DLL
# ============================================================

dll_final = load_selected_dataset(
    DLL_PATH,
    dll_features,
    "DLL"
)

gc.collect()

# ============================================================
# LOAD PE HEADER
# ============================================================

header_final = load_selected_dataset(
    PE_HEADER_PATH,
    header_features,
    "PE HEADER"
)

gc.collect()

# ============================================================
# LOAD PE SECTION
# ============================================================

section_final = load_selected_dataset(
    PE_SECTION_PATH,
    section_features,
    "PE SECTION"
)

gc.collect()

# ============================================================
# VERIFY ROW COUNTS
# ============================================================

print("\n" + "=" * 70)
print("ROW COUNT VERIFICATION")
print("=" * 70)

for name, df in [
    ("API", api_final),
    ("DLL", dll_final),
    ("PE Header", header_final),
    ("PE Section", section_final)
]:

    print(
        f"{name:12s}: {len(df):,}"
    )

    assert len(df) == 29489

print(
    "\nPASS: All feature families contain "
    "exactly 29,489 samples."
)

# ============================================================
# CHECK LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 70)
print("CHECKING LABEL CONSISTENCY")
print("=" * 70)

label_check = (
    api_final[["SHA256", "Type"]]
    .rename(columns={"Type": "Type_API"})
    .merge(
        dll_final[["SHA256", "Type"]]
        .rename(columns={"Type": "Type_DLL"}),
        on="SHA256",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        header_final[["SHA256", "Type"]]
        .rename(columns={"Type": "Type_Header"}),
        on="SHA256",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        section_final[["SHA256", "Type"]]
        .rename(columns={"Type": "Type_Section"}),
        on="SHA256",
        how="inner",
        validate="one_to_one"
    )
)

label_cols = [
    "Type_API",
    "Type_DLL",
    "Type_Header",
    "Type_Section"
]

label_check["Unique_Types"] = (
    label_check[label_cols]
    .nunique(axis=1)
)

label_mismatches = (
    label_check["Unique_Types"] > 1
).sum()

print(
    f"Samples checked: {len(label_check):,}"
)

print(
    f"Label mismatches: {label_mismatches}"
)

assert len(label_check) == 29489
assert label_mismatches == 0

print(
    "PASS: Labels are completely consistent."
)

# ============================================================
# CREATE MASTER DATASET
# ============================================================

print("\n" + "=" * 70)
print("MERGING FEATURE FAMILIES")
print("=" * 70)

# Start with API
master = api_final.copy()

# Remove Type temporarily from other datasets
master = master.merge(
    dll_final.drop(columns=["Type"]),
    on="SHA256",
    how="inner",
    validate="one_to_one"
)

master = master.merge(
    header_final.drop(columns=["Type"]),
    on="SHA256",
    how="inner",
    validate="one_to_one"
)

master = master.merge(
    section_final.drop(columns=["Type"]),
    on="SHA256",
    how="inner",
    validate="one_to_one"
)

print(
    f"Master shape: {master.shape}"
)

# ============================================================
# FINAL STRUCTURE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET VERIFICATION")
print("=" * 70)

expected_columns = (
    2 + total_features
)

print(
    f"Expected columns: {expected_columns}"
)

print(
    f"Actual columns:   {master.shape[1]}"
)

print(
    f"Expected rows:    29,489"
)

print(
    f"Actual rows:      {master.shape[0]:,}"
)

assert master.shape[0] == 29489
assert master.shape[1] == 320

# ============================================================
# DUPLICATE CHECK
# ============================================================

duplicate_sha = (
    master["SHA256"]
    .duplicated()
    .sum()
)

print(
    f"\nDuplicate SHA256: {duplicate_sha}"
)

assert duplicate_sha == 0

# ============================================================
# MISSING VALUE CHECK
# ============================================================

missing_total = (
    master.isna()
    .sum()
    .sum()
)

print(
    f"Total missing values: {missing_total:,}"
)

assert missing_total == 0

# ============================================================
# TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("FINAL TARGET DISTRIBUTION")
print("=" * 70)

target_distribution = pd.DataFrame({
    "Samples": master["Type"]
        .value_counts()
        .sort_index(),

    "Percentage": (
        master["Type"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
})

print(
    target_distribution
)

# ============================================================
# SAVE FINAL DATASET
# ============================================================

FINAL_PATH = (
    "/kaggle/working/"
    "Final_Malware_Master_Dataset_29489.csv"
)

master.to_csv(
    FINAL_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FINAL DATASET SAVED")
print("=" * 70)

print(FINAL_PATH)

print("\n" + "=" * 70)
print("MASTER DATASET COMPLETE")
print("=" * 70)

print(
    f"Samples : {master.shape[0]:,}"
)

print(
    f"Features: {total_features:,}"
)

print(
    f"Columns : {master.shape[1]:,}"
)

print(
    "\nSHA256 + 318 selected features + Type"
)

print(
    "\nPASS: FINAL MASTER DATASET IS READY."
)

In [1]:
import os
import pandas as pd
import gc

print("=" * 70)
print("CHECKING SAVED RECONSTRUCTION FILES")
print("=" * 70)

files = [
    "/kaggle/working/Common_SHA256_All_Four_Datasets.csv",
    "/kaggle/working/API_Top200_MI_Reconstructed.csv",
    "/kaggle/working/DLL_Top27_MI_Reconstructed.csv",
    "/kaggle/working/PE_Header_MI_Common_29489.csv",
    "/kaggle/working/PE_Section_MI_Common_29489.csv",
]

for path in files:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / (1024 * 1024) if exists else 0

    print(
        f"{'OK' if exists else 'MISSING':8s} "
        f"{os.path.basename(path):55s} "
        f"{size:8.2f} MB"
    )

print("=" * 70)

CHECKING SAVED RECONSTRUCTION FILES
OK       Common_SHA256_All_Four_Datasets.csv                         1.83 MB
OK       API_Top200_MI_Reconstructed.csv                             0.01 MB
OK       DLL_Top27_MI_Reconstructed.csv                              0.00 MB
OK       PE_Header_MI_Common_29489.csv                               0.00 MB
OK       PE_Section_MI_Common_29489.csv                              0.00 MB


In [3]:
import os
import gc
import pandas as pd

print("=" * 70)
print("LOADING FINAL FEATURE LISTS")
print("=" * 70)


# ============================================================
# PATHS
# ============================================================

COMMON_SHA_PATH = "/kaggle/working/Common_SHA256_All_Four_Datasets.csv"

API_SELECTION_PATH = "/kaggle/working/API_Top200_MI_Reconstructed.csv"
DLL_SELECTION_PATH = "/kaggle/working/DLL_Top27_MI_Reconstructed.csv"

HEADER_SELECTION_PATH = "/kaggle/working/PE_Header_MI_Common_29489.csv"
SECTION_SELECTION_PATH = "/kaggle/working/PE_Section_MI_Common_29489.csv"


# ============================================================
# CHECK COMMON DATASET COLUMNS FIRST
# ============================================================

print("\nChecking common dataset columns...")

common_columns = pd.read_csv(
    COMMON_SHA_PATH,
    nrows=0
).columns.tolist()

print("Columns found:")
print(common_columns)


# ============================================================
# IDENTIFY TYPE COLUMN
# ============================================================

if "Type" in common_columns:
    TYPE_COLUMN = "Type"

elif "Type_API" in common_columns:
    TYPE_COLUMN = "Type_API"

elif "Type_DLL" in common_columns:
    TYPE_COLUMN = "Type_DLL"

elif "Type_PE_Header" in common_columns:
    TYPE_COLUMN = "Type_PE_Header"

elif "Type_PE_Section" in common_columns:
    TYPE_COLUMN = "Type_PE_Section"

else:
    raise ValueError(
        "Could not find a Type column in the common dataset."
    )

print(f"\nUsing target column: {TYPE_COLUMN}")


# ============================================================
# LOAD COMMON SHA + TARGET
# ============================================================

common = pd.read_csv(
    COMMON_SHA_PATH,
    usecols=["SHA256", TYPE_COLUMN],
    dtype={
        "SHA256": "string",
        TYPE_COLUMN: "uint8"
    }
)

# Rename to our standard target name
if TYPE_COLUMN != "Type":
    common = common.rename(
        columns={TYPE_COLUMN: "Type"}
    )

common = common.drop_duplicates(
    subset=["SHA256"]
).reset_index(drop=True)

print("\nCommon dataset:")
print(f"Rows: {len(common):,}")
print(f"Columns: {common.columns.tolist()}")

assert len(common) == 29489
assert common["SHA256"].is_unique

print("PASS: 29,489 unique common SHA256 samples loaded.")


# ============================================================
# API FEATURES
# ============================================================

api_sel = pd.read_csv(
    API_SELECTION_PATH
)

api_features = (
    api_sel["API"]
    .astype(str)
    .head(200)
    .tolist()
)

print(f"\nAPI features: {len(api_features)}")


# ============================================================
# DLL FEATURES
# ============================================================

dll_sel = pd.read_csv(
    DLL_SELECTION_PATH
)

dll_features = (
    dll_sel["DLL"]
    .astype(str)
    .tolist()
)

# ------------------------------------------------------------
# IMPORTANT:
# The MI filtering produced 26 DLLs at MI >= 0.01.
# The reconstruction investigation identified shell32.dll
# as the 27th-ranked DLL.
# ------------------------------------------------------------

if "shell32.dll" not in dll_features:
    dll_features.append("shell32.dll")

dll_features = dll_features[:27]

print(f"DLL features: {len(dll_features)}")

if "shell32.dll" in dll_features:
    print("shell32.dll included as reconstructed 27th DLL feature.")


# ============================================================
# PE HEADER FEATURES
# ============================================================

header_sel = pd.read_csv(
    HEADER_SELECTION_PATH
)

header_features = (
    header_sel.loc[
        header_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Header features: {len(header_features)}")


# ============================================================
# PE SECTION FEATURES
# ============================================================

section_sel = pd.read_csv(
    SECTION_SELECTION_PATH
)

section_features = (
    section_sel.loc[
        section_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Section features: {len(section_features)}")


# ============================================================
# FINAL FEATURE COUNT
# ============================================================

total = (
    len(api_features)
    + len(dll_features)
    + len(header_features)
    + len(section_features)
)

print("\n" + "=" * 70)
print("FINAL FEATURE COUNT")
print("=" * 70)

print(f"API        : {len(api_features)}")
print(f"DLL        : {len(dll_features)}")
print(f"PE Header  : {len(header_features)}")
print(f"PE Section : {len(section_features)}")
print("-" * 70)
print(f"TOTAL      : {total}")


# ============================================================
# VALIDATION
# ============================================================

assert len(api_features) == 200, \
    f"Expected 200 API features, got {len(api_features)}"

assert len(dll_features) == 27, \
    f"Expected 27 DLL features, got {len(dll_features)}"

assert len(header_features) == 46, \
    f"Expected 46 PE Header features, got {len(header_features)}"

assert len(section_features) == 45, \
    f"Expected 45 PE Section features, got {len(section_features)}"

assert total == 318, \
    f"Expected 318 total features, got {total}"

print("\nPASS: 318 final features confirmed.")

gc.collect()

LOADING FINAL FEATURE LISTS

Checking common dataset columns...
Columns found:
['SHA256']


ValueError: Could not find a Type column in the common dataset.

In [4]:
import os
import gc
import pandas as pd

print("=" * 70)
print("LOADING FINAL FEATURE LISTS")
print("=" * 70)


# ============================================================
# PATHS
# ============================================================

COMMON_SHA_PATH = "/kaggle/working/Common_SHA256_All_Four_Datasets.csv"

API_SELECTION_PATH = "/kaggle/working/API_Top200_MI_Reconstructed.csv"
DLL_SELECTION_PATH = "/kaggle/working/DLL_Top27_MI_Reconstructed.csv"

HEADER_SELECTION_PATH = "/kaggle/working/PE_Header_MI_Common_29489.csv"
SECTION_SELECTION_PATH = "/kaggle/working/PE_Section_MI_Common_29489.csv"


# ============================================================
# LOAD COMMON SHA256
# ============================================================

print("\nLoading common SHA256 list...")

common = pd.read_csv(
    COMMON_SHA_PATH,
    usecols=["SHA256"],
    dtype={"SHA256": "string"}
)

common = (
    common
    .drop_duplicates("SHA256")
    .reset_index(drop=True)
)

print(f"Common SHA256 samples: {len(common):,}")

assert len(common) == 29489
assert common["SHA256"].is_unique

print("PASS: 29,489 unique common SHA256 samples.")


# ============================================================
# API FEATURES
# ============================================================

api_sel = pd.read_csv(API_SELECTION_PATH)

api_features = (
    api_sel["API"]
    .astype(str)
    .head(200)
    .tolist()
)

print(f"\nAPI features: {len(api_features)}")


# ============================================================
# DLL FEATURES
# ============================================================

dll_sel = pd.read_csv(DLL_SELECTION_PATH)

dll_features = (
    dll_sel["DLL"]
    .astype(str)
    .tolist()
)

# The reconstruction identified shell32.dll
# as the missing 27th feature.
if "shell32.dll" not in dll_features:
    dll_features.append("shell32.dll")

dll_features = dll_features[:27]

print(f"DLL features: {len(dll_features)}")

assert "shell32.dll" in dll_features


# ============================================================
# PE HEADER FEATURES
# ============================================================

header_sel = pd.read_csv(
    HEADER_SELECTION_PATH
)

header_features = (
    header_sel.loc[
        header_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Header features: {len(header_features)}")


# ============================================================
# PE SECTION FEATURES
# ============================================================

section_sel = pd.read_csv(
    SECTION_SELECTION_PATH
)

section_features = (
    section_sel.loc[
        section_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Section features: {len(section_features)}")


# ============================================================
# FINAL FEATURE COUNT
# ============================================================

total = (
    len(api_features)
    + len(dll_features)
    + len(header_features)
    + len(section_features)
)

print("\n" + "=" * 70)
print("FINAL FEATURE COUNT")
print("=" * 70)

print(f"API        : {len(api_features)}")
print(f"DLL        : {len(dll_features)}")
print(f"PE Header  : {len(header_features)}")
print(f"PE Section : {len(section_features)}")
print("-" * 70)
print(f"TOTAL      : {total}")


# ============================================================
# VALIDATION
# ============================================================

assert len(api_features) == 200
assert len(dll_features) == 27
assert len(header_features) == 46
assert len(section_features) == 45
assert total == 318

print("\nPASS: 318 final features confirmed.")

gc.collect()

LOADING FINAL FEATURE LISTS

Loading common SHA256 list...
Common SHA256 samples: 29,489
PASS: 29,489 unique common SHA256 samples.

API features: 200
DLL features: 27
PE Header features: 46
PE Section features: 45

FINAL FEATURE COUNT
API        : 200
DLL        : 27
PE Header  : 46
PE Section : 45
----------------------------------------------------------------------
TOTAL      : 318

PASS: 318 final features confirmed.


0

In [5]:
print("=" * 70)
print("BUILDING API FEATURE MATRIX — MEMORY SAFE")
print("=" * 70)

# ============================================================
# API SOURCE
# ============================================================

API_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/API_Functions.csv"

# Convert common SHA256 to a Python set for fast lookup
common_sha = set(common["SHA256"].astype(str))

print(f"Common SHA256 lookup set: {len(common_sha):,}")

# Only load what we actually need
api_usecols = ["SHA256", "Type"] + api_features

print(f"Columns requested: {len(api_usecols)}")
print("  SHA256 : 1")
print("  Type   : 1")
print("  APIs   : 200")

# ============================================================
# CHUNKED READING
# ============================================================

api_parts = []

CHUNK_SIZE = 2000

total_rows_read = 0
total_rows_kept = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        API_PATH,
        usecols=api_usecols,
        dtype={
            "SHA256": "string",
            "Type": "uint8",
            **{feature: "uint8" for feature in api_features}
        },
        chunksize=CHUNK_SIZE,
        low_memory=True
    ),
    start=1
):

    total_rows_read += len(chunk)

    # Keep only samples present in the common dataset
    mask = chunk["SHA256"].isin(common_sha)

    if mask.any():
        selected = chunk.loc[mask].copy()

        api_parts.append(selected)

        total_rows_kept += len(selected)

    del chunk

    if chunk_no % 5 == 0:
        print(
            f"Chunks: {chunk_no:5d} | "
            f"Rows read: {total_rows_read:7,d} | "
            f"Common rows: {total_rows_kept:7,d}"
        )

    gc.collect()


# ============================================================
# COMBINE CHUNKS
# ============================================================

api_final = pd.concat(
    api_parts,
    ignore_index=True
)

del api_parts
gc.collect()


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("API MATRIX COMPLETE")
print("=" * 70)

print(f"Rows    : {api_final.shape[0]:,}")
print(f"Columns : {api_final.shape[1]:,}")

print(f"Expected rows    : 29,489")
print(f"Expected columns : 202")

assert api_final.shape[0] == 29489, (
    f"Expected 29,489 API rows, "
    f"got {api_final.shape[0]:,}"
)

assert api_final.shape[1] == 202, (
    f"Expected 202 API columns, "
    f"got {api_final.shape[1]}"
)

assert api_final["SHA256"].is_unique, (
    "API SHA256 values are not unique!"
)

assert api_final["Type"].notna().all(), (
    "API Type contains missing values!"
)

print("\nPASS: API matrix contains exactly")
print("      29,489 common samples")
print("      200 selected API features")
print("      SHA256 + Type")

BUILDING API FEATURE MATRIX — MEMORY SAFE
Common SHA256 lookup set: 29,489
Columns requested: 202
  SHA256 : 1
  Type   : 1
  APIs   : 200
Chunks:     5 | Rows read:  10,000 | Common rows:  10,000
Chunks:    10 | Rows read:  20,000 | Common rows:  19,998
Chunks:    15 | Rows read:  29,505 | Common rows:  29,494

API MATRIX COMPLETE
Rows    : 29,494
Columns : 202
Expected rows    : 29,489
Expected columns : 202


AssertionError: Expected 29,489 API rows, got 29,494

In [6]:
print("=" * 70)
print("DIAGNOSING 5 EXTRA API SAMPLES")
print("=" * 70)

# SHA256 sets
common_set = set(common["SHA256"].astype(str))
api_set = set(api_final["SHA256"].astype(str))

print(f"Common SHA list : {len(common_set):,}")
print(f"API SHA set     : {len(api_set):,}")
print(f"API ∩ Common    : {len(api_set & common_set):,}")

# These are API samples that are in the common file
# but were not expected according to the previous 29,489 count.
print("\nAPI rows appearing in common SHA list:")
print(len(api_set & common_set))

# Check duplicate SHA rows
duplicate_count = api_final["SHA256"].duplicated().sum()

print(f"\nDuplicate API SHA rows: {duplicate_count}")

if duplicate_count > 0:
    print("\nDuplicate SHA256 values:")
    print(
        api_final.loc[
            api_final["SHA256"].duplicated(keep=False),
            "SHA256"
        ].value_counts()
    )

# Compare API Type against common dataset if available
print("\nAPI Type distribution:")
print(api_final["Type"].value_counts().sort_index())

print("\nCommon SHA list does not contain Type,")
print("so label consistency will be checked later against DLL/PE sources.")

print("=" * 70)

DIAGNOSING 5 EXTRA API SAMPLES
Common SHA list : 29,489
API SHA set     : 29,489
API ∩ Common    : 29,489

API rows appearing in common SHA list:
29489

Duplicate API SHA rows: 5

Duplicate SHA256 values:
SHA256
c7a1c76ba2e7804bef877ca99df2b560906ffe9ebc2cc789b3c260b04db51740    3
0d515d42ef9a899209f86e59ebf4b9cb38ac2a7f0627736b4a2ff23ed47f8a66    2
1ba75ce7fa91e73e2e371601ee8895f71f051eb7d3bb57d330ec31ff186e7612    2
c731613c16ec211d0c2abab13c1c21c0f2064cc32aee0f8ed874710debe96b8a    2
Name: count, dtype: Int64

API Type distribution:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4222
6    3699
Name: count, dtype: int64

Common SHA list does not contain Type,
so label consistency will be checked later against DLL/PE sources.


In [7]:
print("=" * 70)
print("IDENTIFYING THE 5 SAMPLE DIFFERENCE")
print("=" * 70)

# The common file is currently 29,489 rows according to our load,
# so this checks whether API contains duplicate/mismatched representations.

common_sha_order = common["SHA256"].astype(str)

api_sha = api_final["SHA256"].astype(str)

# SHA values in API but not common
api_not_common = sorted(
    set(api_sha) - set(common_sha_order)
)

# SHA values in common but not API
common_not_api = sorted(
    set(common_sha_order) - set(api_sha)
)

print(f"API not in common : {len(api_not_common)}")
print(f"Common not in API : {len(common_not_api)}")

if api_not_common:
    print("\nAPI-only SHA256 values:")
    for sha in api_not_common:
        print(sha)

if common_not_api:
    print("\nCommon-only SHA256 values:")
    for sha in common_not_api:
        print(sha)

print("\n" + "=" * 70)

IDENTIFYING THE 5 SAMPLE DIFFERENCE
API not in common : 0
Common not in API : 0



In [8]:
print("=" * 70)
print("DEDUPLICATING API MATRIX")
print("=" * 70)

before = len(api_final)

# Verify duplicate SHA256 rows
duplicate_rows = api_final["SHA256"].duplicated().sum()

print(f"Rows before deduplication : {before:,}")
print(f"Duplicate rows            : {duplicate_rows:,}")

# Keep exactly one row per SHA256
api_final = (
    api_final
    .drop_duplicates(
        subset=["SHA256"],
        keep="first"
    )
    .reset_index(drop=True)
)

after = len(api_final)

print(f"Rows after deduplication  : {after:,}")

# ============================================================
# VALIDATION
# ============================================================

assert after == 29489
assert api_final["SHA256"].is_unique
assert set(api_final["SHA256"]) == set(common["SHA256"])

print("\n" + "=" * 70)
print("API ALIGNMENT VERIFIED")
print("=" * 70)

print(f"Unique samples : {len(api_final):,}")
print(f"Features       : {len(api_features):,}")
print(f"Total columns  : {api_final.shape[1]:,}")

print("\nPASS:")
print("  ✓ 29,489 unique SHA256")
print("  ✓ 200 API features")
print("  ✓ SHA256 exactly matches common dataset")
print("  ✓ No samples arbitrarily removed")

DEDUPLICATING API MATRIX
Rows before deduplication : 29,494
Duplicate rows            : 5
Rows after deduplication  : 29,489

API ALIGNMENT VERIFIED
Unique samples : 29,489
Features       : 200
Total columns  : 202

PASS:
  ✓ 29,489 unique SHA256
  ✓ 200 API features
  ✓ SHA256 exactly matches common dataset
  ✓ No samples arbitrarily removed


In [9]:
print("=" * 70)
print("VERIFYING ORIGINAL API DUPLICATES")
print("=" * 70)

# Reload only the duplicate SHA256 records from the source
duplicate_shas = [
    "c7a1c76ba2e7804bef877ca99df2b560906ffe9ebc2cc789b3c260b04db51740",
    "0d515d42ef9a899209f86e59ebf4b9cb38ac2a7f0627736b4a2ff23ed47f8a66",
    "1ba75ce7fa91e73e2e371601ee8895f71f051eb7d3bb57d330ec31ff186e7612",
    "c731613c16ec211d0c2abab13c1c21c0f2064cc32aee0f8ed874710debe96b8a"
]

print("Duplicate SHA256 values identified:")
for sha in duplicate_shas:
    print(" ", sha)

print("\nThe final API matrix now contains exactly one record")
print("for each of these SHA256 values.")
print("\nWe will cross-check their Type labels against DLL/PE")
print("during the final alignment validation.")

VERIFYING ORIGINAL API DUPLICATES
Duplicate SHA256 values identified:
  c7a1c76ba2e7804bef877ca99df2b560906ffe9ebc2cc789b3c260b04db51740
  0d515d42ef9a899209f86e59ebf4b9cb38ac2a7f0627736b4a2ff23ed47f8a66
  1ba75ce7fa91e73e2e371601ee8895f71f051eb7d3bb57d330ec31ff186e7612
  c731613c16ec211d0c2abab13c1c21c0f2064cc32aee0f8ed874710debe96b8a

The final API matrix now contains exactly one record
for each of these SHA256 values.

We will cross-check their Type labels against DLL/PE
during the final alignment validation.


In [10]:
print("=" * 70)
print("BUILDING DLL FEATURE MATRIX — MEMORY SAFE")
print("=" * 70)

# ============================================================
# DLL SOURCE
# ============================================================

DLL_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"

common_sha = set(common["SHA256"].astype(str))

print(f"Common SHA256 lookup set: {len(common_sha):,}")

# 27 selected DLL features
dll_usecols = ["SHA256", "Type"] + dll_features

print(f"Columns requested: {len(dll_usecols)}")
print("  SHA256 : 1")
print("  Type   : 1")
print("  DLLs   : 27")


# ============================================================
# CHUNKED READING
# ============================================================

dll_parts = []

CHUNK_SIZE = 2000

total_rows_read = 0
total_rows_kept = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        DLL_PATH,
        usecols=dll_usecols,
        dtype={
            "SHA256": "string",
            "Type": "uint8",
            **{feature: "uint8" for feature in dll_features}
        },
        chunksize=CHUNK_SIZE,
        low_memory=True
    ),
    start=1
):

    total_rows_read += len(chunk)

    # Keep only the exact common SHA256 set
    mask = chunk["SHA256"].isin(common_sha)

    if mask.any():
        selected = chunk.loc[mask].copy()
        dll_parts.append(selected)
        total_rows_kept += len(selected)

    del chunk

    if chunk_no % 5 == 0:
        print(
            f"Chunks: {chunk_no:5d} | "
            f"Rows read: {total_rows_read:7,d} | "
            f"Common rows: {total_rows_kept:7,d}"
        )

    gc.collect()


# ============================================================
# COMBINE
# ============================================================

dll_final = pd.concat(
    dll_parts,
    ignore_index=True
)

del dll_parts
gc.collect()


# ============================================================
# CHECK DUPLICATES
# ============================================================

duplicate_rows = dll_final["SHA256"].duplicated().sum()

print("\n" + "=" * 70)
print("DLL RAW ALIGNED MATRIX")
print("=" * 70)

print(f"Rows    : {len(dll_final):,}")
print(f"Columns : {dll_final.shape[1]:,}")
print(f"Duplicate SHA256 rows: {duplicate_rows}")


# ============================================================
# DEDUPLICATE IF NECESSARY
# ============================================================

if duplicate_rows > 0:

    print("\nDeduplicating DLL records by SHA256...")

    dll_final = (
        dll_final
        .drop_duplicates(
            subset=["SHA256"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    print(
        f"Rows after deduplication: "
        f"{len(dll_final):,}"
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(dll_final) == 29489, (
    f"Expected 29,489 DLL samples, "
    f"got {len(dll_final):,}"
)

assert dll_final["SHA256"].is_unique

assert set(dll_final["SHA256"]) == common_sha

assert dll_final.shape[1] == 29

assert dll_final["Type"].notna().all()

print("\n" + "=" * 70)
print("DLL ALIGNMENT VERIFIED")
print("=" * 70)

print(f"Unique samples : {len(dll_final):,}")
print(f"DLL features   : {len(dll_features):,}")
print(f"Total columns  : {dll_final.shape[1]:,}")

print("\nPASS:")
print("  ✓ 29,489 unique SHA256")
print("  ✓ 27 selected DLL features")
print("  ✓ SHA256 exactly matches common dataset")
print("  ✓ Type present for every sample")

BUILDING DLL FEATURE MATRIX — MEMORY SAFE
Common SHA256 lookup set: 29,489
Columns requested: 29
  SHA256 : 1
  Type   : 1
  DLLs   : 27
Chunks:     5 | Rows read:  10,000 | Common rows:  10,000
Chunks:    10 | Rows read:  20,000 | Common rows:  19,998
Chunks:    15 | Rows read:  29,498 | Common rows:  29,492

DLL RAW ALIGNED MATRIX
Rows    : 29,492
Columns : 29
Duplicate SHA256 rows: 3

Deduplicating DLL records by SHA256...
Rows after deduplication: 29,489

DLL ALIGNMENT VERIFIED
Unique samples : 29,489
DLL features   : 27
Total columns  : 29

PASS:
  ✓ 29,489 unique SHA256
  ✓ 27 selected DLL features
  ✓ SHA256 exactly matches common dataset
  ✓ Type present for every sample


In [13]:
# ============================================================
# BUILDING PE HEADER FEATURE MATRIX — MEMORY SAFE
# ============================================================

import pandas as pd
import gc

print("=" * 70)
print("BUILDING PE HEADER FEATURE MATRIX — MEMORY SAFE")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PE_HEADER_PATH = "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"

HEADER_SELECTION_PATH = (
    "/kaggle/working/PE_Header_MI_Common_29489.csv"
)

COMMON_SHA_PATH = (
    "/kaggle/working/Common_SHA256_All_Four_Datasets.csv"
)

# ------------------------------------------------------------
# LOAD COMMON SHA256
# ------------------------------------------------------------

common_sha = pd.read_csv(
    COMMON_SHA_PATH,
    usecols=["SHA256"],
    dtype={"SHA256": "string"}
)

common_sha = common_sha.drop_duplicates("SHA256")

common_set = set(common_sha["SHA256"])

print(f"Common SHA256 lookup set: {len(common_set):,}")

assert len(common_sha) == 29489

# ------------------------------------------------------------
# LOAD SELECTED PE HEADER FEATURES
# ------------------------------------------------------------

header_sel = pd.read_csv(HEADER_SELECTION_PATH)

header_features = (
    header_sel.loc[
        header_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Header features selected: {len(header_features)}")

assert len(header_features) == 46

# ------------------------------------------------------------
# COLUMNS TO LOAD
# ------------------------------------------------------------

required_cols = ["SHA256", "Type"] + header_features

print(f"Columns requested: {len(required_cols)}")
print(f"  SHA256 : 1")
print(f"  Type   : 1")
print(f"  Header : {len(header_features)}")

# ------------------------------------------------------------
# MEMORY-SAFE CHUNKED READ
# ------------------------------------------------------------

chunks = []

rows_read = 0
common_rows = 0
chunk_no = 0

for chunk in pd.read_csv(
    PE_HEADER_PATH,
    usecols=required_cols,
    chunksize=2000,
    low_memory=False
):
    
    chunk_no += 1
    rows_read += len(chunk)

    # Keep only common SHA256s
    chunk = chunk[
        chunk["SHA256"].isin(common_set)
    ]

    common_rows += len(chunk)

    if len(chunk) > 0:
        chunks.append(chunk)

    if chunk_no % 5 == 0:
        print(
            f"Chunks: {chunk_no:5d} | "
            f"Rows read: {rows_read:7,d} | "
            f"Common rows: {common_rows:7,d}"
        )

# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

header_raw = pd.concat(
    chunks,
    ignore_index=True
)

del chunks
gc.collect()

print()
print("=" * 70)
print("PE HEADER RAW ALIGNED MATRIX")
print("=" * 70)

print(f"Rows    : {len(header_raw):,}")
print(f"Columns : {header_raw.shape[1]}")

# ------------------------------------------------------------
# CHECK DUPLICATES
# ------------------------------------------------------------

duplicate_count = (
    header_raw["SHA256"].duplicated().sum()
)

print(f"Duplicate SHA256 rows: {duplicate_count}")

# ------------------------------------------------------------
# DEDUPLICATE
# ------------------------------------------------------------

if duplicate_count > 0:

    print()
    print("Deduplicating PE Header records by SHA256...")

    header_final = (
        header_raw
        .drop_duplicates(
            subset=["SHA256"],
            keep="first"
        )
        .reset_index(drop=True)
    )

else:

    header_final = header_raw

del header_raw
gc.collect()

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

print()
print("=" * 70)
print("PE HEADER ALIGNMENT VERIFICATION")
print("=" * 70)

print(f"Unique samples : {header_final['SHA256'].nunique():,}")
print(f"Header features: {len(header_features)}")
print(f"Total columns  : {header_final.shape[1]}")

# Exact sample count
assert len(header_final) == 29489

# Unique SHA256
assert header_final["SHA256"].nunique() == 29489

# Exact feature count
assert len(header_features) == 46

# Expected columns
assert header_final.shape[1] == 48

# SHA256 must exactly match common dataset
header_sha = set(header_final["SHA256"])

assert header_sha == common_set

# Type must exist
assert "Type" in header_final.columns

# Check missing values
missing_total = header_final[required_cols].isna().sum().sum()

print(f"Missing feature values: {missing_total}")

assert missing_total == 0

print()
print("PASS:")
print("  ✓ 29,489 unique SHA256")
print("  ✓ 46 selected PE Header features")
print("  ✓ SHA256 exactly matches common dataset")
print("  ✓ Type present for every sample")
print("  ✓ No missing feature values")
print("  ✓ 48 total columns")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

HEADER_FINAL_PATH = (
    "/kaggle/working/PE_Header_Final_29489.csv"
)

header_final.to_csv(
    HEADER_FINAL_PATH,
    index=False
)

print()
print("Saved:")
print(HEADER_FINAL_PATH)

print()
print("=" * 70)
print("PE HEADER MATRIX COMPLETE")
print("=" * 70)

gc.collect()

BUILDING PE HEADER FEATURE MATRIX — MEMORY SAFE
Common SHA256 lookup set: 29,489
PE Header features selected: 46
Columns requested: 48
  SHA256 : 1
  Type   : 1
  Header : 46
Chunks:     5 | Rows read:  10,000 | Common rows:   9,834
Chunks:    10 | Rows read:  20,000 | Common rows:  19,718
Chunks:    15 | Rows read:  29,807 | Common rows:  29,489

PE HEADER RAW ALIGNED MATRIX
Rows    : 29,489
Columns : 48
Duplicate SHA256 rows: 0

PE HEADER ALIGNMENT VERIFICATION
Unique samples : 29,489
Header features: 46
Total columns  : 48
Missing feature values: 0

PASS:
  ✓ 29,489 unique SHA256
  ✓ 46 selected PE Header features
  ✓ SHA256 exactly matches common dataset
  ✓ Type present for every sample
  ✓ No missing feature values
  ✓ 48 total columns

Saved:
/kaggle/working/PE_Header_Final_29489.csv

PE HEADER MATRIX COMPLETE


0

In [14]:
# ============================================================
# BUILDING PE SECTION FEATURE MATRIX — MEMORY SAFE
# ============================================================

import pandas as pd
import gc

print("=" * 70)
print("BUILDING PE SECTION FEATURE MATRIX — MEMORY SAFE")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PE_SECTION_PATH = (
    "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"
)

SECTION_SELECTION_PATH = (
    "/kaggle/working/PE_Section_MI_Common_29489.csv"
)

COMMON_SHA_PATH = (
    "/kaggle/working/Common_SHA256_All_Four_Datasets.csv"
)

# ------------------------------------------------------------
# LOAD COMMON SHA256
# ------------------------------------------------------------

common_sha = pd.read_csv(
    COMMON_SHA_PATH,
    usecols=["SHA256"],
    dtype={"SHA256": "string"}
)

common_sha = common_sha.drop_duplicates("SHA256")

common_set = set(common_sha["SHA256"])

print(f"Common SHA256 lookup set: {len(common_set):,}")

assert len(common_sha) == 29489

# ------------------------------------------------------------
# LOAD SELECTED PE SECTION FEATURES
# ------------------------------------------------------------

section_sel = pd.read_csv(SECTION_SELECTION_PATH)

section_features = (
    section_sel.loc[
        section_sel["MI_Score"] >= 0.01,
        "Feature"
    ]
    .astype(str)
    .tolist()
)

print(f"PE Section features selected: {len(section_features)}")

assert len(section_features) == 45

# ------------------------------------------------------------
# COLUMNS TO LOAD
# ------------------------------------------------------------

required_cols = [
    "SHA256",
    "Type"
] + section_features

print(f"Columns requested: {len(required_cols)}")
print(f"  SHA256  : 1")
print(f"  Type    : 1")
print(f"  Section : {len(section_features)}")

# ------------------------------------------------------------
# MEMORY-SAFE CHUNKED READ
# ------------------------------------------------------------

chunks = []

rows_read = 0
common_rows = 0
chunk_no = 0

for chunk in pd.read_csv(
    PE_SECTION_PATH,
    usecols=required_cols,
    chunksize=2000,
    low_memory=False
):

    chunk_no += 1
    rows_read += len(chunk)

    # Keep only common SHA256 samples
    chunk = chunk[
        chunk["SHA256"].isin(common_set)
    ]

    common_rows += len(chunk)

    if len(chunk) > 0:
        chunks.append(chunk)

    if chunk_no % 5 == 0:
        print(
            f"Chunks: {chunk_no:5d} | "
            f"Rows read: {rows_read:7,d} | "
            f"Common rows: {common_rows:7,d}"
        )

# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

section_raw = pd.concat(
    chunks,
    ignore_index=True
)

del chunks
gc.collect()

print()
print("=" * 70)
print("PE SECTION RAW ALIGNED MATRIX")
print("=" * 70)

print(f"Rows    : {len(section_raw):,}")
print(f"Columns : {section_raw.shape[1]}")

# ------------------------------------------------------------
# CHECK DUPLICATES
# ------------------------------------------------------------

duplicate_count = (
    section_raw["SHA256"].duplicated().sum()
)

print(f"Duplicate SHA256 rows: {duplicate_count}")

# ------------------------------------------------------------
# DEDUPLICATE IF NECESSARY
# ------------------------------------------------------------

if duplicate_count > 0:

    print()
    print("Deduplicating PE Section records by SHA256...")

    section_final = (
        section_raw
        .drop_duplicates(
            subset=["SHA256"],
            keep="first"
        )
        .reset_index(drop=True)
    )

else:

    section_final = section_raw

del section_raw
gc.collect()

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

print()
print("=" * 70)
print("PE SECTION ALIGNMENT VERIFICATION")
print("=" * 70)

print(
    f"Unique samples : "
    f"{section_final['SHA256'].nunique():,}"
)

print(
    f"Section features: "
    f"{len(section_features)}"
)

print(
    f"Total columns  : "
    f"{section_final.shape[1]}"
)

# Expected sample count
assert len(section_final) == 29489

# Unique SHA256
assert section_final["SHA256"].nunique() == 29489

# Expected feature count
assert len(section_features) == 45

# Expected total columns
assert section_final.shape[1] == 47

# Exact SHA256 alignment
section_sha = set(section_final["SHA256"])

assert section_sha == common_set

# Type must exist
assert "Type" in section_final.columns

# Missing values
missing_total = (
    section_final[required_cols]
    .isna()
    .sum()
    .sum()
)

print(f"Missing feature values: {missing_total}")

assert missing_total == 0

print()
print("PASS:")
print("  ✓ 29,489 unique SHA256")
print("  ✓ 45 selected PE Section features")
print("  ✓ SHA256 exactly matches common dataset")
print("  ✓ Type present for every sample")
print("  ✓ No missing feature values")
print("  ✓ 47 total columns")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

SECTION_FINAL_PATH = (
    "/kaggle/working/PE_Section_Final_29489.csv"
)

section_final.to_csv(
    SECTION_FINAL_PATH,
    index=False
)

print()
print("Saved:")
print(SECTION_FINAL_PATH)

print()
print("=" * 70)
print("PE SECTION MATRIX COMPLETE")
print("=" * 70)

gc.collect()


BUILDING PE SECTION FEATURE MATRIX — MEMORY SAFE
Common SHA256 lookup set: 29,489
PE Section features selected: 45
Columns requested: 47
  SHA256  : 1
  Type    : 1
  Section : 45
Chunks:     5 | Rows read:  10,000 | Common rows:   9,834
Chunks:    10 | Rows read:  20,000 | Common rows:  19,745
Chunks:    15 | Rows read:  29,760 | Common rows:  29,489

PE SECTION RAW ALIGNED MATRIX
Rows    : 29,489
Columns : 47
Duplicate SHA256 rows: 0

PE SECTION ALIGNMENT VERIFICATION
Unique samples : 29,489
Section features: 45
Total columns  : 47
Missing feature values: 0

PASS:
  ✓ 29,489 unique SHA256
  ✓ 45 selected PE Section features
  ✓ SHA256 exactly matches common dataset
  ✓ Type present for every sample
  ✓ No missing feature values
  ✓ 47 total columns

Saved:
/kaggle/working/PE_Section_Final_29489.csv

PE SECTION MATRIX COMPLETE


0

In [15]:
# ============================================================
# FINAL DATASET MERGE — 318 FEATURES
# ============================================================

import pandas as pd
import gc

print("=" * 70)
print("FINAL DATASET MERGE")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

API_FINAL_PATH = "/kaggle/working/API_Final_29489.csv"
DLL_FINAL_PATH = "/kaggle/working/DLL_Final_29489.csv"
HEADER_FINAL_PATH = "/kaggle/working/PE_Header_Final_29489.csv"
SECTION_FINAL_PATH = "/kaggle/working/PE_Section_Final_29489.csv"

FINAL_PATH = "/kaggle/working/Windows_Malware_Final_318_Features.csv"

# ------------------------------------------------------------
# LOAD MATRICES
# ------------------------------------------------------------

print("\nLoading API...")
api = pd.read_csv(API_FINAL_PATH)

print("Loading DLL...")
dll = pd.read_csv(DLL_FINAL_PATH)

print("Loading PE Header...")
header = pd.read_csv(HEADER_FINAL_PATH)

print("Loading PE Section...")
section = pd.read_csv(SECTION_FINAL_PATH)

# ------------------------------------------------------------
# BASIC SHAPE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INPUT MATRIX SHAPES")
print("=" * 70)

print(f"API        : {api.shape}")
print(f"DLL        : {dll.shape}")
print(f"PE Header  : {header.shape}")
print(f"PE Section : {section.shape}")

assert api.shape == (29489, 202)
assert dll.shape == (29489, 29)
assert header.shape == (29489, 48)
assert section.shape == (29489, 47)

# ------------------------------------------------------------
# SHA256 UNIQUENESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING SHA256 UNIQUENESS")
print("=" * 70)

for name, df in [
    ("API", api),
    ("DLL", dll),
    ("PE Header", header),
    ("PE Section", section)
]:

    duplicates = df["SHA256"].duplicated().sum()

    print(
        f"{name:12s}: "
        f"{df['SHA256'].nunique():,} unique | "
        f"{duplicates} duplicates"
    )

    assert df["SHA256"].nunique() == 29489
    assert duplicates == 0

# ------------------------------------------------------------
# SHA256 ALIGNMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING SHA256 ALIGNMENT")
print("=" * 70)

api_sha = set(api["SHA256"])
dll_sha = set(dll["SHA256"])
header_sha = set(header["SHA256"])
section_sha = set(section["SHA256"])

assert api_sha == dll_sha
assert api_sha == header_sha
assert api_sha == section_sha

print("PASS: All four matrices contain exactly the same SHA256 set.")

# ------------------------------------------------------------
# TYPE CONSISTENCY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING TYPE CONSISTENCY")
print("=" * 70)

# Align by SHA256 before comparison
dll_type = (
    dll.set_index("SHA256")["Type"]
    .reindex(api["SHA256"])
    .reset_index(drop=True)
)

header_type = (
    header.set_index("SHA256")["Type"]
    .reindex(api["SHA256"])
    .reset_index(drop=True)
)

section_type = (
    section.set_index("SHA256")["Type"]
    .reindex(api["SHA256"])
    .reset_index(drop=True)
)

api_type = api["Type"].reset_index(drop=True)

print(
    "API vs DLL inconsistencies    :",
    (api_type != dll_type).sum()
)

print(
    "API vs PE Header inconsistencies:",
    (api_type != header_type).sum()
)

print(
    "API vs PE Section inconsistencies:",
    (api_type != section_type).sum()
)

assert (api_type == dll_type).all()
assert (api_type == header_type).all()
assert (api_type == section_type).all()

print("PASS: Type labels are completely consistent.")

# ------------------------------------------------------------
# REMOVE DUPLICATE TYPE COLUMNS
# ------------------------------------------------------------

dll_features = [
    c for c in dll.columns
    if c not in ["SHA256", "Type"]
]

header_features = [
    c for c in header.columns
    if c not in ["SHA256", "Type"]
]

section_features = [
    c for c in section.columns
    if c not in ["SHA256", "Type"]
]

api_features = [
    c for c in api.columns
    if c not in ["SHA256", "Type"]
]

print("\n" + "=" * 70)
print("FEATURE COUNTS")
print("=" * 70)

print(f"API        : {len(api_features)}")
print(f"DLL        : {len(dll_features)}")
print(f"PE Header  : {len(header_features)}")
print(f"PE Section : {len(section_features)}")

total_features = (
    len(api_features)
    + len(dll_features)
    + len(header_features)
    + len(section_features)
)

print("-" * 70)
print(f"TOTAL      : {total_features}")

assert len(api_features) == 200
assert len(dll_features) == 27
assert len(header_features) == 46
assert len(section_features) == 45
assert total_features == 318

# ------------------------------------------------------------
# CHECK FEATURE NAME COLLISIONS
# ------------------------------------------------------------

all_features = (
    api_features
    + dll_features
    + header_features
    + section_features
)

duplicate_features = (
    pd.Series(all_features)
    [pd.Series(all_features).duplicated()]
    .tolist()
)

print(f"\nDuplicate feature names: {len(duplicate_features)}")

assert len(duplicate_features) == 0

print("PASS: All 318 feature names are unique.")

# ------------------------------------------------------------
# CONSTRUCT FINAL DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSTRUCTING FINAL DATASET")
print("=" * 70)

# API is our base because it already has the exact common SHA order
final = api[["SHA256", "Type"] + api_features].copy()

# Add DLL features in SHA256 order
dll_aligned = (
    dll.set_index("SHA256")
    .reindex(final["SHA256"])
)

final[dll_features] = (
    dll_aligned[dll_features]
    .to_numpy()
)

del dll_aligned
gc.collect()

# Add PE Header features
header_aligned = (
    header.set_index("SHA256")
    .reindex(final["SHA256"])
)

final[header_features] = (
    header_aligned[header_features]
    .to_numpy()
)

del header_aligned
gc.collect()

# Add PE Section features
section_aligned = (
    section.set_index("SHA256")
    .reindex(final["SHA256"])
)

final[section_features] = (
    section_aligned[section_features]
    .to_numpy()
)

del section_aligned
gc.collect()

# ------------------------------------------------------------
# FINAL SHAPE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DATASET SHAPE")
print("=" * 70)

print(f"Rows    : {final.shape[0]:,}")
print(f"Columns : {final.shape[1]:,}")

assert final.shape == (29489, 320)

# ------------------------------------------------------------
# FINAL SHA CHECK
# ------------------------------------------------------------

assert final["SHA256"].nunique() == 29489

# ------------------------------------------------------------
# FINAL MISSING VALUE CHECK
# ------------------------------------------------------------

missing_total = final.isna().sum().sum()

print(f"Missing values: {missing_total}")

assert missing_total == 0

# ------------------------------------------------------------
# FINAL TYPE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TARGET DISTRIBUTION")
print("=" * 70)

type_distribution = (
    final["Type"]
    .value_counts()
    .sort_index()
    .rename_axis("Type")
    .reset_index(name="Samples")
)

type_distribution["Percentage"] = (
    type_distribution["Samples"]
    / len(final)
    * 100
)

print(type_distribution.to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

final.to_csv(
    FINAL_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FINAL DATASET CREATED")
print("=" * 70)

print(f"Saved:")
print(FINAL_PATH)

print()
print("PASS:")
print("  ✓ 29,489 samples")
print("  ✓ 318 ML features")
print("  ✓ 1 SHA256 identifier")
print("  ✓ 1 Type target")
print("  ✓ 320 total columns")
print("  ✓ Unique SHA256")
print("  ✓ Complete SHA256 alignment")
print("  ✓ Consistent Type labels")
print("  ✓ No duplicate feature names")
print("  ✓ No missing values")

gc.collect()

FINAL DATASET MERGE

Loading API...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/API_Final_29489.csv'

In [16]:
import os

for f in [
    "/kaggle/working/API_Final_29489.csv",
    "/kaggle/working/DLL_Final_29489.csv",
    "/kaggle/working/PE_Header_Final_29489.csv",
    "/kaggle/working/PE_Section_Final_29489.csv"
]:
    print(
        "OK" if os.path.exists(f) else "MISSING",
        os.path.basename(f)
    )

MISSING API_Final_29489.csv
MISSING DLL_Final_29489.csv
OK PE_Header_Final_29489.csv
OK PE_Section_Final_29489.csv


In [17]:
print("api_final exists   :", "api_final" in globals())
print("dll_final exists   :", "dll_final" in globals())
print("header_final exists:", "header_final" in globals())
print("section_final exists:", "section_final" in globals())

api_final exists   : True
dll_final exists   : True
header_final exists: True
section_final exists: True


In [18]:
api_final.to_csv(
    "/kaggle/working/API_Final_29489.csv",
    index=False
)

dll_final.to_csv(
    "/kaggle/working/DLL_Final_29489.csv",
    index=False
)

print("API and DLL final matrices saved.")

API and DLL final matrices saved.


In [19]:
# ============================================================
# SAVING API + DLL FINAL MATRICES
# ============================================================

API_FINAL_PATH = "/kaggle/working/API_Final_29489.csv"
DLL_FINAL_PATH = "/kaggle/working/DLL_Final_29489.csv"

api_final.to_csv(
    API_FINAL_PATH,
    index=False
)

dll_final.to_csv(
    DLL_FINAL_PATH,
    index=False
)

print("=" * 70)
print("API + DLL FINAL MATRICES SAVED")
print("=" * 70)

print(f"API : {API_FINAL_PATH}")
print(f"DLL : {DLL_FINAL_PATH}")

print()
print("API shape :", api_final.shape)
print("DLL shape :", dll_final.shape)

assert api_final.shape == (29489, 202)
assert dll_final.shape == (29489, 29)

print()
print("PASS: Both matrices saved successfully.")

API + DLL FINAL MATRICES SAVED
API : /kaggle/working/API_Final_29489.csv
DLL : /kaggle/working/DLL_Final_29489.csv

API shape : (29489, 202)
DLL shape : (29489, 29)

PASS: Both matrices saved successfully.


In [20]:
# ============================================================
# FINAL MERGE — 318 FEATURE DATASET
# ============================================================

import pandas as pd
import gc

print("=" * 70)
print("FINAL MERGE OF ALL FOUR FEATURE MATRICES")
print("=" * 70)

# ------------------------------------------------------------
# EXPECTED SHAPES
# ------------------------------------------------------------

print("\nInput shapes:")
print(f"API        : {api_final.shape}")
print(f"DLL        : {dll_final.shape}")
print(f"PE Header  : {header_final.shape}")
print(f"PE Section : {section_final.shape}")

assert api_final.shape == (29489, 202)
assert dll_final.shape == (29489, 29)
assert header_final.shape == (29489, 48)
assert section_final.shape == (29489, 47)

# ------------------------------------------------------------
# GET FEATURE NAMES
# ------------------------------------------------------------

api_features = [
    c for c in api_final.columns
    if c not in ["SHA256", "Type"]
]

dll_features = [
    c for c in dll_final.columns
    if c not in ["SHA256", "Type"]
]

header_features = [
    c for c in header_final.columns
    if c not in ["SHA256", "Type"]
]

section_features = [
    c for c in section_final.columns
    if c not in ["SHA256", "Type"]
]

print("\nFeature counts:")
print(f"API        : {len(api_features)}")
print(f"DLL        : {len(dll_features)}")
print(f"PE Header  : {len(header_features)}")
print(f"PE Section : {len(section_features)}")

assert len(api_features) == 200
assert len(dll_features) == 27
assert len(header_features) == 46
assert len(section_features) == 45

all_features = (
    api_features
    + dll_features
    + header_features
    + section_features
)

assert len(all_features) == 318

# ------------------------------------------------------------
# CHECK FEATURE NAME COLLISIONS
# ------------------------------------------------------------

duplicate_features = (
    pd.Series(all_features)
    .duplicated()
    .sum()
)

print(f"\nDuplicate feature names: {duplicate_features}")

assert duplicate_features == 0

print("PASS: All 318 feature names are unique.")

# ------------------------------------------------------------
# CHECK SHA256 ALIGNMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING SHA256 ALIGNMENT")
print("=" * 70)

api_sha = set(api_final["SHA256"])
dll_sha = set(dll_final["SHA256"])
header_sha = set(header_final["SHA256"])
section_sha = set(section_final["SHA256"])

assert len(api_sha) == 29489
assert len(dll_sha) == 29489
assert len(header_sha) == 29489
assert len(section_sha) == 29489

assert api_sha == dll_sha
assert api_sha == header_sha
assert api_sha == section_sha

print("PASS: All four matrices contain exactly the same 29,489 SHA256 values.")

# ------------------------------------------------------------
# CHECK TYPE CONSISTENCY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING TYPE CONSISTENCY")
print("=" * 70)

# Compare labels by SHA256 rather than row position
api_type = (
    api_final
    .set_index("SHA256")["Type"]
)

dll_type = (
    dll_final
    .set_index("SHA256")["Type"]
    .reindex(api_type.index)
)

header_type = (
    header_final
    .set_index("SHA256")["Type"]
    .reindex(api_type.index)
)

section_type = (
    section_final
    .set_index("SHA256")["Type"]
    .reindex(api_type.index)
)

dll_mismatch = (api_type != dll_type).sum()
header_mismatch = (api_type != header_type).sum()
section_mismatch = (api_type != section_type).sum()

print(f"API vs DLL        : {dll_mismatch}")
print(f"API vs PE Header  : {header_mismatch}")
print(f"API vs PE Section : {section_mismatch}")

assert dll_mismatch == 0
assert header_mismatch == 0
assert section_mismatch == 0

print("PASS: Type labels are consistent across all datasets.")

# ------------------------------------------------------------
# BUILD FINAL DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSTRUCTING FINAL DATASET")
print("=" * 70)

# API provides the base SHA256 + Type
final_df = api_final[
    ["SHA256", "Type"] + api_features
].copy()

# ------------------------------------------------------------
# ADD DLL FEATURES
# ------------------------------------------------------------

dll_aligned = (
    dll_final
    .set_index("SHA256")
    .reindex(final_df["SHA256"])
)

final_df[dll_features] = (
    dll_aligned[dll_features]
    .to_numpy()
)

del dll_aligned
gc.collect()

print("✓ API + DLL merged")

# ------------------------------------------------------------
# ADD PE HEADER FEATURES
# ------------------------------------------------------------

header_aligned = (
    header_final
    .set_index("SHA256")
    .reindex(final_df["SHA256"])
)

final_df[header_features] = (
    header_aligned[header_features]
    .to_numpy()
)

del header_aligned
gc.collect()

print("✓ PE Header merged")

# ------------------------------------------------------------
# ADD PE SECTION FEATURES
# ------------------------------------------------------------

section_aligned = (
    section_final
    .set_index("SHA256")
    .reindex(final_df["SHA256"])
)

final_df[section_features] = (
    section_aligned[section_features]
    .to_numpy()
)

del section_aligned
gc.collect()

print("✓ PE Section merged")

# ------------------------------------------------------------
# FINAL STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DATASET STRUCTURE")
print("=" * 70)

print(f"Rows    : {final_df.shape[0]:,}")
print(f"Columns : {final_df.shape[1]:,}")

assert final_df.shape == (29489, 320)

# ------------------------------------------------------------
# FINAL SHA VALIDATION
# ------------------------------------------------------------

assert final_df["SHA256"].nunique() == 29489
assert final_df["SHA256"].isna().sum() == 0

# ------------------------------------------------------------
# FINAL FEATURE VALIDATION
# ------------------------------------------------------------

model_features = [
    c for c in final_df.columns
    if c not in ["SHA256", "Type"]
]

assert len(model_features) == 318

assert len(set(model_features)) == 318

print(f"ML features: {len(model_features)}")

# ------------------------------------------------------------
# FINAL MISSING VALUE CHECK
# ------------------------------------------------------------

missing_total = final_df.isna().sum().sum()

print(f"Missing values: {missing_total}")

assert missing_total == 0

# ------------------------------------------------------------
# FINAL DUPLICATE ROW CHECK
# ------------------------------------------------------------

duplicate_sha = final_df["SHA256"].duplicated().sum()

print(f"Duplicate SHA256: {duplicate_sha}")

assert duplicate_sha == 0

# ------------------------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TARGET DISTRIBUTION")
print("=" * 70)

target_distribution = (
    final_df["Type"]
    .value_counts()
    .sort_index()
)

print(target_distribution)

print("\nPercentages:")

print(
    (target_distribution / len(final_df) * 100)
    .round(2)
)

# ------------------------------------------------------------
# SAVE FINAL DATASET
# ------------------------------------------------------------

FINAL_PATH = (
    "/kaggle/working/Windows_Malware_Final_318_Features.csv"
)

final_df.to_csv(
    FINAL_PATH,
    index=False
)

print("\n" + "=" * 70)
print("FINAL DATASET SAVED")
print("=" * 70)

print(FINAL_PATH)

print("\n" + "=" * 70)
print("FINAL VALIDATION PASS")
print("=" * 70)

print("✓ 29,489 samples")
print("✓ 318 ML features")
print("✓ 1 SHA256 identifier")
print("✓ 1 Type target")
print("✓ 320 total columns")
print("✓ SHA256 aligned across all four sources")
print("✓ SHA256 unique")
print("✓ Type labels consistent")
print("✓ Feature names unique")
print("✓ No missing values")

gc.collect()

FINAL MERGE OF ALL FOUR FEATURE MATRICES

Input shapes:
API        : (29489, 202)
DLL        : (29489, 29)
PE Header  : (29489, 48)
PE Section : (29489, 47)

Feature counts:
API        : 200
DLL        : 27
PE Header  : 46
PE Section : 45

Duplicate feature names: 0
PASS: All 318 feature names are unique.

CHECKING SHA256 ALIGNMENT
PASS: All four matrices contain exactly the same 29,489 SHA256 values.

CHECKING TYPE CONSISTENCY
API vs DLL        : 0
API vs PE Header  : 0
API vs PE Section : 0
PASS: Type labels are consistent across all datasets.

CONSTRUCTING FINAL DATASET
✓ API + DLL merged
✓ PE Header merged
✓ PE Section merged

FINAL DATASET STRUCTURE
Rows    : 29,489
Columns : 320
ML features: 318
Missing values: 0
Duplicate SHA256: 0

FINAL TARGET DISTRIBUTION
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64

Percentages:
Type
0     6.37
1    17.03
2    15.74
3    16.80
4    17.21
5    14.30
6    12.54
Name: count, dtype: float64

/tmp/ipykernel_264/2083021471.py:220: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[section_features] = (
/tmp/ipykernel_264/2083021471.py:220: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[section_features] = (
/tmp/ipykernel_264/2083021471.py:220: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra


FINAL DATASET SAVED
/kaggle/working/Windows_Malware_Final_318_Features.csv

FINAL VALIDATION PASS
✓ 29,489 samples
✓ 318 ML features
✓ 1 SHA256 identifier
✓ 1 Type target
✓ 320 total columns
✓ SHA256 aligned across all four sources
✓ SHA256 unique
✓ Type labels consistent
✓ Feature names unique
✓ No missing values


0

In [21]:
# ============================================================
# FINAL DATASET SANITY CHECK
# ============================================================

import os
import gc
import pandas as pd
import numpy as np

FINAL_PATH = "/kaggle/working/Windows_Malware_Final_318_Features.csv"

print("=" * 70)
print("FINAL DATASET SANITY CHECK")
print("=" * 70)

# ============================================================
# 1. FILE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FILE CHECK")
print("=" * 70)

assert os.path.exists(FINAL_PATH), f"Dataset not found: {FINAL_PATH}"

file_size_mb = os.path.getsize(FINAL_PATH) / (1024 ** 2)

print(f"File exists : PASS")
print(f"File size   : {file_size_mb:.2f} MB")
print(f"Path        : {FINAL_PATH}")

# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\n" + "=" * 70)
print("LOADING DATASET")
print("=" * 70)

df = pd.read_csv(FINAL_PATH)

print(f"Shape: {df.shape}")

# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURE CHECK")
print("=" * 70)

assert df.shape[0] == 29489, \
    f"Expected 29,489 rows, got {df.shape[0]}"

assert df.shape[1] == 320, \
    f"Expected 320 columns, got {df.shape[1]}"

assert "SHA256" in df.columns
assert "Type" in df.columns

feature_columns = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

assert len(feature_columns) == 318

print(f"Rows          : {len(df):,}")
print(f"ML features   : {len(feature_columns)}")
print(f"SHA256 column : present")
print(f"Type column   : present")

# ============================================================
# 4. SHA256 CHECK
# ============================================================

print("\n" + "=" * 70)
print("SHA256 CHECK")
print("=" * 70)

sha_unique = df["SHA256"].nunique()
sha_missing = df["SHA256"].isna().sum()

print(f"Total SHA256 values : {len(df):,}")
print(f"Unique SHA256       : {sha_unique:,}")
print(f"Missing SHA256      : {sha_missing}")

assert sha_unique == 29489
assert sha_missing == 0

print("PASS: SHA256 values are unique and complete.")

# ============================================================
# 5. TARGET CHECK
# ============================================================

print("\n" + "=" * 70)
print("TARGET CHECK")
print("=" * 70)

print("\nTarget dtype:")
print(df["Type"].dtype)

print("\nTarget values:")
print(sorted(df["Type"].unique()))

print("\nTarget distribution:")
target_counts = df["Type"].value_counts().sort_index()
target_percent = (target_counts / len(df) * 100).round(2)

target_summary = pd.DataFrame({
    "Samples": target_counts,
    "Percentage": target_percent
})

print(target_summary)

assert df["Type"].isna().sum() == 0
assert set(df["Type"].unique()) == set(range(7))

print("\nPASS: All 7 target classes are present.")

# ============================================================
# 6. MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

missing_total = df.isna().sum().sum()

print(f"Total missing values: {missing_total}")

if missing_total > 0:
    missing_features = df.columns[df.isna().any()].tolist()

    print("\nFeatures containing missing values:")
    for col in missing_features:
        print(f"  {col}")

else:
    print("PASS: No missing values.")

assert missing_total == 0

# ============================================================
# 7. DUPLICATE FEATURE NAMES
# ============================================================

print("\n" + "=" * 70)
print("FEATURE NAME CHECK")
print("=" * 70)

duplicate_columns = df.columns[df.columns.duplicated()].tolist()

print(f"Duplicate column names: {len(duplicate_columns)}")

assert len(duplicate_columns) == 0

print("PASS: All column names are unique.")

# ============================================================
# 8. DATA TYPES
# ============================================================

print("\n" + "=" * 70)
print("DATA TYPE CHECK")
print("=" * 70)

print(df.dtypes.value_counts())

non_numeric_features = df[feature_columns].select_dtypes(
    exclude=[np.number]
).columns.tolist()

print(f"\nNon-numeric ML features: {len(non_numeric_features)}")

if non_numeric_features:
    print(non_numeric_features)

assert len(non_numeric_features) == 0

print("PASS: All 318 ML features are numeric.")

# ============================================================
# 9. INFINITE VALUES
# ============================================================

print("\n" + "=" * 70)
print("INFINITE VALUE CHECK")
print("=" * 70)

numeric_values = df[feature_columns].to_numpy(dtype=np.float64)

inf_count = np.isinf(numeric_values).sum()

print(f"Infinite values: {inf_count}")

assert inf_count == 0

print("PASS: No infinite feature values.")

del numeric_values
gc.collect()

# ============================================================
# 10. CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CONSTANT FEATURE CHECK")
print("=" * 70)

constant_features = [
    col for col in feature_columns
    if df[col].nunique(dropna=False) <= 1
]

print(f"Constant features: {len(constant_features)}")

if constant_features:
    print("\nConstant features:")
    for col in constant_features:
        print(f"  {col}")

# We expect none because constants were already removed
assert len(constant_features) == 0

print("PASS: No constant ML features.")

# ============================================================
# 11. FEATURE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FEATURE RANGE SUMMARY")
print("=" * 70)

summary = pd.DataFrame({
    "Min": df[feature_columns].min(),
    "Max": df[feature_columns].max(),
    "Unique": df[feature_columns].nunique()
})

print("\nLowest-cardinality features:")
print(
    summary.sort_values("Unique")
    .head(15)
    .to_string()
)

print("\nHighest-cardinality features:")
print(
    summary.sort_values("Unique", ascending=False)
    .head(15)
    .to_string()
)

# ============================================================
# 12. FINAL VERDICT
# ============================================================

print("\n" + "=" * 70)
print("FINAL SANITY CHECK RESULT")
print("=" * 70)

print("PASS: Dataset is ready for ML preprocessing.")
print()
print("Samples       : 29,489")
print("ML features   : 318")
print("Target        : Type")
print("Classes       : 7")
print("Missing       : 0")
print("Infinite      : 0")
print("Duplicate SHA : 0")
print("Constant      : 0")
print("Duplicate cols: 0")
print()
print("NEXT STEP: TRAIN / TEST SPLIT + PREPROCESSING")
print("=" * 70)

gc.collect()

FINAL DATASET SANITY CHECK

FILE CHECK
File exists : PASS
File size   : 24.19 MB
Path        : /kaggle/working/Windows_Malware_Final_318_Features.csv

LOADING DATASET
Shape: (29489, 320)

STRUCTURE CHECK
Rows          : 29,489
ML features   : 318
SHA256 column : present
Type column   : present

SHA256 CHECK
Total SHA256 values : 29,489
Unique SHA256       : 29,489
Missing SHA256      : 0
PASS: SHA256 values are unique and complete.

TARGET CHECK

Target dtype:
int64

Target values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Target distribution:
      Samples  Percentage
Type                     
0        1877        6.37
1        5022       17.03
2        4643       15.74
3        4955       16.80
4        5076       17.21
5        4217       14.30
6        3699       12.54

PASS: All 7 target classes are present.

MISSING VALUE CHECK
Total missing values: 0
PASS: No missing values.

FEATURE NAME CHECK
Duplicate column names: 0
PASS: Al

0

In [22]:
# ============================================================
# TRAIN / TEST SPLIT + PREPROCESSING SETUP
# ============================================================

import pandas as pd
import numpy as np
import gc

from sklearn.model_selection import train_test_split

print("=" * 70)
print("TRAIN / TEST SPLIT + PREPROCESSING SETUP")
print("=" * 70)

FINAL_PATH = "/kaggle/working/Windows_Malware_Final_318_Features.csv"

# ============================================================
# 1. LOAD FINAL DATASET
# ============================================================

print("\nLoading final dataset...")

df = pd.read_csv(FINAL_PATH)

print(f"Dataset shape: {df.shape}")

# ============================================================
# 2. SEPARATE IDENTIFIER / TARGET / FEATURES
# ============================================================

print("\n" + "=" * 70)
print("SEPARATING DATA")
print("=" * 70)

X = df.drop(columns=["SHA256", "Type"])
y = df["Type"]

sha256 = df["SHA256"]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]}")

assert X.shape == (29489, 318)
assert len(y) == 29489

# SHA256 must NOT enter the ML model
print("\nPASS: SHA256 excluded from ML features.")

# ============================================================
# 3. STRATIFIED TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 70)
print("STRATIFIED TRAIN / TEST SPLIT")
print("=" * 70)

X_train, X_test, y_train, y_test, sha_train, sha_test = train_test_split(
    X,
    y,
    sha256,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTraining samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")

print("\nTraining distribution:")
train_dist = y_train.value_counts().sort_index()
print(train_dist)

print("\nTest distribution:")
test_dist = y_test.value_counts().sort_index()
print(test_dist)

# ============================================================
# 4. VERIFY STRATIFICATION
# ============================================================

print("\n" + "=" * 70)
print("STRATIFICATION CHECK")
print("=" * 70)

original_pct = y.value_counts(normalize=True).sort_index() * 100
train_pct = y_train.value_counts(normalize=True).sort_index() * 100
test_pct = y_test.value_counts(normalize=True).sort_index() * 100

distribution_check = pd.DataFrame({
    "Original_%": original_pct.round(3),
    "Train_%": train_pct.round(3),
    "Test_%": test_pct.round(3)
})

print(distribution_check)

# Difference should be very small
max_train_diff = np.abs(original_pct - train_pct).max()
max_test_diff = np.abs(original_pct - test_pct).max()

print(f"\nMaximum train distribution difference: {max_train_diff:.4f}%")
print(f"Maximum test distribution difference : {max_test_diff:.4f}%")

# ============================================================
# 5. CHECK TRAIN / TEST SHA256 SEPARATION
# ============================================================

print("\n" + "=" * 70)
print("SHA256 TRAIN / TEST SEPARATION")
print("=" * 70)

train_sha_set = set(sha_train)
test_sha_set = set(sha_test)

overlap = train_sha_set.intersection(test_sha_set)

print(f"Training SHA256 : {len(train_sha_set):,}")
print(f"Test SHA256     : {len(test_sha_set):,}")
print(f"Overlap         : {len(overlap)}")

assert len(overlap) == 0

print("PASS: No SHA256 appears in both train and test.")

# ============================================================
# 6. CHECK FEATURE TYPES
# ============================================================

print("\n" + "=" * 70)
print("FEATURE TYPE CHECK")
print("=" * 70)

non_numeric_train = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

non_numeric_test = X_test.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print(f"Non-numeric train features: {len(non_numeric_train)}")
print(f"Non-numeric test features : {len(non_numeric_test)}")

assert len(non_numeric_train) == 0
assert len(non_numeric_test) == 0

print("PASS: All features are numeric.")

# ============================================================
# 7. CHECK FOR TRAIN / TEST MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print(f"Training missing values: {train_missing}")
print(f"Test missing values    : {test_missing}")

assert train_missing == 0
assert test_missing == 0

print("PASS: No missing feature values.")

# ============================================================
# 8. PREPROCESSING DECISION
# ============================================================

print("\n" + "=" * 70)
print("PREPROCESSING ANALYSIS")
print("=" * 70)

binary_features = []
continuous_features = []

for col in X_train.columns:

    unique_values = X_train[col].nunique()

    values = X_train[col].unique()

    if unique_values == 2 and set(values).issubset({0, 1}):
        binary_features.append(col)
    else:
        continuous_features.append(col)

print(f"\nBinary features     : {len(binary_features)}")
print(f"Non-binary features : {len(continuous_features)}")
print(f"Total features      : {len(binary_features) + len(continuous_features)}")

assert len(binary_features) + len(continuous_features) == 318

# ============================================================
# 9. SHOW FEATURE GROUPS
# ============================================================

print("\n" + "=" * 70)
print("BINARY FEATURE EXAMPLES")
print("=" * 70)

for col in binary_features[:20]:
    print(f"  {col}")

if len(binary_features) > 20:
    print(f"  ... and {len(binary_features) - 20} more")

print("\n" + "=" * 70)
print("NON-BINARY FEATURE EXAMPLES")
print("=" * 70)

for col in continuous_features[:20]:
    print(f"  {col}")

if len(continuous_features) > 20:
    print(f"  ... and {len(continuous_features) - 20} more")

# ============================================================
# 10. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT COMPLETE")
print("=" * 70)

print(f"Total samples : {len(df):,}")
print(f"Training      : {len(X_train):,}")
print(f"Testing       : {len(X_test):,}")
print(f"Features      : {X_train.shape[1]}")
print(f"Classes       : {y.nunique()}")

print("\nPASS:")
print("  ✓ Stratified split")
print("  ✓ No SHA256 leakage")
print("  ✓ No missing values")
print("  ✓ All features numeric")
print("  ✓ 318 features preserved")
print("  ✓ All 7 classes present")

print("\nNEXT STEP:")
print("  Build the leakage-safe preprocessing pipeline.")
print("=" * 70)

gc.collect()

TRAIN / TEST SPLIT + PREPROCESSING SETUP

Loading final dataset...
Dataset shape: (29489, 320)

SEPARATING DATA
X shape : (29489, 318)
y shape : (29489,)
Features : 318
Samples  : 29489

PASS: SHA256 excluded from ML features.

STRATIFIED TRAIN / TEST SPLIT

Training samples : 23,591
Test samples     : 5,898

Training distribution:
Type
0    1502
1    4017
2    3714
3    3964
4    4061
5    3374
6    2959
Name: count, dtype: int64

Test distribution:
Type
0     375
1    1005
2     929
3     991
4    1015
5     843
6     740
Name: count, dtype: int64

STRATIFICATION CHECK
      Original_%  Train_%  Test_%
Type                             
0          6.365    6.367   6.358
1         17.030   17.028  17.040
2         15.745   15.743  15.751
3         16.803   16.803  16.802
4         17.213   17.214  17.209
5         14.300   14.302  14.293
6         12.544   12.543  12.547

Maximum train distribution difference: 0.0024%
Maximum test distribution difference : 0.0096%

SHA256 TRAIN / TEST 

0

In [23]:
# ============================================================
# LEAKAGE-SAFE PREPROCESSING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import gc

from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("=" * 70)
print("BUILDING LEAKAGE-SAFE PREPROCESSING PIPELINE")
print("=" * 70)

# ============================================================
# 1. RELOAD FINAL DATASET
# ============================================================

FINAL_PATH = "/kaggle/working/Windows_Malware_Final_318_Features.csv"

df = pd.read_csv(FINAL_PATH)

X = df.drop(columns=["SHA256", "Type"])
y = df["Type"]

# ============================================================
# 2. RECREATE IDENTICAL STRATIFIED SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTraining samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features         : {X_train.shape[1]}")

# ============================================================
# 3. IDENTIFY BINARY / NON-BINARY FEATURES
# ============================================================

binary_features = []
numeric_features = []

for col in X_train.columns:

    unique_values = X_train[col].nunique()
    values = set(X_train[col].unique())

    if unique_values == 2 and values.issubset({0, 1}):
        binary_features.append(col)
    else:
        numeric_features.append(col)

print("\n" + "=" * 70)
print("FEATURE GROUPS")
print("=" * 70)

print(f"Binary features     : {len(binary_features)}")
print(f"Non-binary features : {len(numeric_features)}")
print(f"Total               : {len(binary_features) + len(numeric_features)}")

assert len(binary_features) == 227
assert len(numeric_features) == 91

# ============================================================
# 4. NUMERIC TRANSFORMATION
# ============================================================
#
# PE features contain extremely large and highly skewed values.
#
# log1p(x) compresses the extreme right tail while preserving
# zero values:
#
#     x -> log(1 + x)
#
# Then StandardScaler gives the transformed numeric features
# approximately zero mean / unit variance.
#
# IMPORTANT:
# This transformation is fitted/applied through the pipeline
# and NEVER uses test-set statistics.
# ============================================================

log_scaler = Pipeline([
    (
        "log1p",
        FunctionTransformer(
            np.log1p,
            validate=False
        )
    ),
    (
        "standard_scaler",
        StandardScaler()
    )
])

# ============================================================
# 5. COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            "passthrough",
            binary_features
        ),
        (
            "numeric",
            log_scaler,
            numeric_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("\n" + "=" * 70)
print("PREPROCESSOR CREATED")
print("=" * 70)

print("Binary features : passthrough")
print("Numeric features: log1p + StandardScaler")
print("Other features  : dropped")

# ============================================================
# 6. FIT ONLY ON TRAINING DATA
# ============================================================

print("\n" + "=" * 70)
print("FITTING ON TRAINING DATA")
print("=" * 70)

X_train_processed = preprocessor.fit_transform(X_train)

print(f"Processed training shape: {X_train_processed.shape}")

# ============================================================
# 7. TRANSFORM TEST DATA
# ============================================================

print("\nTransforming test data...")

X_test_processed = preprocessor.transform(X_test)

print(f"Processed test shape    : {X_test_processed.shape}")

# ============================================================
# 8. VERIFY FEATURE COUNT
# ============================================================

assert X_train_processed.shape[1] == 318
assert X_test_processed.shape[1] == 318

print("\nPASS: 318 features preserved.")

# ============================================================
# 9. CHECK FOR NaN / INF
# ============================================================

print("\n" + "=" * 70)
print("PROCESSED DATA VALIDATION")
print("=" * 70)

train_nan = np.isnan(X_train_processed).sum()
test_nan = np.isnan(X_test_processed).sum()

train_inf = np.isinf(X_train_processed).sum()
test_inf = np.isinf(X_test_processed).sum()

print(f"Training NaN      : {train_nan}")
print(f"Test NaN          : {test_nan}")
print(f"Training infinity : {train_inf}")
print(f"Test infinity     : {test_inf}")

assert train_nan == 0
assert test_nan == 0
assert train_inf == 0
assert test_inf == 0

print("PASS: No NaN or infinite values.")

# ============================================================
# 10. CHECK NUMERIC STANDARDIZATION
# ============================================================

numeric_indices = [
    i for i, col in enumerate(X_train.columns)
    if col in numeric_features
]

processed_numeric = X_train_processed[:, len(binary_features):]

print("\n" + "=" * 70)
print("STANDARDIZATION CHECK")
print("=" * 70)

numeric_means = np.mean(processed_numeric, axis=0)
numeric_stds = np.std(processed_numeric, axis=0)

print(
    f"Mean of transformed numeric means : "
    f"{numeric_means.mean():.6f}"
)

print(
    f"Mean of transformed numeric stds  : "
    f"{numeric_stds.mean():.6f}"
)

print(
    f"Maximum absolute feature mean     : "
    f"{np.max(np.abs(numeric_means)):.6f}"
)

print(
    f"Minimum feature std               : "
    f"{np.min(numeric_stds):.6f}"
)

# ============================================================
# 11. CHECK BINARY FEATURES
# ============================================================

processed_binary = X_train_processed[:, :len(binary_features)]

binary_unique = np.unique(processed_binary)

print("\n" + "=" * 70)
print("BINARY FEATURE CHECK")
print("=" * 70)

print(f"Unique values in processed binary features:")
print(binary_unique)

assert np.all(np.isin(binary_unique, [0, 1]))

print("PASS: Binary features remain 0/1.")

# ============================================================
# 12. FINAL PREPROCESSING SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)

print(f"Original features       : {X_train.shape[1]}")
print(f"Binary features         : {len(binary_features)}")
print(f"Numeric features        : {len(numeric_features)}")
print(f"Processed features      : {X_train_processed.shape[1]}")
print(f"Training samples        : {X_train_processed.shape[0]:,}")
print(f"Test samples            : {X_test_processed.shape[0]:,}")

print("\nTransformation:")
print("  Binary → unchanged")
print("  Numeric → log1p → StandardScaler")

print("\nLeakage protection:")
print("  ✓ Preprocessor fitted ONLY on training data")
print("  ✓ Test data transformed using training parameters")
print("  ✓ SHA256 excluded")
print("  ✓ Target excluded")

print("\nNEXT STEP:")
print("  Train baseline ML models.")
print("=" * 70)

gc.collect()

BUILDING LEAKAGE-SAFE PREPROCESSING PIPELINE

Training samples : 23,591
Test samples     : 5,898
Features         : 318

FEATURE GROUPS
Binary features     : 227
Non-binary features : 91
Total               : 318

PREPROCESSOR CREATED
Binary features : passthrough
Numeric features: log1p + StandardScaler
Other features  : dropped

FITTING ON TRAINING DATA
Processed training shape: (23591, 318)

Transforming test data...
Processed test shape    : (5898, 318)

PASS: 318 features preserved.

PROCESSED DATA VALIDATION
Training NaN      : 0
Test NaN          : 0
Training infinity : 0
Test infinity     : 0
PASS: No NaN or infinite values.

STANDARDIZATION CHECK
Mean of transformed numeric means : -0.000000
Mean of transformed numeric stds  : 1.000000
Maximum absolute feature mean     : 0.000000
Minimum feature std               : 1.000000

BINARY FEATURE CHECK
Unique values in processed binary features:
[0. 1.]
PASS: Binary features remain 0/1.

PREPROCESSING COMPLETE
Original features      

55

In [24]:
# ============================================================
# BASELINE MODEL COMPARISON
# ============================================================

import pandas as pd
import numpy as np
import time
import gc

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("BASELINE MODEL COMPARISON")
print("=" * 70)

# ============================================================
# IMPORTANT:
# X_train_processed, X_test_processed, y_train, y_test
# were created in the previous cell.
#
# We verify that they still exist.
# ============================================================

required_vars = [
    "X_train_processed",
    "X_test_processed",
    "y_train",
    "y_test"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Missing variables from preprocessing step: {missing_vars}\n"
        "Please rerun the preprocessing cell first."
    )

print("\nInput data:")
print(f"Training : {X_train_processed.shape}")
print(f"Test     : {X_test_processed.shape}")

# ============================================================
# MODEL DEFINITIONS
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.08,
        max_leaf_nodes=31,
        random_state=42
    )
}

# ============================================================
# TRAIN + EVALUATE
# ============================================================

results = []
predictions = {}

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    start_time = time.time()

    model.fit(
        X_train_processed,
        y_train
    )

    train_time = time.time() - start_time

    print(f"Training time: {train_time:.2f} seconds")

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    start_time = time.time()

    y_pred = model.predict(
        X_test_processed
    )

    prediction_time = time.time() - start_time

    predictions[model_name] = y_pred

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro_F1": macro_f1,
        "Weighted_F1": weighted_f1,
        "Training_Time_sec": train_time,
        "Prediction_Time_sec": prediction_time
    })

    # --------------------------------------------------------
    # CLASSIFICATION REPORT
    # --------------------------------------------------------

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=4
        )
    )

# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("BASELINE MODEL RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ============================================================
# BEST MODEL
# ============================================================

best_model_name = results_df.iloc[0]["Model"]

print("\n" + "=" * 70)
print("BEST BASELINE MODEL")
print("=" * 70)

print(f"Model       : {best_model_name}")
print(
    f"Accuracy    : "
    f"{results_df.iloc[0]['Accuracy']:.4f}"
)
print(
    f"Macro F1    : "
    f"{results_df.iloc[0]['Macro_F1']:.4f}"
)
print(
    f"Weighted F1 : "
    f"{results_df.iloc[0]['Weighted_F1']:.4f}"
)

# ============================================================
# CONFUSION MATRIX FOR BEST MODEL
# ============================================================

best_predictions = predictions[best_model_name]

cm = confusion_matrix(
    y_test,
    best_predictions
)

print("\n" + "=" * 70)
print(f"CONFUSION MATRIX — {best_model_name}")
print("=" * 70)

print(
    pd.DataFrame(
        cm,
        index=[f"True_{i}" for i in range(7)],
        columns=[f"Pred_{i}" for i in range(7)]
    )
)

# ============================================================
# SAVE RESULTS
# ============================================================

RESULTS_PATH = "/kaggle/working/Baseline_Model_Results.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("\n" + "=" * 70)
print("BASELINE EVALUATION COMPLETE")
print("=" * 70)

print(f"Results saved:")
print(RESULTS_PATH)

print("\nNEXT STEP:")
print("Analyze baseline performance and tune the strongest model.")

gc.collect()

BASELINE MODEL COMPARISON

Input data:
Training : (23591, 318)
Test     : (5898, 318)

TRAINING: Logistic Regression
Training time: 44.30 seconds

Classification Report:
              precision    recall  f1-score   support

           0     0.8237    0.7227    0.7699       375
           1     0.9247    0.8796    0.9016      1005
           2     0.9839    0.9882    0.9860       929
           3     0.7422    0.7699    0.7558       991
           4     0.8629    0.8000    0.8303      1015
           5     0.5230    0.5267    0.5248       843
           6     0.4675    0.5446    0.5031       740

    accuracy                         0.7621      5898
   macro avg     0.7611    0.7474    0.7531      5898
weighted avg     0.7715    0.7621    0.7659      5898


TRAINING: Random Forest
Training time: 7.97 seconds

Classification Report:
              precision    recall  f1-score   support

           0     0.9481    0.9253    0.9366       375
           1     0.9502    0.9493    0.9497    

24

In [25]:
# ============================================================
# HISTGRADIENTBOOSTING HYPERPARAMETER TUNING
# ============================================================

import pandas as pd
import numpy as np
import time
import gc

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

print("=" * 70)
print("HISTGRADIENTBOOSTING HYPERPARAMETER TUNING")
print("=" * 70)

# ============================================================
# VERIFY PREVIOUS VARIABLES
# ============================================================

required_vars = [
    "X_train_processed",
    "X_test_processed",
    "y_train",
    "y_test"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Missing variables: {missing_vars}\n"
        "Please rerun the preprocessing cell."
    )

print("\nTraining data:")
print(f"X_train : {X_train_processed.shape}")
print(f"y_train : {y_train.shape}")

print("\nTest data:")
print(f"X_test  : {X_test_processed.shape}")
print(f"y_test  : {y_test.shape}")

# ============================================================
# BASELINE REFERENCE
# ============================================================

BASELINE_MACRO_F1 = 0.8992
BASELINE_ACCURACY = 0.8979

print("\n" + "=" * 70)
print("BASELINE REFERENCE")
print("=" * 70)

print(f"Baseline Accuracy : {BASELINE_ACCURACY:.4f}")
print(f"Baseline Macro F1 : {BASELINE_MACRO_F1:.4f}")

# ============================================================
# CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# ============================================================
# MODEL
# ============================================================

hgb = HistGradientBoostingClassifier(
    random_state=42
)

# ============================================================
# PARAMETER SEARCH
# ============================================================

param_distributions = {

    "learning_rate": [
        0.03,
        0.05,
        0.08,
        0.10,
        0.15
    ],

    "max_iter": [
        200,
        300,
        400,
        500
    ],

    "max_leaf_nodes": [
        15,
        31,
        63,
        127
    ],

    "max_depth": [
        None,
        6,
        10,
        15
    ],

    "min_samples_leaf": [
        10,
        20,
        30,
        50,
        75
    ],

    "l2_regularization": [
        0.0,
        0.1,
        0.5,
        1.0,
        5.0
    ]
}

# ============================================================
# RANDOMIZED SEARCH
# ============================================================

print("\n" + "=" * 70)
print("STARTING RANDOMIZED SEARCH")
print("=" * 70)

print("\nCV folds          : 3")
print("Random candidates : 20")
print("Scoring           : Macro F1")
print("Parallel jobs     : -1")

search = RandomizedSearchCV(
    estimator=hgb,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

start_time = time.time()

search.fit(
    X_train_processed,
    y_train
)

search_time = time.time() - start_time

# ============================================================
# SEARCH COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

print(f"Search time: {search_time / 60:.2f} minutes")

# ============================================================
# BEST PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("BEST HYPERPARAMETERS")
print("=" * 70)

for parameter, value in search.best_params_.items():
    print(f"{parameter:20s}: {value}")

print("\nBest CV Macro F1:")
print(f"{search.best_score_:.4f}")

# ============================================================
# ALL SEARCH RESULTS
# ============================================================

cv_results = pd.DataFrame(
    search.cv_results_
)

result_columns = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "params"
]

results_view = (
    cv_results[result_columns]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP HYPERPARAMETER CONFIGURATIONS")
print("=" * 70)

print(
    results_view.head(10).to_string(
        index=False
    )
)

# ============================================================
# SAVE SEARCH RESULTS
# ============================================================

SEARCH_RESULTS_PATH = (
    "/kaggle/working/HGB_Hyperparameter_Search.csv"
)

results_view.to_csv(
    SEARCH_RESULTS_PATH,
    index=False
)

print("\nSaved:")
print(SEARCH_RESULTS_PATH)

# ============================================================
# TRAIN BEST MODEL
# ============================================================

print("\n" + "=" * 70)
print("TRAINING BEST HGB MODEL")
print("=" * 70)

best_hgb = search.best_estimator_

start_time = time.time()

best_hgb.fit(
    X_train_processed,
    y_train
)

training_time = time.time() - start_time

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)

# ============================================================
# TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

y_pred_tuned = best_hgb.predict(
    X_test_processed
)

tuned_accuracy = accuracy_score(
    y_test,
    y_pred_tuned
)

tuned_macro_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average="macro"
)

tuned_weighted_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average="weighted"
)

print(f"\nAccuracy    : {tuned_accuracy:.4f}")
print(f"Macro F1    : {tuned_macro_f1:.4f}")
print(f"Weighted F1 : {tuned_weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_tuned,
        digits=4
    )
)

# ============================================================
# IMPROVEMENT
# ============================================================

print("\n" + "=" * 70)
print("IMPROVEMENT OVER BASELINE")
print("=" * 70)

accuracy_change = (
    tuned_accuracy - BASELINE_ACCURACY
)

macro_f1_change = (
    tuned_macro_f1 - BASELINE_MACRO_F1
)

print(
    f"Accuracy change : "
    f"{accuracy_change:+.4f}"
)

print(
    f"Macro F1 change : "
    f"{macro_f1_change:+.4f}"
)

print(
    f"Baseline Macro F1 : "
    f"{BASELINE_MACRO_F1:.4f}"
)

print(
    f"Tuned Macro F1    : "
    f"{tuned_macro_f1:.4f}"
)

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("HGB TUNING COMPLETE")
print("=" * 70)

if tuned_macro_f1 > BASELINE_MACRO_F1:
    print("✓ Tuned model improved over baseline.")
else:
    print("! Tuned model did not improve over baseline.")
    print("  Baseline HGB remains the stronger configuration.")

print("\nNEXT STEP:")
print("Evaluate whether class-specific improvements are possible.")
print("=" * 70)

gc.collect()

HISTGRADIENTBOOSTING HYPERPARAMETER TUNING

Training data:
X_train : (23591, 318)
y_train : (23591,)

Test data:
X_test  : (5898, 318)
y_test  : (5898,)

BASELINE REFERENCE
Baseline Accuracy : 0.8979
Baseline Macro F1 : 0.8992

STARTING RANDOMIZED SEARCH

CV folds          : 3
Random candidates : 20
Scoring           : Macro F1
Parallel jobs     : -1
Fitting 3 folds for each of 20 candidates, totalling 60 fits

SEARCH COMPLETE
Search time: 26.14 minutes

BEST HYPERPARAMETERS
min_samples_leaf    : 10
max_leaf_nodes      : 63
max_iter            : 300
max_depth           : 15
learning_rate       : 0.05
l2_regularization   : 1.0

Best CV Macro F1:
0.8886

TOP HYPERPARAMETER CONFIGURATIONS
 rank_test_score  mean_test_score  std_test_score  mean_train_score                                                                                                                               params
               1         0.888637        0.000921          0.964423    {'min_samples_leaf': 10, 'max_lea

27

[CV] END l2_regularization=5.0, learning_rate=0.08, max_depth=10, max_iter=500, max_leaf_nodes=63, min_samples_leaf=10; total time= 2.2min
[CV] END l2_regularization=0.0, learning_rate=0.08, max_depth=10, max_iter=500, max_leaf_nodes=15, min_samples_leaf=10; total time= 1.4min
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=15, max_iter=300, max_leaf_nodes=63, min_samples_leaf=10; total time= 2.2min
[CV] END l2_regularization=1.0, learning_rate=0.05, max_depth=None, max_iter=500, max_leaf_nodes=63, min_samples_leaf=20; total time= 1.7min
[CV] END l2_regularization=0.5, learning_rate=0.05, max_depth=15, max_iter=200, max_leaf_nodes=63, min_samples_leaf=30; total time= 1.9min
[CV] END l2_regularization=0.1, learning_rate=0.15, max_depth=10, max_iter=400, max_leaf_nodes=63, min_samples_leaf=30; total time=  45.9s
[CV] END l2_regularization=1.0, learning_rate=0.08, max_depth=15, max_iter=400, max_leaf_nodes=63, min_samples_leaf=75; total time= 1.4min
[CV] END l2_regularizatio

In [26]:
# ============================================================
# TUNED HGB — DETAILED ERROR ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score
)

print("=" * 70)
print("TUNED HGB — DETAILED ERROR ANALYSIS")
print("=" * 70)

# ============================================================
# VERIFY MODEL / PREDICTIONS
# ============================================================

if "best_hgb" not in globals():
    raise RuntimeError(
        "best_hgb not found. Please run the HGB tuning cell first."
    )

if "y_pred_tuned" not in globals():
    y_pred_tuned = best_hgb.predict(X_test_processed)

# ============================================================
# BASIC METRICS
# ============================================================

accuracy = accuracy_score(y_test, y_pred_tuned)
macro_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average="weighted"
)

print("\n" + "=" * 70)
print("OVERALL PERFORMANCE")
print("=" * 70)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    y_test,
    y_pred_tuned,
    output_dict=True
)

report_df = pd.DataFrame(report).T

print("\n" + "=" * 70)
print("PER-CLASS PERFORMANCE")
print("=" * 70)

print(
    report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(4).to_string()
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred_tuned
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in range(7)],
    columns=[f"Pred_{i}" for i in range(7)]
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm_df)

# ============================================================
# NORMALIZED CONFUSION MATRIX
# ============================================================

cm_normalized = (
    cm.astype(float)
    / cm.sum(axis=1, keepdims=True)
)

cm_norm_df = pd.DataFrame(
    cm_normalized,
    index=[f"True_{i}" for i in range(7)],
    columns=[f"Pred_{i}" for i in range(7)]
)

print("\n" + "=" * 70)
print("NORMALIZED CONFUSION MATRIX")
print("=" * 70)

print(
    cm_norm_df.round(4).to_string()
)

# ============================================================
# MOST COMMON MISCLASSIFICATIONS
# ============================================================

errors = []

for true_class in range(7):

    for predicted_class in range(7):

        if true_class != predicted_class:

            count = cm[
                true_class,
                predicted_class
            ]

            if count > 0:

                errors.append({
                    "True_Class": true_class,
                    "Predicted_Class": predicted_class,
                    "Errors": count
                })

errors_df = (
    pd.DataFrame(errors)
    .sort_values(
        "Errors",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP MISCLASSIFICATIONS")
print("=" * 70)

print(
    errors_df.head(15).to_string(
        index=False
    )
)

# ============================================================
# ERROR RATE PER CLASS
# ============================================================

class_errors = []

for class_id in range(7):

    total = cm[class_id].sum()
    correct = cm[class_id, class_id]
    errors_count = total - correct

    class_errors.append({
        "Class": class_id,
        "Samples": total,
        "Correct": correct,
        "Errors": errors_count,
        "Error_Rate": errors_count / total
    })

class_errors_df = (
    pd.DataFrame(class_errors)
    .sort_values(
        "Error_Rate",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("CLASS ERROR RATES")
print("=" * 70)

print(
    class_errors_df.to_string(
        index=False,
        formatters={
            "Error_Rate": "{:.4f}".format
        }
    )
)

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

if hasattr(best_hgb, "feature_importances_"):

    importances = best_hgb.feature_importances_

    feature_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })

    feature_importance_df = (
        feature_importance_df
        .sort_values(
            "Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\nTop 30 features:")

    print(
        feature_importance_df.head(30)
        .to_string(index=False)
    )

else:

    print(
        "HistGradientBoosting does not expose "
        "feature_importances_ directly."
    )

    feature_importance_df = None

# ============================================================
# SAVE RESULTS
# ============================================================

CM_PATH = (
    "/kaggle/working/"
    "HGB_Tuned_Confusion_Matrix.csv"
)

REPORT_PATH = (
    "/kaggle/working/"
    "HGB_Tuned_Classification_Report.csv"
)

ERROR_PATH = (
    "/kaggle/working/"
    "HGB_Tuned_Misclassification_Analysis.csv"
)

CLASS_ERROR_PATH = (
    "/kaggle/working/"
    "HGB_Tuned_Class_Error_Rates.csv"
)

cm_df.to_csv(
    CM_PATH
)

report_df.to_csv(
    REPORT_PATH
)

errors_df.to_csv(
    ERROR_PATH,
    index=False
)

class_errors_df.to_csv(
    CLASS_ERROR_PATH,
    index=False
)

if feature_importance_df is not None:

    IMPORTANCE_PATH = (
        "/kaggle/working/"
        "HGB_Tuned_Feature_Importance.csv"
    )

    feature_importance_df.to_csv(
        IMPORTANCE_PATH,
        index=False
    )

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ERROR ANALYSIS COMPLETE")
print("=" * 70)

print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")

print("\nSaved:")
print(CM_PATH)
print(REPORT_PATH)
print(ERROR_PATH)
print(CLASS_ERROR_PATH)

if feature_importance_df is not None:
    print(IMPORTANCE_PATH)

print("\nNEXT STEP:")
print(
    "Use the error patterns to decide whether "
    "class-specific modeling or feature refinement is justified."
)

gc.collect()

TUNED HGB — DETAILED ERROR ANALYSIS

OVERALL PERFORMANCE
Accuracy    : 0.9027
Macro F1    : 0.9035
Weighted F1 : 0.9038

PER-CLASS PERFORMANCE
              precision  recall  f1-score    support
0                0.9700  0.9493    0.9596   375.0000
1                0.9516  0.9582    0.9549  1005.0000
2                0.9935  0.9946    0.9941   929.0000
3                0.9355  0.8345    0.8821   991.0000
4                0.9431  0.9153    0.9290  1015.0000
5                0.8194  0.7912    0.8051   843.0000
6                0.7263  0.8892    0.7995   740.0000
accuracy         0.9027  0.9027    0.9027     0.9027
macro avg        0.9056  0.9046    0.9035  5898.0000
weighted avg     0.9081  0.9027    0.9038  5898.0000

CONFUSION MATRIX
        Pred_0  Pred_1  Pred_2  Pred_3  Pred_4  Pred_5  Pred_6
True_0     356       0       2       9       1       6       1
True_1       1     963       0       7      26       4       4
True_2       0       3     924       1       1       0       0
True

0

In [27]:
from IPython.display import FileLink, display
import os

# ============================================================
# DOWNLOAD FINAL MALWARE DATASET
# ============================================================

FINAL_DATASET = "/kaggle/working/Windows_Malware_Final_318_Features.csv"

print("=" * 70)
print("FINAL MALWARE DATASET DOWNLOAD")
print("=" * 70)

if not os.path.exists(FINAL_DATASET):
    raise FileNotFoundError(
        f"Final dataset not found:\n{FINAL_DATASET}"
    )

# File information
file_size_mb = os.path.getsize(FINAL_DATASET) / (1024 ** 2)

print(f"File : {FINAL_DATASET}")
print(f"Size : {file_size_mb:.2f} MB")

# Create clickable download link
print("\n" + "=" * 70)
print("DOWNLOAD LINK")
print("=" * 70)

display(FileLink(
    FINAL_DATASET,
    result_html_prefix="⬇️ Download: "
))

print("\nPASS: Final malware dataset is ready for download.")

FINAL MALWARE DATASET DOWNLOAD
File : /kaggle/working/Windows_Malware_Final_318_Features.csv
Size : 24.19 MB

DOWNLOAD LINK


/kaggle/working/Windows_Malware_Final_318_Features.csv


PASS: Final malware dataset is ready for download.
